In [ ]:
# -*- coding: utf-8 -*-
"""
ENHANCED SILVER SWING TRADING SYSTEM V3.5 - DEDICATED SHORT MODEL
==================================================================================
V3.5 additions over V3.4 Balanced:
- Proper dedicated SHORT model (LSTM + RF + XGB trained on SHORT_Label)
  instead of inverting BUY probability (short_prob = 1 - buy_prob was wrong)
- COT commercial-covering momentum features (COT_comm_covering, COT_covering_momentum)
- NW_Position velocity features (NW_Position_ROC_5/20, NW_VELOCITY)
- Cross-metal rotation score (SI_OUTPERFORMS_GOLD/COPPER, METAL_ROTATION_SCORE)
- Yield curve steepening + z-score (YIELD_CURVE_STEEPENING, YIELD_CURVE_ZSCORE)

Key philosophy:
- LONGS: Moderate thresholds, 2/3 agreement required
- SHORTS: Separate SHORT_Label model — "don't buy" ≠ "actively short"
- Shorts enabled in: STRONG_DOWNTREND, HIGH_VOLATILITY
- Shorts blocked in: CONSOLIDATION, STRONG_UPTREND
"""
import os
os.environ["UN_COMTRADE_API_KEY"] = "403062414ab94eb7bd251699a14536ff"

import yfinance as yf
import pandas as pd
import numpy as np
import pandas_ta as ta
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.dates import DateFormatter
import matplotlib.dates as mdates
from matplotlib.gridspec import GridSpec
from scipy.stats import norm
from scipy.optimize import minimize
from datetime import datetime, timedelta
from dataclasses import dataclass
from enum import Enum
from typing import Tuple, Dict, Optional, List

import warnings
warnings.filterwarnings('ignore')

# Suppress TensorFlow deprecation and info logs — these route through TF's own
# logging system and bypass Python's warnings module entirely.
# '3' = ERROR only (0=all, 1=INFO+, 2=WARNING+, 3=ERROR only)
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# absl-py is the logging backend TF/Keras use internally — suppress it too
import logging
logging.getLogger('tensorflow').setLevel(logging.ERROR)
logging.getLogger('absl').setLevel(logging.ERROR)

from statsmodels.nonparametric.kernel_regression import KernelReg

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix, 
                            accuracy_score, precision_score, recall_score, f1_score)
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif
from sklearn.calibration import CalibratedClassifierCV
from sklearn.decomposition import PCA
from sklearn.linear_model import LassoCV, LogisticRegressionCV
from sklearn.feature_selection import SelectFromModel

from imblearn.over_sampling import SMOTE

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, backend as K
import xgboost as xgb

from scipy.stats import ttest_ind, mannwhitneyu

from pandas_datareader import data as web

import urllib.request
import zipfile
import io
import gc

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("="*70)
print("ENHANCED SILVER SWING TRADING SYSTEM V3.4")
print("BALANCED: HIGH SHARPE + SMART SHORTS")
print("="*70)


# ══════════════════════════════════════════════════════════════════════
# ENUMS AND REGIME CONFIGURATION - OPTIMIZED FOR SHARPE
# ══════════════════════════════════════════════════════════════════════

class MarketRegime(Enum):
    STRONG_UPTREND = "STRONG_UPTREND"
    STRONG_DOWNTREND = "STRONG_DOWNTREND"
    CONSOLIDATION = "CONSOLIDATION"
    HIGH_VOLATILITY = "HIGH_VOLATILITY"


class TradeDirection(Enum):
    LONG = "LONG"
    SHORT = "SHORT"
    FLAT = "FLAT"


@dataclass
class RegimeConfig:
    """Configuration for each market regime - OPTIMIZED FOR SHARPE"""
    long_threshold: float
    short_threshold: float
    allow_long: bool
    allow_short: bool
    position_size_mult: float
    stop_loss_pct: float
    take_profit_pct: float
    use_mean_reversion: bool
    min_model_agreement: int  # NEW: require N/3 models to agree
    description: str


# V3.4 BALANCED REGIME CONFIGS
# High Sharpe focus + Smart Shorts (only best setups)
REGIME_CONFIGS = {
    MarketRegime.STRONG_UPTREND: RegimeConfig(
        long_threshold=0.65,      # Raised 0.55→0.65: stricter entry at extended prices
        short_threshold=0.95,     # Block shorts in uptrend
        allow_long=True,
        allow_short=False,        # Don't fight the trend
        position_size_mult=1.0,
        stop_loss_pct=0.04,
        take_profit_pct=0.08,
        use_mean_reversion=False,
        min_model_agreement=2,
        description="📈 Strong Uptrend - Quality longs only"
    ),
    MarketRegime.STRONG_DOWNTREND: RegimeConfig(
        long_threshold=0.99,      # Effectively blocked
        short_threshold=0.99,     # Effectively blocked
        allow_long=False,
        allow_short=False,        # REVERTED to all-trading-blocked.
                                  # The shorts-enabled experiment (0.72 thr,
                                  # 3/3 agreement, 8% stop) was A/B tested
                                  # against this on identical data/seed and
                                  # LOST on every headline metric:
                                  #   blocked : $796,532 | Sharpe 1.912
                                  #   shorts  : $728,579 | Sharpe 1.619
                                  # The dedicated-short robustness printout
                                  # showed STRONG_DOWNTREND shorts = 8 trades,
                                  # 12.5% win, -$46,121. Shorts pay in
                                  # CONSOLIDATION and HIGH_VOLATILITY (which
                                  # remain enabled), NOT inside a confirmed
                                  # downtrend, where entry timing gets whipsawed
                                  # and the wide stop only enlarges the losers
                                  # (worst trade -5% -> -8%). Standing aside
                                  # is the better system. Any further
                                  # downtrend-short work should be a separate
                                  # scoped study, not the live config.
        position_size_mult=0.0,
        stop_loss_pct=0.04,
        take_profit_pct=0.08,
        use_mean_reversion=False,
        min_model_agreement=3,
        description="📉 Strong Downtrend - ALL TRADING BLOCKED"
    ),
    MarketRegime.CONSOLIDATION: RegimeConfig(
        long_threshold=0.62,      # Run 12→13: specialist probability distribution is
                                  # compressed vs generalist (trains on ~200-288 pure
                                  # consolidation bars where moves are 1-2%).  0.62
                                  # matches genuine specialist output.  Dead zones
                                  # W9-W21 were caused by the old 0.70 threshold being
                                  # calibrated to a generalist, not this specialist.
        short_threshold=0.70,     # Run 13: shorts now enabled. Dedicated short model
                                  # architecture justifies this — NOT the old inverted-BUY
                                  # proxy.  0.70 = slightly higher bar than longs (0.62)
                                  # because silver is in a secular bull market.
        allow_long=True,
        allow_short=True,         # Run 13: enabled. Dedicated short model architecture
                                  # justifies this — NOT the old inverted-BUY proxy.
        position_size_mult=0.5,   # Applies to both directions: lower threshold vs
                                  # HIGH_VOL = smaller size to compensate.
        stop_loss_pct=0.025,      # Tightened from 0.035: in a 1-2% vol environment
                                  # a 3.5% adverse move means the idea has already
                                  # failed.  2.5% cuts at ~1σ. Applies both directions.
        take_profit_pct=0.05,     # R:R = 2:1 both directions.
        use_mean_reversion=True,
        min_model_agreement=2,
        description="➡️ Consolidation - Bidirectional"
    ),
    MarketRegime.HIGH_VOLATILITY: RegimeConfig(
        long_threshold=0.60,      # Moderate for longs
        short_threshold=0.65,     # ✅ SHORTS ENABLED with HIGHER bar
        allow_long=True,
        allow_short=True,         # ✅ SHORTS ENABLED - vol works both ways
        position_size_mult=0.5,   # Smaller due to vol
        stop_loss_pct=0.05,
        take_profit_pct=0.10,
        use_mean_reversion=False,
        min_model_agreement=2,
        description="⚡ High Volatility - Bidirectional, high conviction only"
    ),
}


# ══════════════════════════════════════════════════════════════════════════════
# COMEX SILVER FUTURES SPECIFICATIONS
# ══════════════════════════════════════════════════════════════════════════════

@dataclass
class ComexSilverSpecs:
    """Official COMEX Silver Futures Contract Specifications"""
    
    # Full-size contract (SI)
    FULL_CONTRACT_SIZE: int = 5000  # troy ounces
    FULL_TICK_SIZE: float = 0.005   # $0.005 per oz
    FULL_TICK_VALUE: float = 25.00  # $25 per tick
    FULL_POINT_VALUE: float = 5000  # $5000 per $1 move
    FULL_SYMBOL: str = "SI"
    FULL_MARGIN_INITIAL: float = 16500
    FULL_MARGIN_MAINTENANCE: float = 15000
    
    # Mini contract (QI)
    MINI_CONTRACT_SIZE: int = 2500
    MINI_TICK_SIZE: float = 0.005
    MINI_TICK_VALUE: float = 12.50
    MINI_POINT_VALUE: float = 2500
    MINI_SYMBOL: str = "QI"
    MINI_MARGIN_INITIAL: float = 8250
    MINI_MARGIN_MAINTENANCE: float = 7500
    
    # Micro contract (SIL) - most accessible for retail
    MICRO_CONTRACT_SIZE: int = 1000
    MICRO_TICK_SIZE: float = 0.005
    MICRO_TICK_VALUE: float = 5.00
    MICRO_POINT_VALUE: float = 1000
    MICRO_SYMBOL: str = "SIL"
    MICRO_MARGIN_INITIAL: float = 3300
    MICRO_MARGIN_MAINTENANCE: float = 3000
    
    TRADING_HOURS: str = "Sunday 6pm - Friday 5pm ET"


class ContractType(Enum):
    """Available contract types"""
    FULL = "SI"      # 5000 oz
    MINI = "QI"      # 2500 oz  
    MICRO = "SIL"    # 1000 oz


class GreeksCalculator:
    """Black-76 model for options on silver futures (for hedging)"""
    
    @staticmethod
    def calculate_greeks(F: float, K: float, T: float, r: float, sigma: float,
                        option_type: str = 'put', contract_size: int = 1000) -> Dict:
        """Calculate Greeks for silver futures options"""
        if T <= 0:
            return {'price': 0, 'delta': 0, 'gamma': 0, 'theta': 0, 'vega': 0}
        
        sqrt_T = np.sqrt(T)
        d1 = (np.log(F / K) + (sigma**2 / 2) * T) / (sigma * sqrt_T)
        d2 = d1 - sigma * sqrt_T
        
        discount = np.exp(-r * T)
        pdf_d1 = norm.pdf(d1)
        
        if option_type == 'call':
            price = discount * (F * norm.cdf(d1) - K * norm.cdf(d2))
            delta = discount * norm.cdf(d1)
        else:
            price = discount * (K * norm.cdf(-d2) - F * norm.cdf(-d1))
            delta = -discount * norm.cdf(-d1)
        
        gamma = discount * pdf_d1 / (F * sigma * sqrt_T)
        theta = (-discount * F * pdf_d1 * sigma / (2 * sqrt_T)) / 365
        vega = F * discount * pdf_d1 * sqrt_T / 100
        
        return {
            'price': price * contract_size,
            'delta': delta,
            'gamma': gamma,
            'theta': theta * contract_size,
            'vega': vega * contract_size
        }


class FuturesPositionSizer:
    """Position sizing for COMEX Silver Futures"""
    
    def __init__(self, account_size: float = 100000, max_risk_pct: float = 0.02):
        self.account_size = account_size
        self.max_risk_pct = max_risk_pct
        self.specs = ComexSilverSpecs()
    
    def calculate_position(self, current_price: float, stop_loss_pct: float,
                          signal_confidence: float, realized_vol: float,
                          contract_type: str = "MICRO") -> Dict:
        """Calculate optimal position size with full risk metrics"""
        
        # Get contract specs
        if contract_type == "FULL":
            contract_size = self.specs.FULL_CONTRACT_SIZE
            margin = self.specs.FULL_MARGIN_INITIAL
            symbol = self.specs.FULL_SYMBOL
        elif contract_type == "MINI":
            contract_size = self.specs.MINI_CONTRACT_SIZE
            margin = self.specs.MINI_MARGIN_INITIAL
            symbol = self.specs.MINI_SYMBOL
        else:
            contract_size = self.specs.MICRO_CONTRACT_SIZE
            margin = self.specs.MICRO_MARGIN_INITIAL
            symbol = self.specs.MICRO_SYMBOL
        
        # Risk per contract
        price_risk = current_price * stop_loss_pct
        risk_per_contract = price_risk * contract_size
        
        # Max contracts by risk
        max_risk_dollars = self.account_size * self.max_risk_pct
        risk_based_contracts = int(max_risk_dollars / risk_per_contract)
        
        # Max contracts by margin (50% utilization max)
        max_margin = self.account_size * 0.5
        margin_based_contracts = int(max_margin / margin)
        
        # Volatility adjustment
        vol_mult = min(1.0, 0.25 / realized_vol) if realized_vol > 0 else 1.0
        
        # Confidence adjustment
        conf_mult = 0.5 + (signal_confidence - 0.5) * 1.5
        conf_mult = max(0.5, min(conf_mult, 1.25))
        
        # Take minimum of all constraints
        base_contracts = min(risk_based_contracts, margin_based_contracts)
        final_contracts = max(1, int(base_contracts * vol_mult * conf_mult))
        
        # Calculate VaR (30-day, 95%)
        notional = current_price * contract_size * final_contracts
        period_vol = realized_vol * np.sqrt(30 / 252)
        var_95 = notional * norm.ppf(0.95) * period_vol
        cvar_95 = notional * period_vol * norm.pdf(norm.ppf(0.95)) / 0.05
        
        return {
            'symbol': symbol,
            'contract_type': contract_type,
            'contract_size_oz': contract_size,
            'recommended_contracts': final_contracts,
            'margin_per_contract': margin,
            'total_margin': margin * final_contracts,
            'margin_pct_account': (margin * final_contracts) / self.account_size * 100,
            'risk_per_contract': risk_per_contract,
            'total_risk': risk_per_contract * final_contracts,
            'risk_pct_account': (risk_per_contract * final_contracts) / self.account_size * 100,
            'notional_exposure': notional,
            'var_30d_95': var_95,
            'cvar_30d_95': cvar_95,
            'var_pct_account': var_95 / self.account_size * 100,
            'sizing_factors': {
                'risk_based_max': risk_based_contracts,
                'margin_based_max': margin_based_contracts,
                'vol_multiplier': vol_mult,
                'confidence_multiplier': conf_mult
            }
        }


def generate_futures_trade_ticket(signal_info: Dict, position_info: Dict, 
                                   version: str = "V3.4") -> str:
    """Generate formatted trade ticket for Charles Schwab"""
    
    if signal_info.get('signal') in ['⏸️ WAIT', '🚫 BLOCKED', 'NO TRADE', None]:
        return """
╔══════════════════════════════════════════════════════════════════════════════╗
║                        NO TRADE RECOMMENDED                                   ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  Signal: {signal}
║  Reason: {reason}
╚══════════════════════════════════════════════════════════════════════════════╝
""".format(signal=signal_info.get('signal', 'WAIT'), 
           reason=signal_info.get('reason', 'Conditions not met'))
    
    is_short = 'SHORT' in str(signal_info.get('signal', ''))
    direction = "SHORT" if is_short else "LONG"
    action = "SELL TO OPEN" if is_short else "BUY TO OPEN"
    exit_action = "BUY TO CLOSE" if is_short else "SELL TO CLOSE"
    
    current_price = signal_info.get('price', 0)
    stop_pct = signal_info.get('stop_loss_pct', 0.04)
    target_pct = signal_info.get('take_profit_pct', 0.08)
    
    if is_short:
        stop_price = current_price * (1 + stop_pct)
        target_price = current_price * (1 - target_pct)
    else:
        stop_price = current_price * (1 - stop_pct)
        target_price = current_price * (1 + target_pct)
    
    reward = abs(target_price - current_price) * position_info.get('contract_size_oz', 1000)
    risk = abs(stop_price - current_price) * position_info.get('contract_size_oz', 1000)
    rr_ratio = reward / risk if risk > 0 else 0
    
    ticket = f"""
╔══════════════════════════════════════════════════════════════════════════════╗
║                   COMEX SILVER FUTURES TRADE TICKET                          ║
║                        Charles Schwab Execution                               ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  Strategy Version: {version}
║  Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
╠══════════════════════════════════════════════════════════════════════════════╣
║  
║  ┌─────────────────────────────────────────────────────────────────────────┐
║  │  SIGNAL: {'🔴' if is_short else '🟢'} {direction:6}                                               │
║  │  Confidence: {signal_info.get('probability', 0)*100:.1f}%                                           │
║  │  Model Agreement: {signal_info.get('model_agreement', 0)}/3                                       │
║  │  Regime: {signal_info.get('regime', 'Unknown'):<30}                    │
║  └─────────────────────────────────────────────────────────────────────────┘
║  
╠══════════════════════════════════════════════════════════════════════════════╣
║  ORDER ENTRY:
║  ─────────────────────────────────────────────────────────────────────────────
║    Action:     {action}
║    Symbol:     {position_info.get('symbol', 'SIL')} (COMEX Silver {position_info.get('contract_type', 'Micro')})
║    Quantity:   {position_info.get('recommended_contracts', 1)} contract(s)
║    Order Type: LIMIT @ ${current_price:.2f} (or MARKET)
║  
╠══════════════════════════════════════════════════════════════════════════════╣
║  RISK MANAGEMENT ORDERS:
║  ─────────────────────────────────────────────────────────────────────────────
║    STOP LOSS:   {exit_action} @ ${stop_price:.2f} ({stop_pct*100:.1f}% {'above' if is_short else 'below'})
║    TAKE PROFIT: {exit_action} @ ${target_price:.2f} ({target_pct*100:.1f}% {'below' if is_short else 'above'})
║    MAX HOLD:    30 calendar days
║  
╠══════════════════════════════════════════════════════════════════════════════╣
║  POSITION DETAILS:
║  ─────────────────────────────────────────────────────────────────────────────
║    Contract Size:     {position_info.get('contract_size_oz', 1000):,} troy oz
║    Notional Exposure: ${position_info.get('notional_exposure', 0):,.0f}
║    Margin Required:   ${position_info.get('total_margin', 0):,.0f} ({position_info.get('margin_pct_account', 0):.1f}% of account)
║  
╠══════════════════════════════════════════════════════════════════════════════╣
║  RISK ANALYSIS:
║  ─────────────────────────────────────────────────────────────────────────────
║    Risk per Contract:   ${position_info.get('risk_per_contract', 0):,.0f}
║    Total Risk:          ${position_info.get('total_risk', 0):,.0f} ({position_info.get('risk_pct_account', 0):.1f}% of account)
║    Reward per Contract: ${reward / position_info.get('recommended_contracts', 1):,.0f}
║    Total Reward:        ${reward * position_info.get('recommended_contracts', 1):,.0f}
║    Reward:Risk Ratio:   {rr_ratio:.1f}:1
║  
╠══════════════════════════════════════════════════════════════════════════════╣
║  VALUE AT RISK (30-day, 95% confidence):
║  ─────────────────────────────────────────────────────────────────────────────
║    VaR:  ${position_info.get('var_30d_95', 0):,.0f} ({position_info.get('var_pct_account', 0):.1f}% of account)
║    CVaR: ${position_info.get('cvar_30d_95', 0):,.0f}
║  
╚══════════════════════════════════════════════════════════════════════════════╝
"""
    return ticket


def generate_options_hedge_analysis(current_price: float, position_contracts: int,
                                    realized_vol: float, is_short: bool = True) -> str:
    """Generate options hedging analysis for the futures position"""
    
    # For a SHORT futures, buy CALLS to hedge
    # For a LONG futures, buy PUTS to hedge
    hedge_type = "CALL" if is_short else "PUT"
    
    # Calculate Greeks for ATM and OTM options
    strikes = [
        current_price * 0.95,  # 5% OTM
        current_price,          # ATM
        current_price * 1.05,  # 5% ITM (for short hedge)
    ]
    
    greeks_calc = GreeksCalculator()
    
    analysis = f"""
╔══════════════════════════════════════════════════════════════════════════════╗
║                    OPTIONS HEDGE ANALYSIS                                     ║
║                    (For {'SHORT' if is_short else 'LONG'} Futures Position)                                    ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  Recommendation: Buy {hedge_type}S to hedge {'upside' if is_short else 'downside'} risk
║  Current Silver: ${current_price:.2f}
║  Position: {position_contracts} contract(s)
║  Implied Vol: {realized_vol*100:.1f}%
╠══════════════════════════════════════════════════════════════════════════════╣
║  {hedge_type} OPTIONS (30 DTE):
║  ─────────────────────────────────────────────────────────────────────────────
"""
    
    for strike in strikes:
        greeks = greeks_calc.calculate_greeks(
            F=current_price, K=strike, T=30/365, r=0.05,
            sigma=realized_vol, option_type=hedge_type.lower(), contract_size=1000
        )
        otm_pct = ((strike - current_price) / current_price) * 100
        
        analysis += f"""║    Strike ${strike:.2f} ({otm_pct:+.1f}%):
║      Price: ${greeks['price']:,.0f} | Delta: {greeks['delta']:.3f} | Theta: ${greeks['theta']:.0f}/day
"""
    
    analysis += """╠══════════════════════════════════════════════════════════════════════════════╣
║  HEDGE COST CONSIDERATION:
║  Buying protective options reduces max loss but costs premium.
║  Consider if signal confidence < 70% or position size > 2 contracts.
╚══════════════════════════════════════════════════════════════════════════════╝
"""
    return analysis


# ══════════════════════════════════════════════════════════════════════
# CONFIGURATION - SHARPE OPTIMIZED
# ══════════════════════════════════════════════════════════════════════

class Config:
    """Central configuration - V3.4 BALANCED (High Sharpe + Smart Shorts)"""
    
    # Data parameters
    START_DATE = "2016-02-24"
    END_DATE = None  # Will use current date
    
    # Trading parameters
    LOOKAHEAD_DAYS = 30
    THRESHOLD_PCT = 0.02
    
    # Walk-forward parameters
    TRAIN_YEARS = 2
    TEST_MONTHS = 3
    MIN_TRAIN_SAMPLES = 300
    WINDOW_SIZE = 60
    ROLLING_LOOKBACK = 126
    
    # Ensemble weights - balanced
    LSTM_WEIGHT = 0.15
    RF_WEIGHT = 0.425
    XGB_WEIGHT = 0.425
    
    # BALANCED THRESHOLDS
    BASE_THRESHOLD = 0.60
    UPTREND_THRESHOLD = 0.55
    DOWNTREND_THRESHOLD = 0.95  # Block longs
    HIGH_VOL_THRESHOLD = 0.60
    
    # Risk management - balanced
    STOP_LOSS_PCT = 0.04
    TAKE_PROFIT_PCT = 0.08
    MAX_POSITION_CONTRACTS = 4  # Middle ground
    ACCOUNT_SIZE = 100000
    
    # Model parameters
    LSTM_EPOCHS = 100    # raised from 30; patience=7 on val_loss will fire well before this
    LSTM_BATCH_SIZE = 64  # 32→64: halves gradient steps/epoch, faster with early stopping
    RF_ESTIMATORS = 100   # 150→100: variance converges well before 150 on 500 samples
    RF_MAX_DEPTH = 7      # 10→7: reduces overfitting and tree build time on small windows
    XGB_ESTIMATORS = 75
    
    # Feature selection
    RF_TOP_FEATURES = 50
    USE_LASSO_SELECTION = True
    USE_PCA = False
    PCA_VARIANCE = 0.95
    CORRELATION_THRESHOLD = 0.90
    XGB_REG_ALPHA = 0.1
    XGB_REG_LAMBDA = 1.0
    
    # Quality filters
    MIN_MODEL_AGREEMENT = 2     # 2/3 models must agree
    MIN_SIGNAL_CONFIDENCE = 0.52  # Slightly lower than V3.2
    MAX_DAILY_TRADES = 2        # Allow some flexibility


# ══════════════════════════════════════════════════════════════════════
# MEAN REVERSION DETECTION
# ══════════════════════════════════════════════════════════════════════

class MeanReversionDetector:
    """Quantitative mean reversion detection"""
    
    def __init__(self, lookback_short: int = 20, lookback_long: int = 60):
        self.lookback_short = lookback_short
        self.lookback_long = lookback_long
    
    def hurst_exponent(self, series: pd.Series, max_lag: int = 20) -> float:
        """Calculate Hurst exponent using R/S analysis"""
        from scipy import stats
        
        series = series.dropna()
        if len(series) < max_lag * 2:
            return 0.5
        
        lags = range(2, max_lag)
        tau = []
        rs_values = []
        
        for lag in lags:
            chunks = len(series) // lag
            if chunks < 1:
                continue
            
            rs_chunk = []
            for i in range(chunks):
                chunk = series.iloc[i*lag:(i+1)*lag]
                if len(chunk) < 2:
                    continue
                
                mean_adj = chunk - chunk.mean()
                cumdev = mean_adj.cumsum()
                R = cumdev.max() - cumdev.min()
                S = chunk.std()
                
                if S > 0:
                    rs_chunk.append(R / S)
            
            if rs_chunk:
                tau.append(lag)
                rs_values.append(np.mean(rs_chunk))
        
        if len(tau) < 3:
            return 0.5
        
        try:
            log_tau = np.log(tau)
            log_rs  = np.log(rs_values)
            slope, _, r_value, _, _ = stats.linregress(log_tau, log_rs)
            # R² guard: R/S analysis only yields a meaningful Hurst estimate when
            # the log-log relationship is actually linear (R² ≥ 0.80).  A weak fit
            # means the series doesn't follow clean R/S scaling — return the
            # random-walk default rather than a noisy slope artefact.
            # NOTE ON ABSOLUTE SCALE: uncorrected small-sample R/S reads high (a
            # random walk measures ~0.6-0.7 at these lags, not 0.5), so HURST_*
            # is trustworthy as a RELATIVE signal (higher = more trending) but
            # not as an absolute persistence measure.  An Anis-Lloyd correction
            # was trialed and removed — at these short lags it over-subtracted
            # and destroyed trend/RW separation, which is worse than a known,
            # consistent bias.  Downstream consumers (trend_score centering, the
            # relative ranking in feature selection) treat it relatively, so the
            # bias is harmless where it is used.  Ceiling at 0.99 avoids a
            # numeric 1.0 saturating trend_score.
            if r_value ** 2 < 0.80:
                return 0.5
            return float(np.clip(slope, 0.0, 0.99))
        except:
            return 0.5
    
    def half_life(self, series: pd.Series) -> float:
        """Calculate half-life of mean reversion"""
        series = series.dropna()
        if len(series) < 20:
            return np.inf
        
        lag = series.shift(1)
        delta = series - lag
        lag = lag.iloc[1:]
        delta = delta.iloc[1:]
        
        try:
            X = np.column_stack([np.ones(len(lag)), lag.values])
            y = delta.values
            beta = np.linalg.lstsq(X, y, rcond=None)[0]
            theta = -beta[1]
            
            if theta <= 0:
                return np.inf
            
            return max(1, np.log(2) / theta)
        except:
            return np.inf
    
    def variance_ratio(self, series: pd.Series, lag: int = 10) -> float:
        """Variance ratio test"""
        series = series.dropna()
        if len(series) < lag * 2:
            return 1.0
        
        returns_1 = series.diff(1).dropna()
        returns_k = series.diff(lag).dropna()
        
        if len(returns_1) < 10 or len(returns_k) < 10:
            return 1.0
        
        var_1 = returns_1.var()
        var_k = returns_k.var()
        
        if var_1 == 0:
            return 1.0
        
        return var_k / (lag * var_1)
    
    def z_score(self, series: pd.Series, lookback: int = 60) -> float:
        """Calculate current z-score"""
        if len(series) < lookback:
            return 0.0
        
        rolling_mean = series.rolling(lookback).mean()
        rolling_std = series.rolling(lookback).std()
        
        current_mean = rolling_mean.iloc[-1]
        current_std = rolling_std.iloc[-1]
        
        if current_std == 0 or pd.isna(current_std):
            return 0.0
        
        return (series.iloc[-1] - current_mean) / current_std
    
    def calculate_all_features(self, price_series: pd.Series) -> Dict[str, float]:
        """Calculate all mean reversion features"""
        log_prices = np.log(price_series.replace(0, np.nan)).dropna()
        # R/S Hurst must run on the INCREMENTS (log returns), not price levels.
        # On a near-monotonic price level series, cumulative deviations scale
        # ~linearly with the window, so the R/S slope saturates at H→1 and the
        # estimate pins at the 0.99 clip ceiling — feeding a degenerate value
        # into trend_score.  Classical R/S (Hurst 1951, Lo 1991) analyses the
        # increment series; H>0.5 then genuinely means persistent RETURNS.
        # Half-life and variance ratio stay on levels — both are defined on
        # the level series (OU regression on levels; VR uses level diffs
        # internally).
        log_returns = log_prices.diff().dropna()
        
        features = {}
        
        # Hurst exponents — on log-return increments (tail sized to match the
        # old level-series lookbacks: 60 and 180 obs).
        # NOTE: uncorrected small-sample R/S reads HIGH — a pure random walk
        # measures ~0.70-0.72 at these lengths, not 0.50.  These are kept as
        # model FEATURES and diagnostics, but they no longer gate MR_SIGNAL:
        # measured on simulated ground truth, a fixed H<0.45 gate fired on
        # 0% of strongly mean-reverting (OU) series — it was permanently shut.
        features['HURST_20'] = self.hurst_exponent(log_returns.tail(60), max_lag=15)
        features['HURST_60'] = self.hurst_exponent(log_returns.tail(180), max_lag=30)
        features['MEAN_REV_SCORE_20'] = 1 - 2 * features['HURST_20']
        features['MEAN_REV_SCORE_60'] = 1 - 2 * features['HURST_60']
        
        # Half-life (OU regression on LEVELS — measures price mean reversion)
        hl = self.half_life(log_prices.tail(120))
        features['HALF_LIFE'] = min(hl, 100)
        features['HALF_LIFE_FAST'] = 1 if hl < 10 else 0
        features['HALF_LIFE_SLOW'] = 1 if hl > 30 else 0
        
        # Variance ratios (on LEVELS — VR<1 means level moves partially undo).
        # WINDOWED to the same 120-bar tail as half_life: previously these ran
        # on the FULL history since inception, so on a 10-year dataset VR was
        # a near-constant decade-scale average — useless as a current-regime
        # gate and stale as a model feature.
        _vr_window = log_prices.tail(120)
        features['VAR_RATIO_5'] = self.variance_ratio(_vr_window, lag=5)
        features['VAR_RATIO_10'] = self.variance_ratio(_vr_window, lag=10)
        features['VAR_RATIO_20'] = self.variance_ratio(_vr_window, lag=20)
        avg_vr = (features['VAR_RATIO_5'] + features['VAR_RATIO_10'] + features['VAR_RATIO_20']) / 3
        features['VAR_RATIO_AVG'] = avg_vr
        features['VR_MEAN_REVERTING'] = 1 if avg_vr < 0.9 else 0
        features['VR_TRENDING'] = 1 if avg_vr > 1.1 else 0
        
        # ── MR GATE: level-based joint test (replaces dead H<0.45 gate) ──────
        # Increments-Hurst measures RETURN persistence, not PRICE reversion —
        # wrong object for this flag.  HALF_LIFE and VAR_RATIO are computed on
        # levels and are the right tools.  Each alone over-fires (OU theta on
        # a finite random walk is almost always negative: HL<30 alone fired on
        # 63% of simulated random walks).  Joint gate re-calibrated for the
        # 120-bar VR window (80 sims/regime): trending 3.8%, random walk
        # 11.2%, slow OU 21.2%, strong OU 87.5%.  Expect this to be 0 MOST
        # of the time on silver — that is
        # correct behaviour for a mostly-trending metal, not a bug.
        features['IS_MEAN_REVERTING'] = 1 if (hl < 10 and avg_vr < 0.72) else 0
        # Trending flag likewise moves to the level-based detector: with the
        # R/S small-sample bias, H>0.55 was true ~always and pinned MR_REGIME.
        features['IS_TRENDING'] = 1 if avg_vr > 1.05 else 0
        
        # Z-scores
        features['ZSCORE_20'] = self.z_score(price_series, 20)
        features['ZSCORE_60'] = self.z_score(price_series, 60)
        features['ZSCORE_120'] = self.z_score(price_series, 120)
        features['ZSCORE_EXTREME_HIGH'] = 1 if features['ZSCORE_60'] > 2.0 else 0
        features['ZSCORE_EXTREME_LOW'] = 1 if features['ZSCORE_60'] < -2.0 else 0
        features['ZSCORE_OVERBOUGHT'] = 1 if features['ZSCORE_60'] > 1.5 else 0
        features['ZSCORE_OVERSOLD'] = 1 if features['ZSCORE_60'] < -1.5 else 0
        
        # Composite signal — strength now derives from the variance ratio,
        # not (1-2H): the old term was capped at 0.10 by the very gate that
        # enabled it, so even a fired signal could not clear the |MR|>0.3
        # filter downstream.  1-VR at the strong-OU median (VR=0.54) gives
        # 0.46, so z=1.5 -> |MR_SIGNAL|≈0.68: a fired gate now MEANS something.
        if features['IS_MEAN_REVERTING']:
            mr_strength = float(np.clip(1.0 - avg_vr, 0.0, 1.0))
            z = features['ZSCORE_60']
            features['MR_SIGNAL'] = np.clip(-z * mr_strength, -1, 1)
        else:
            features['MR_SIGNAL'] = 0.0
        
        # Regime
        if features['IS_MEAN_REVERTING'] and abs(features['ZSCORE_60']) > 1.5:
            features['MR_REGIME'] = 1
        elif features['IS_TRENDING']:
            features['MR_REGIME'] = -1
        else:
            features['MR_REGIME'] = 0
        
        return features


def add_mean_reversion_features(df: pd.DataFrame) -> Tuple[pd.DataFrame, List[str]]:
    """Add mean reversion features to DataFrame"""
    print("\n" + "-"*70)
    print("[QUANT] Mean Reversion Features...")
    
    detector = MeanReversionDetector()
    features_list = []
    price_col = 'SI=F_Close'
    
    if price_col not in df.columns:
        print("  ⚠ Price column not found")
        return df, []
    
    mr_feature_names = [
        'HURST_20', 'HURST_60', 'MEAN_REV_SCORE_20', 'MEAN_REV_SCORE_60',
        'IS_MEAN_REVERTING', 'IS_TRENDING', 'HALF_LIFE', 'HALF_LIFE_FAST',
        'HALF_LIFE_SLOW', 'VAR_RATIO_5', 'VAR_RATIO_10', 'VAR_RATIO_20',
        'VAR_RATIO_AVG', 'VR_MEAN_REVERTING', 'VR_TRENDING', 'ZSCORE_20',
        'ZSCORE_60', 'ZSCORE_120', 'ZSCORE_EXTREME_HIGH', 'ZSCORE_EXTREME_LOW',
        'ZSCORE_OVERBOUGHT', 'ZSCORE_OVERSOLD', 'MR_SIGNAL', 'MR_REGIME'
    ]
    
    for feat in mr_feature_names:
        df[feat] = np.nan
    
    min_history = 200
    print(f"  Calculating mean reversion features...")
    
    for i in range(min_history, len(df)):
        price_history = df[price_col].iloc[:i+1]
        features = detector.calculate_all_features(price_history)
        
        for feat_name, feat_value in features.items():
            df.loc[df.index[i], feat_name] = feat_value
    
    # No bfill — backfilling pulls FUTURE feature values into the warmup rows
    # (same species of leak as the old full-sample EGARCH fit).  Warmup rows
    # instead get causal neutral defaults: H=0.5 random walk, VR=1.0, etc.
    _mr_neutral = {
        'HURST_20': 0.5, 'HURST_60': 0.5,
        'VAR_RATIO_5': 1.0, 'VAR_RATIO_10': 1.0, 'VAR_RATIO_20': 1.0,
        'VAR_RATIO_AVG': 1.0,
        'HALF_LIFE': 100.0,
    }
    for feat in mr_feature_names:
        df[feat] = df[feat].fillna(_mr_neutral.get(feat, 0))
        features_list.append(feat)
    
    # Historical fire-rate report: sanity-check the new level-based gate on
    # REAL data.  Expected order of magnitude from simulation: ~0-10%.
    # 0.0% would mean the gate is still dead; >25% would mean it over-fires.
    _valid = df.iloc[min_history:]
    if len(_valid) > 0:
        _fire = 100.0 * (_valid['IS_MEAN_REVERTING'] > 0).mean()
        _live = 100.0 * (_valid['MR_SIGNAL'].abs() > 1e-9).mean()
        _strong = 100.0 * (_valid['MR_SIGNAL'].abs() > 0.3).mean()
        print(f"  MR gate fire-rate: {_fire:.1f}% of bars | MR_SIGNAL≠0: "
              f"{_live:.1f}% | |MR_SIGNAL|>0.3 (filter-relevant): {_strong:.1f}%")
    
    if len(df) > 0:
        hurst = df['HURST_60'].iloc[-1]
        mr_signal = df['MR_SIGNAL'].iloc[-1]
        z_score = df['ZSCORE_60'].iloc[-1]
        
        regime_str = "MEAN REVERTING" if hurst < 0.45 else ("TRENDING" if hurst > 0.55 else "RANDOM WALK")
        
        print(f"\n  → Hurst (60d):     {hurst:.3f} → {regime_str}")
        print(f"  → Z-Score (60d):   {z_score:+.2f}")
        print(f"  → MR Signal:       {mr_signal:+.2f}")
        print(f"  ✓ Added {len(features_list)} mean reversion features")
    
    return df, features_list


# ══════════════════════════════════════════════════════════════════════
# ENHANCED REGIME FILTER - SHARPE OPTIMIZED
# ══════════════════════════════════════════════════════════════════════

class EnhancedRegimeFilter:
    """Enhanced regime filtering with model agreement for higher Sharpe"""
    
    @staticmethod
    def filter_signal(signal_prob: float, 
                      regime: MarketRegime,
                      direction: TradeDirection = TradeDirection.LONG,
                      mr_features: Optional[Dict] = None,
                      model_agreement: int = 0) -> Tuple[bool, float, str]:
        """Filter trading signal based on regime rules and model agreement"""
        config = REGIME_CONFIGS[regime]
        
        # Check direction allowed
        if direction == TradeDirection.LONG and not config.allow_long:
            return False, 0.0, f"BLOCKED: Longs not allowed in {regime.value}"
        
        if direction == TradeDirection.SHORT and not config.allow_short:
            return False, 0.0, f"BLOCKED: Shorts not allowed in {regime.value}"
        
        # Check model agreement (NEW for Sharpe optimization)
        if model_agreement < config.min_model_agreement:
            return False, signal_prob, f"LOW AGREEMENT: {model_agreement}/3 models (need {config.min_model_agreement})"
        
        # Check threshold
        threshold = config.long_threshold if direction == TradeDirection.LONG else config.short_threshold
        
        if signal_prob < threshold:
            return False, signal_prob, f"Below threshold ({signal_prob:.3f} < {threshold:.3f})"
        
        # Minimum confidence check
        if signal_prob < Config.MIN_SIGNAL_CONFIDENCE:
            return False, signal_prob, f"Below min confidence ({signal_prob:.3f} < {Config.MIN_SIGNAL_CONFIDENCE})"
        
        # Mean reversion boost — symmetric in both directions (Run 13)
        # When the market is statistically mean-reverting, price extended from
        # its 60d mean is a structural edge:
        #   LONG:  MR_SIGNAL > 0.3  OR  (IS_MEAN_REVERTING AND ZSCORE_60 < -1.5)
        #          → price below mean → buy the reversion
        #   SHORT: MR_SIGNAL < -0.3 OR  (IS_MEAN_REVERTING AND ZSCORE_60 >  1.5)
        #          → price above mean → short the reversion
        # 10% boost only applies to signals that already cleared threshold.
        if config.use_mean_reversion and mr_features:
            mr_signal = mr_features.get('MR_SIGNAL', 0)
            is_mr     = bool(mr_features.get('IS_MEAN_REVERTING', 0))
            zscore    = float(mr_features.get('ZSCORE_60', 0))
            if direction == TradeDirection.LONG:
                if mr_signal > 0.3 or (is_mr and zscore < -1.5):
                    signal_prob = min(signal_prob * 1.1, 1.0)
            elif direction == TradeDirection.SHORT:
                if mr_signal < -0.3 or (is_mr and zscore > 1.5):
                    signal_prob = min(signal_prob * 1.1, 1.0)
        
        return True, signal_prob, "PASS"
    
    @staticmethod
    def get_position_size(base_contracts: int, regime: MarketRegime, 
                         signal_prob: float, model_agreement: int) -> int:
        """Adjust position size - V3.4 balanced approach"""
        config = REGIME_CONFIGS[regime]
        adjusted = base_contracts * config.position_size_mult
        
        # Scale by confidence and agreement
        if signal_prob > 0.70 and model_agreement == 3:
            adjusted *= 1.15  # Modest boost for high conviction
        elif signal_prob > 0.60:
            adjusted *= 0.9
        else:
            adjusted *= 0.7
        
        return max(1, min(int(adjusted), Config.MAX_POSITION_CONTRACTS))


# ══════════════════════════════════════════════════════════════════════
# BLACK'S MODEL FOR COMEX FUTURES OPTIONS
# ══════════════════════════════════════════════════════════════════════

class BlackScholesFutures:
    """Black's Model (1976) for futures options"""
    
    def __init__(self, F, K, T, r, sigma, option_type='call'):
        self.F = F
        self.K = K
        self.T = T
        self.r = r
        self.sigma = sigma
        self.option_type = option_type.lower()
        
    def d1(self):
        if self.T <= 0:
            return 0
        return (np.log(self.F / self.K) + (0.5 * self.sigma**2) * self.T) / (self.sigma * np.sqrt(self.T))
    
    def d2(self):
        if self.T <= 0:
            return 0
        return self.d1() - self.sigma * np.sqrt(self.T)
    
    def price(self):
        if self.T <= 0:
            if self.option_type == 'call':
                return max(0, self.F - self.K)
            else:
                return max(0, self.K - self.F)
        
        d1 = self.d1()
        d2 = self.d2()
        discount = np.exp(-self.r * self.T)
        
        if self.option_type == 'call':
            return discount * (self.F * norm.cdf(d1) - self.K * norm.cdf(d2))
        else:
            return discount * (self.K * norm.cdf(-d2) - self.F * norm.cdf(-d1))
    
    def delta(self):
        if self.T <= 0:
            if self.option_type == 'call':
                return 1.0 if self.F > self.K else 0.0
            else:
                return -1.0 if self.F < self.K else 0.0
        
        d1 = self.d1()
        discount = np.exp(-self.r * self.T)
        
        if self.option_type == 'call':
            return discount * norm.cdf(d1)
        else:
            return -discount * norm.cdf(-d1)


class COMEXSilverSpecs:
    CONTRACT_SIZE = 5000
    TICK_SIZE = 0.005
    TICK_VALUE = 25.0


# ══════════════════════════════════════════════════════════════════════
# CUSTOM LOSS FUNCTIONS
# ══════════════════════════════════════════════════════════════════════

def focal_loss(gamma=2., alpha=0.25):
    """Focal loss for handling class imbalance"""
    def focal_loss_fixed(y_true, y_pred):
        epsilon = K.epsilon()
        y_pred = K.clip(y_pred, epsilon, 1. - epsilon)
        p_t = y_true * y_pred + (1 - y_true) * (1 - y_pred)
        focal_weight = (1 - p_t) ** gamma
        return -K.mean(alpha * focal_weight * K.log(p_t))
    return focal_loss_fixed


# ══════════════════════════════════════════════════════════════════════
# MARKET REGIME DETECTION
# ══════════════════════════════════════════════════════════════════════

class MarketRegimeDetector:
    """Detect market regime"""
    
    @staticmethod
    def detect_regime(df, lookback=60):
        if len(df) < lookback:
            return 'CONSOLIDATION'

        # ── PRICE / VOLATILITY SIGNALS ────────────────────────────────────────
        close         = df['SI=F_Close']
        returns       = close.pct_change(lookback).iloc[-1]
        daily_rets    = close.pct_change()
        vol_series_d  = daily_rets.rolling(20).std()
        vol_series_a  = vol_series_d * np.sqrt(252)
        volatility    = vol_series_a.iloc[-1]
        vol_pct       = vol_series_a.rolling(252, min_periods=60).rank(pct=True).iloc[-1]
        sma_20        = close.rolling(20).mean().iloc[-1]
        sma_50        = close.rolling(50).mean().iloc[-1]
        current_price = close.iloc[-1]

        # ── VOL-NORMALISED RETURN (Run 9) ─────────────────────────────────────
        # Raw 60d return threshold is price-level dependent. Use z-score of
        # 60d return relative to expected period volatility instead.
        period_vol   = vol_series_d.iloc[-1] * np.sqrt(lookback)
        norm_returns = float(returns / period_vol) if period_vol > 1e-6 else 0.0
        norm_returns = float(np.clip(norm_returns, -5.0, 5.0))

        # ── NW CHANNEL FEATURES (Run 9) ───────────────────────────────────────
        nw_position = (df['NW_Position'].iloc[-1]
                       if 'NW_Position' in df.columns else 0.5)
        nw_position = float(np.clip(nw_position, 0.0, 1.0))

        nw_bw_pct     = 0.5
        nw_compressed = False
        nw_wide       = False
        if 'NW_BAND_WIDTH' in df.columns:
            bw_series     = df['NW_BAND_WIDTH']
            nw_bw_pct     = float(bw_series.rolling(252, min_periods=60)
                                  .rank(pct=True).iloc[-1])
            nw_bw_pct     = float(np.nan_to_num(nw_bw_pct, nan=0.5))
            nw_compressed = nw_bw_pct < 0.30
            nw_wide       = nw_bw_pct > 0.65

        nw_upper_extended = nw_position >= 0.78
        nw_lower_extended = nw_position <= 0.22

        # ── MARKET STRUCTURE TREND SCORE ──────────────────────────────────────
        mr_features_present = all(c in df.columns for c in
                                  ['HURST_60', 'AUTOCORR_1', 'VAR_RATIO_AVG', 'IS_MEAN_REVERTING'])

        if mr_features_present:
            hurst      = df['HURST_60'].iloc[-1]
            autocorr   = df['AUTOCORR_1'].iloc[-1]
            var_ratio  = df['VAR_RATIO_AVG'].iloc[-1]
            is_mr      = df['IS_MEAN_REVERTING'].iloc[-1]

            # Uncorrected small-sample R/S is biased high: a random walk reads
            # ~0.70 at these lags, not 0.50.  Center on that empirical median so
            # hurst_score isn't a near-constant positive offset (which made the
            # trend gates rubber stamps and classified chop as trend).
            hurst_score    = np.clip((hurst - 0.70) * 5,    -1.0, 1.0)
            autocorr_score = np.clip(autocorr * 3,          -1.0, 1.0)
            vr_score       = np.clip((var_ratio - 1.0) * 5, -1.0, 1.0)
            mr_penalty     = -0.4 if is_mr else 0.0

            trend_score = float(np.clip(
                0.45 * hurst_score +
                0.25 * autocorr_score +
                0.25 * vr_score +
                0.05 * mr_penalty,
                -1.0, 1.0))
        else:
            adx = df['SI=F_ADX'].iloc[-1] if 'SI=F_ADX' in df.columns else 25
            trend_score = float(np.clip((adx - 25) / 25, -1.0, 1.0))

        # ── COMPOSITE REGIME SIGNALS ───────────────────────────────────────────
        _extreme_vol  = vol_pct > 0.92 or volatility > 0.50
        _elevated_vol = vol_pct > 0.85 or volatility > 0.40

        _up_primary   = (norm_returns > 1.5 and returns > 0.08
                         and current_price > sma_20 > sma_50
                         and trend_score > 0.20)
        _up_secondary = (returns > 0.10 and current_price > sma_20 > sma_50
                         and trend_score > 0.0 and nw_upper_extended)

        _dn_primary   = (norm_returns < -1.5 and returns < -0.08
                         and current_price < sma_20 < sma_50
                         and trend_score > 0.20)
        _dn_secondary = (returns < -0.10 and current_price < sma_20 < sma_50
                         and trend_score > 0.0 and nw_lower_extended)

        _strong_up   = _up_primary   or _up_secondary
        _strong_down = _dn_primary   or _dn_secondary

        # Bearish market structure — price under both moving averages, MAs
        # inverted.  Used to split "crash" from "melt-up" inside extreme vol.
        _bearish_structure = current_price < sma_20 < sma_50

        _consolidation_structural = (nw_compressed and
                                     abs(norm_returns) < 1.0 and
                                     not _extreme_vol)
        _vol_not_consolidation = _elevated_vol and nw_wide

        # ── REGIME CLASSIFICATION ─────────────────────────────────────────────
        # ORDERING FIX (2026 crash post-mortem): _extreme_vol used to be the
        # FIRST check, so a crash — which is always extreme-vol — could never
        # reach STRONG_DOWNTREND.  The one regime built to stand aside
        # (allow_long=False, allow_short=False) was structurally unreachable
        # exactly when it was needed, and price fell through to
        # HIGH_VOLATILITY, whose 0.60 long threshold is the MOST permissive
        # in the book.  Result: repeated longs into a 50% decline, stopped
        # out at -5% each time.  A crash and a melt-up are both "high
        # volatility"; they must not be treated identically.
        #   1. Confirmed downtrend  -> stand aside, regardless of vol level
        #   2. Extreme vol + bearish structure (crash-in-progress that hasn't
        #      yet met the full downtrend tests) -> stand aside too
        #   3. Extreme vol otherwise (melt-up / two-sided chaos) -> HIGH_VOL
        #   4. Then the original ladder unchanged
        if _strong_down:
            return 'STRONG_DOWNTREND'
        elif _extreme_vol and _bearish_structure:
            return 'STRONG_DOWNTREND'
        elif _extreme_vol:
            return 'HIGH_VOLATILITY'
        elif _strong_up:
            return 'STRONG_UPTREND'
        elif _vol_not_consolidation:
            return 'HIGH_VOLATILITY'
        elif _elevated_vol and not nw_compressed:
            return 'HIGH_VOLATILITY'
        elif _consolidation_structural:
            return 'CONSOLIDATION'
        elif returns > 0.10 and current_price > sma_20 > sma_50 and trend_score > 0:
            return 'STRONG_UPTREND'
        elif returns < -0.10 and current_price < sma_20 < sma_50 and trend_score > 0:
            return 'STRONG_DOWNTREND'
        else:
            return 'CONSOLIDATION'
    
    @staticmethod
    def get_threshold_for_regime(regime):
        try:
            regime_enum = MarketRegime(regime)
            config = REGIME_CONFIGS[regime_enum]
            if not config.allow_long:
                return 0.99
            return config.long_threshold
        except:
            pass
        
        thresholds = {
            'STRONG_UPTREND': Config.UPTREND_THRESHOLD,
            'STRONG_DOWNTREND': Config.DOWNTREND_THRESHOLD,
            'HIGH_VOLATILITY': Config.HIGH_VOL_THRESHOLD,
            'CONSOLIDATION': Config.BASE_THRESHOLD
        }
        return thresholds.get(regime, Config.BASE_THRESHOLD)
    

    @staticmethod
    def build_regime_map(df, test_df_indices, confirmation_days=5):
        """
        Pre-compute a CONFIRMED regime for every bar in the test window.

        Problem this solves:
            Detecting regime once at window-open and holding it fixed means a
            regime shift 6 weeks in (e.g. uptrend -> high-vol spike) goes
            undetected -- the model keeps applying uptrend thresholds to a
            completely different market structure.

        Stability mechanism:
            A *candidate* regime must be returned by detect_regime() for N
            consecutive bars before the *confirmed* regime officially switches.
            This prevents the detector flickering between labels day-to-day and
            whipsawing the rule-set.

            Confirmation windows (asymmetric by regime type):
            - HIGH_VOLATILITY : 2 bars  -- fast risk trigger; waiting 5 days
                                           during a vol spike costs real money.
            - STRONG_UPTREND  : 5 bars  -- Hurst/VR/autocorr must already all
              STRONG_DOWNTREND            agree (see detect_regime trend_score
                                           gate), THEN hold for a full trading week.
            - CONSOLIDATION   : 3 bars  -- quicker to confirm return to neutral
                                           than to confirm a new directional trend.

        Returns:
            dict {df_idx (int): confirmed_regime (str)} for every index in
            test_df_indices.  Starts as CONSOLIDATION before the first
            confirmation.
        """
        CONFIRM = {
            'HIGH_VOLATILITY':  2,
            'STRONG_UPTREND':   confirmation_days,
            'STRONG_DOWNTREND': confirmation_days,
            'CONSOLIDATION':    3,
        }

        regime_map       = {}
        confirmed_regime = 'CONSOLIDATION'   # safe fallback before first confirmation
        candidate        = 'CONSOLIDATION'
        candidate_streak = 0

        for df_idx in test_df_indices:
            # detect_regime on everything up to and including this bar
            raw = MarketRegimeDetector.detect_regime(df.iloc[:df_idx + 1])

            if raw == candidate:
                candidate_streak += 1
            else:
                candidate        = raw
                candidate_streak = 1

            required = CONFIRM.get(candidate, confirmation_days)
            if candidate_streak >= required and candidate != confirmed_regime:
                confirmed_regime = candidate  # official switch

            regime_map[df_idx] = confirmed_regime

        return regime_map

    @staticmethod
    
    def get_regime_description(regime):
        try:
            regime_enum = MarketRegime(regime)
            return REGIME_CONFIGS[regime_enum].description
        except:
            pass

        descriptions = {
            'STRONG_UPTREND': '📈 Strong Uptrend - Selective longs',
            'STRONG_DOWNTREND': '📉 Strong Downtrend - LONGS BLOCKED',
            'HIGH_VOLATILITY': '⚡ High Volatility - Small positions',
            'CONSOLIDATION': '➡️ Consolidation - Longs only'
        }
        return descriptions.get(regime, 'Unknown regime')


# ══════════════════════════════════════════════════════════════════════
# POSITION SIZING - CONSERVATIVE FOR SHARPE
# ══════════════════════════════════════════════════════════════════════

class PositionSizer:
    """Position sizing - V3.4 Balanced approach"""
    
    @staticmethod
    def calculate_position_size(signal_prob, current_vol, win_rate=0.65, 
                                avg_win=0.08, avg_loss=0.04,
                                account_size=Config.ACCOUNT_SIZE):
        # Balanced base
        base_contracts = account_size / 60000  # Middle ground
        
        win_loss_ratio = avg_win / avg_loss if avg_loss > 0 else 2
        kelly_fraction = (win_rate * win_loss_ratio - (1 - win_rate)) / win_loss_ratio
        kelly_fraction = max(0, min(kelly_fraction, 0.20))  # 20% cap
        
        confidence_mult = 0.5 + (signal_prob - 0.5) * 1.75
        confidence_mult = max(0.5, min(confidence_mult, 1.3))
        
        target_vol = 0.28
        vol_mult = min(1.0, target_vol / current_vol) if current_vol > 0 else 1.0
        
        contracts = base_contracts * kelly_fraction * confidence_mult * vol_mult * 3.5
        contracts = int(np.clip(contracts, 1, Config.MAX_POSITION_CONTRACTS))
        
        return contracts
    
    @staticmethod
    def adjust_for_regime(contracts, regime):
        try:
            regime_enum = MarketRegime(regime)
            config = REGIME_CONFIGS[regime_enum]
            mult = config.position_size_mult
        except:
            adjustments = {
                'STRONG_UPTREND': 1.0,
                'STRONG_DOWNTREND': 0.75,
                'HIGH_VOLATILITY': 0.5,
                'CONSOLIDATION': 0.6
            }
            mult = adjustments.get(regime, 0.8)
        
        return int(np.clip(contracts * mult, 1, Config.MAX_POSITION_CONTRACTS))


# ══════════════════════════════════════════════════════════════════════
# BIDIRECTIONAL TRADE SIMULATOR
# ══════════════════════════════════════════════════════════════════════

class BidirectionalTradeSimulator:
    """Trade simulator for both LONG and SHORT positions"""
    
    COMEX_CONTRACT_SIZE = 5000
    
    @classmethod
    def simulate_trade(cls, df, entry_idx, direction=TradeDirection.LONG,
                       position_size=1, stop_loss_pct=0.04,
                       take_profit_pct=0.08, max_hold_days=30):
        if entry_idx >= len(df) - 1:
            return None
        
        if direction == TradeDirection.FLAT:
            return None
        
        entry_price = df.iloc[entry_idx]['SI=F_Close']
        entry_date = df.iloc[entry_idx]['Date']
        
        is_long = direction == TradeDirection.LONG
        
        if is_long:
            stop_price = entry_price * (1 - stop_loss_pct)
            target_price = entry_price * (1 + take_profit_pct)
        else:
            stop_price = entry_price * (1 + stop_loss_pct)
            target_price = entry_price * (1 - take_profit_pct)
        
        for days_held in range(1, max_hold_days + 1):
            exit_idx = entry_idx + days_held
            if exit_idx >= len(df):
                break
            
            row = df.iloc[exit_idx]
            high = row['SI=F_High']
            low = row['SI=F_Low']
            
            if is_long:
                if low <= stop_price:
                    pnl = (stop_price - entry_price) * cls.COMEX_CONTRACT_SIZE * position_size
                    return cls._create_result(entry_date, row['Date'], 'STOP_LOSS',
                        days_held, entry_price, stop_price, -stop_loss_pct * 100,
                        pnl, position_size, direction)
                
                if high >= target_price:
                    pnl = (target_price - entry_price) * cls.COMEX_CONTRACT_SIZE * position_size
                    return cls._create_result(entry_date, row['Date'], 'TAKE_PROFIT',
                        days_held, entry_price, target_price, take_profit_pct * 100,
                        pnl, position_size, direction)
            else:
                if high >= stop_price:
                    pnl = (entry_price - stop_price) * cls.COMEX_CONTRACT_SIZE * position_size
                    return cls._create_result(entry_date, row['Date'], 'STOP_LOSS',
                        days_held, entry_price, stop_price, -stop_loss_pct * 100,
                        pnl, position_size, direction)
                
                if low <= target_price:
                    pnl = (entry_price - target_price) * cls.COMEX_CONTRACT_SIZE * position_size
                    return cls._create_result(entry_date, row['Date'], 'TAKE_PROFIT',
                        days_held, entry_price, target_price, take_profit_pct * 100,
                        pnl, position_size, direction)
        
        final_idx = min(entry_idx + max_hold_days, len(df) - 1)
        final_price = df.iloc[final_idx]['SI=F_Close']
        
        if is_long:
            return_pct = (final_price / entry_price - 1) * 100
            pnl = (final_price - entry_price) * cls.COMEX_CONTRACT_SIZE * position_size
        else:
            return_pct = (entry_price / final_price - 1) * 100
            pnl = (entry_price - final_price) * cls.COMEX_CONTRACT_SIZE * position_size
        
        return cls._create_result(entry_date, df.iloc[final_idx]['Date'], 'TIME_EXIT',
            max_hold_days, entry_price, final_price, return_pct, pnl, position_size, direction)
    
    @staticmethod
    def _create_result(entry_date, exit_date, exit_type, days_held,
                       entry_price, exit_price, return_pct, pnl,
                       position_size, direction):
        return {
            'entry_date': entry_date,
            'exit_date': exit_date,
            'exit_type': exit_type,
            'days_held': days_held,
            'entry_price': entry_price,
            'exit_price': exit_price,
            'return_pct': return_pct,
            'pnl': pnl,
            'position_size': position_size,
            'direction': direction.value if isinstance(direction, TradeDirection) else direction
        }


class TradeSimulator:
    """Legacy compatibility"""
    
    @staticmethod
    def simulate_trade(df, entry_idx, position_size=1,
                      stop_loss_pct=Config.STOP_LOSS_PCT,
                      take_profit_pct=Config.TAKE_PROFIT_PCT,
                      max_hold_days=Config.LOOKAHEAD_DAYS):
        return BidirectionalTradeSimulator.simulate_trade(
            df, entry_idx, TradeDirection.LONG, position_size,
            stop_loss_pct, take_profit_pct, max_hold_days
        )


# ══════════════════════════════════════════════════════════════════════
# ADVANCED FEATURE SELECTION
# ══════════════════════════════════════════════════════════════════════

class AdvancedFeatureSelector:
    """Feature selection with LASSO + correlation filtering"""
    
    def __init__(self, correlation_threshold=0.90, use_lasso=True,
                 use_pca=False, pca_variance=0.95, max_features=50, verbose=True):
        self.correlation_threshold = correlation_threshold
        self.use_lasso = use_lasso
        self.use_pca = use_pca
        self.pca_variance = pca_variance
        self.max_features = max_features
        self.verbose = verbose
        
        self.features_to_keep = None
        self.corr_features_removed = []
        self.lasso_selector = None
        self.lasso_features = None
        self.pca_model = None
        self.final_features = None
        self.scaler = None
        
    def fit(self, X, y, feature_names=None):
        if self.verbose:
            print("  Advanced feature selection...")
        
        self.scaler = StandardScaler()
        X_scaled = self.scaler.fit_transform(X)
        
        if feature_names is None:
            feature_names = [f'f{i}' for i in range(X.shape[1])]
        
        df = pd.DataFrame(X_scaled, columns=feature_names)
        
        # Remove highly correlated features
        corr_matrix = df.corr().abs()
        upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
        to_drop = [column for column in upper.columns if any(upper[column] > self.correlation_threshold)]
        self.corr_features_removed = to_drop
        features_after_corr = [f for f in feature_names if f not in to_drop]
        X_filtered = df[features_after_corr].values
        
        # LASSO selection
        if self.use_lasso and len(features_after_corr) > self.max_features:
            try:
                self.lasso_selector = LogisticRegressionCV(
                    penalty='l1', solver='saga', cv=5, max_iter=1000,
                    random_state=42, n_jobs=-1
                )
                self.lasso_selector.fit(X_filtered, y)
                importances = np.abs(self.lasso_selector.coef_).ravel()
                n_select = min(self.max_features, len(features_after_corr))
                top_indices = np.argsort(importances)[-n_select:]
                self.lasso_features = [features_after_corr[i] for i in top_indices if importances[i] > 0]
                
                if len(self.lasso_features) < self.max_features // 2:
                    self.lasso_features = [features_after_corr[i] for i in top_indices]
                
                features_after_lasso = self.lasso_features
                X_filtered = df[features_after_lasso].values
            except Exception as e:
                features_after_lasso = features_after_corr
                self.lasso_features = None
        else:
            features_after_lasso = features_after_corr
        
        if self.use_pca:
            self.pca_model = PCA(n_components=self.pca_variance, svd_solver='full')
            X_pca = self.pca_model.fit_transform(X_filtered)
            self.final_features = [f'PC{i+1}' for i in range(X_pca.shape[1])]
        else:
            self.final_features = features_after_lasso
        
        if not self.use_pca and len(self.final_features) > self.max_features:
            selector = SelectKBest(f_classif, k=self.max_features)
            X_filtered = df[self.final_features].values
            selector.fit(X_filtered, y)
            selected_mask = selector.get_support()
            self.final_features = [f for f, m in zip(self.final_features, selected_mask) if m]
        
        self.features_to_keep = self.final_features
        
        if self.verbose:
            print(f"    Selected {len(self.final_features)} features from {X.shape[1]}")
        
        return self
    
    def transform(self, X, feature_names=None):
        if feature_names is None:
            feature_names = [f'f{i}' for i in range(X.shape[1])]
        
        X_scaled = self.scaler.transform(X)
        df = pd.DataFrame(X_scaled, columns=feature_names)
        available_features = [f for f in self.features_to_keep if f in df.columns]
        X_filtered = df[available_features].values
        
        if self.use_pca and self.pca_model is not None:
            X_filtered = self.pca_model.transform(X_filtered)
        
        return X_filtered
    
    def fit_transform(self, X, y, feature_names=None):
        self.fit(X, y, feature_names)
        return self.transform(X, feature_names)


# ══════════════════════════════════════════════════════════════════════
# OPTIMIZED ENSEMBLE MODEL
# ══════════════════════════════════════════════════════════════════════

class RevIN(layers.Layer):
    """
    Reversible Instance Normalization (Kim et al. 2022).

    Motivation:
        Walk-forward windows span silver prices from ~$14 (2016) to ~$92 (2026).
        A StandardScaler fit on a 2-year training window learns population
        statistics for that regime; when the test window is in a different price
        regime the scaler's mean/std are stale.  RevIN solves this by normalising
        each input sequence by its *own* mean and variance across the time
        dimension — the LSTM never sees absolute price levels, only relative
        structure within each 60-bar window.

    Forward pass (applied to LSTM inputs):
        x_norm = (x - μ_instance) / (σ_instance + ε)  ×  γ  +  β
        where μ, σ are computed over the time axis (axis=1) per sample,
        and γ, β are learnable per-feature affine parameters.

    Reverse pass (for regression/forecasting — denormalises outputs back to
        original scale) is not used here since the model outputs a probability.

    Learnable affine parameters allow the layer to learn that some features
    should retain more of their original scale (e.g. volume is meaningful in
    relative terms but RSI is already bounded).
    """

    def __init__(self, eps: float = 1e-5, **kwargs):
        super().__init__(**kwargs)
        self.eps = eps

    def build(self, input_shape):
        # input_shape: (batch, timesteps, features)
        n_features = input_shape[-1]
        self.gamma = self.add_weight(
            name='revin_gamma', shape=(n_features,),
            initializer='ones', trainable=True
        )
        self.beta = self.add_weight(
            name='revin_beta', shape=(n_features,),
            initializer='zeros', trainable=True
        )
        super().build(input_shape)

    def call(self, x, training=None):
        # Per-instance statistics over the time axis — shape (batch, 1, features)
        mean = tf.reduce_mean(x, axis=1, keepdims=True)
        var  = tf.math.reduce_variance(x, axis=1, keepdims=True)
        x_norm = (x - mean) / tf.sqrt(var + self.eps)
        return x_norm * self.gamma + self.beta

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'eps': self.eps})
        return cfg


class OptimizedEnsemble:
    """Optimized ensemble with model agreement tracking"""
    
    def __init__(self):
        self.lstm_model = None
        self.rf_model = None
        self.xgb_model = None
        # Dedicated short models — trained on SHORT_Label, not inverted BUY prob
        self.lstm_short = None
        self.rf_short  = None
        self.xgb_short = None
        self.feature_selector = None
        # Dedicated short feature selector — fit in train() on SHORT_Label.
        # Initialized here so predict()/model-storage can't AttributeError if
        # the short training path never ran (e.g. too few short labels).
        self.short_feature_selector = None
        self.scaler_lstm = None
        self.feature_names = None
        # Per-window Brier-score weights (set in train(), used in predict())
        # Fallback to Config values if train() hasn't run yet
        self.buy_weights   = np.array([Config.LSTM_WEIGHT, Config.RF_WEIGHT, Config.XGB_WEIGHT])
        self.short_weights = np.array([Config.LSTM_WEIGHT, Config.RF_WEIGHT, Config.XGB_WEIGHT])
        # Regime-specialist RF/XGB models (Run 12+)
        self.rf_specialists          = {}   # regime → CalibratedClassifierCV
        self.xgb_specialists         = {}   # regime → CalibratedClassifierCV
        self.specialist_selectors    = {}   # regime → AdvancedFeatureSelector
        self.specialist_buy_weights  = {}   # regime → np.array([lstm_w, rf_w, xgb_w])
        # Base (uncalibrated) models for cross-window Platt raw prob extraction
        self.rf_model_base  = None
        self.xgb_model_base = None
        self.rf_short_base  = None
        self.xgb_short_base = None
        
    def build_lstm(self, input_shape):
        """
        LSTM with RevIN input normalization + linear skip connection.

        Two LSTM layers (64→32) with a single skip projection between them.
        Dense head goes directly 32→1 — no intermediate 16-unit layer.
        Smaller architecture reduces overfitting risk on 500-sample walk-forward
        windows and speeds up training without sacrificing representational
        capacity for this problem size.
        """
        w, f = input_shape
        inp = layers.Input((w, f))

        # ── RevIN: per-instance normalisation ────────────────────────────────
        x = RevIN(name='revin')(inp)

        # ── Block 1: LSTM(64) ─────────────────────────────────────────────────
        x = layers.LSTM(64, return_sequences=True,
                        dropout=0.2, recurrent_dropout=0.1)(x)

        # ── Block 2: LSTM(32) + linear skip from block 1 ─────────────────────
        skip      = layers.Dense(32, use_bias=False, name='skip_1')(x)
        skip_last = layers.Lambda(lambda t: t[:, -1, :], name='skip_last')(skip)
        x2        = layers.LSTM(32, return_sequences=False, dropout=0.2)(x)
        x         = layers.Add()([skip_last, x2])

        # ── Dense head: 32 → 1 ───────────────────────────────────────────────
        x   = layers.Dense(32, activation='relu',
                           kernel_regularizer=tf.keras.regularizers.l2(0.01))(x)
        x   = layers.Dropout(0.3)(x)
        out = layers.Dense(1, activation='sigmoid')(x)

        model = models.Model(inp, out)
        model.compile(
            loss=focal_loss(gamma=2.0, alpha=0.25),
            optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
            metrics=['accuracy']
        )
        return model
    
    def train(self, X_lstm_train, X_rf_train, y_train,
              X_lstm_val=None, X_rf_val=None, y_val=None, feature_names=None,
              y_short_train=None, oos_brier_scores=None, oos_brier_scores_short=None,
              prev2_oos_brier_scores=None, prev2_oos_brier_scores_short=None,
              prev_cal_rf_raw=None, prev_cal_xgb_raw=None, prev_cal_y=None,
              prev_cal_rf_raw_short=None, prev_cal_xgb_raw_short=None,
              prev_cal_y_short=None):
        
        print("  Training optimized LSTM...")
        ns, w, f = X_lstm_train.shape

        # ── Fix 7: LSTM column-mean fill for VXSLV-derived NaN features ──────
        # Tree models receive NaN directly (they handle it natively).  LSTM
        # cannot: a NaN anywhere in a sequence becomes NaN loss and kills
        # gradient updates.  Fill each feature column with the mean of its
        # non-NaN training values, computed fresh per window so early windows
        # (pre-2022, no VXSLV) get a sensible neutral value instead of zero.
        X_lstm_train = X_lstm_train.copy()
        for _fi in range(f):
            _col = X_lstm_train[:, :, _fi].ravel()
            _mask = ~np.isnan(_col)
            _fill = _col[_mask].mean() if _mask.any() else 0.0
            X_lstm_train[:, :, _fi] = np.where(
                np.isnan(X_lstm_train[:, :, _fi]), _fill, X_lstm_train[:, :, _fi])
        # ─────────────────────────────────────────────────────────────────────

        # ── TEMPORAL VALIDATION SPLIT (last 15%, order preserved) ────────────
        # Must be carved BEFORE scaling so the scaler is fit only on training
        # data — no information leakage from future samples into the scaler.
        val_size  = max(int(ns * 0.15), 1)
        split     = ns - val_size
        X_lstm_t, X_lstm_v = X_lstm_train[:split], X_lstm_train[split:]
        y_t,      y_v      = y_train[:split],       y_train[split:]
        # Keep a reference so SHORT model can reuse the same val window
        self._val_split_idx = split
        
        self.scaler_lstm = StandardScaler()
        # Fit scaler on train portion only — transform val separately
        X_lstm_t_scaled = self.scaler_lstm.fit_transform(
            X_lstm_t.reshape(-1, f)
        ).reshape(X_lstm_t.shape)
        X_lstm_v_scaled = self.scaler_lstm.transform(
            X_lstm_v.reshape(-1, f)
        ).reshape(X_lstm_v.shape)
        
        self.lstm_model = self.build_lstm((w, f))
        
        pos_ratio = np.sum(y_t) / len(y_t)
        neg_ratio = 1 - pos_ratio
        class_weight = {0: 1.0, 1: neg_ratio / pos_ratio if pos_ratio > 0 else 1.0}
        
        # monitor='val_loss' — early stopping now watches held-out data,
        # not training loss, so it halts when generalisation stops improving.
        callbacks_list = [
            callbacks.EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True),
            callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)
        ]
        
        buy_history = self.lstm_model.fit(
            X_lstm_t_scaled, y_t,
            epochs=Config.LSTM_EPOCHS,
            batch_size=Config.LSTM_BATCH_SIZE,
            validation_data=(X_lstm_v_scaled, y_v),
            callbacks=callbacks_list,
            class_weight=class_weight,
            verbose=0
        )
        
        # ── TRAIN / VAL GAP LOGGING ───────────────────────────────────────────
        # Makes overfitting visible across walk-forward windows instead of silent.
        _buy_train_loss = buy_history.history['loss'][-1]
        _buy_val_loss   = buy_history.history['val_loss'][-1]
        _buy_gap        = _buy_val_loss - _buy_train_loss
        _buy_epochs     = len(buy_history.history['loss'])
        print(f"  BUY  LSTM → train_loss={_buy_train_loss:.4f}  "
              f"val_loss={_buy_val_loss:.4f}  gap={_buy_gap:+.4f}  "
              f"epochs={_buy_epochs}/{Config.LSTM_EPOCHS}")
        
        if feature_names is None:
            feature_names = [f'feature_{i}' for i in range(X_rf_train.shape[1])]

        # RF and XGB train on the FULL training window -- they have no early
        # stopping mechanism and gain nothing from withholding the last 15%.
        # The val slice (X_rf_v / y_v) is carved here solely to serve as the
        # calibration set for CalibratedClassifierCV(cv='prefit'), aligning
        # all three model outputs onto the same probability scale before
        # ensemble averaging.  'sigmoid' (Platt scaling) is used over
        # 'isotonic' because isotonic needs ~1000+ calibration samples.
        X_rf_v = X_rf_train[split:]        # val slice for calibration only

        self.feature_names = feature_names

        # Feature selector fits fresh each window on the full training data
        self.feature_selector = AdvancedFeatureSelector(
            correlation_threshold=Config.CORRELATION_THRESHOLD,
            use_lasso=Config.USE_LASSO_SELECTION,
            use_pca=Config.USE_PCA,
            pca_variance=Config.PCA_VARIANCE,
            max_features=Config.RF_TOP_FEATURES,
            verbose=False
        )
        X_rf_full_selected = self.feature_selector.fit_transform(
            X_rf_train, y_train, feature_names)
        X_rf_v_selected = self.feature_selector.transform(X_rf_v, feature_names)

        print("  Training Random Forest...")
        _rf_base = RandomForestClassifier(
            n_estimators=Config.RF_ESTIMATORS,
            max_depth=Config.RF_MAX_DEPTH,
            min_samples_split=5,
            min_samples_leaf=2,
            random_state=42,
            class_weight='balanced',
            n_jobs=-1
        )
        _rf_base.fit(X_rf_full_selected, y_train)

        print("  Training XGBoost...")
        scale_pos_weight = len(y_train[y_train==0]) / max(len(y_train[y_train==1]), 1)
        _xgb_base = xgb.XGBClassifier(
            objective='binary:logistic',
            scale_pos_weight=scale_pos_weight,
            n_estimators=Config.XGB_ESTIMATORS,
            max_depth=5,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=Config.XGB_REG_ALPHA,
            reg_lambda=Config.XGB_REG_LAMBDA,
            random_state=42,
            eval_metric='logloss',
            n_jobs=-1
        )
        _xgb_base.fit(X_rf_full_selected, y_train)

        # ── BRIER SCORES (pre-calibration) ───────────────────────────────────
        # Evaluated on val_slice using the BASE (uncalibrated) models.
        # Base models trained on X_rf_full_selected (full window) → val_slice
        # is genuinely held-out, giving an honest probability estimate.
        # Calibrating THEN scoring on the same slice was the leakage: the
        # sigmoid was fit to minimise error on those exact samples, so its
        # Brier score reflected in-sample fit rather than generalisation.
        def _brier(p, y):
            return float(np.mean((p - y) ** 2))

        lstm_val_probs    = self.lstm_model.predict(X_lstm_v_scaled, verbose=0).ravel()
        rf_base_val_probs = _rf_base.predict_proba(X_rf_v_selected)[:, 1]
        xgb_base_val_probs= _xgb_base.predict_proba(X_rf_v_selected)[:, 1]

        bs_lstm = max(_brier(lstm_val_probs,     y_v), 1e-6)
        bs_rf   = max(_brier(rf_base_val_probs,  y_v), 1e-6)
        bs_xgb  = max(_brier(xgb_base_val_probs, y_v), 1e-6)

        # Val-slice Brier for diagnostics + W1 weight fallback
        bs_lstm_val = max(_brier(lstm_val_probs,     y_v), 1e-6)
        bs_rf_val   = max(_brier(rf_base_val_probs,  y_v), 1e-6)
        bs_xgb_val  = max(_brier(xgb_base_val_probs, y_v), 1e-6)

        # ── OOS Brier weighting (R18) ─────────────────────────────────────────
        if oos_brier_scores is not None:
            # EMA blend: 0.6 × most-recent OOS + 0.4 × two-windows-back OOS.
            if prev2_oos_brier_scores is not None:
                def _ema_brier(k):
                    return 0.6 * oos_brier_scores[k] + 0.4 * prev2_oos_brier_scores[k]
                bs_lstm = max(_ema_brier('lstm'), 1e-6)
                bs_rf   = max(_ema_brier('rf'),   1e-6)
                bs_xgb  = max(_ema_brier('xgb'),  1e-6)
                _weight_source = "OOS-EMA"
            else:
                bs_lstm = max(oos_brier_scores['lstm'], 1e-6)
                bs_rf   = max(oos_brier_scores['rf'],   1e-6)
                bs_xgb  = max(oos_brier_scores['xgb'],  1e-6)
                _weight_source = "OOS"
            raw_w = np.array([1/bs_lstm, 1/bs_rf, 1/bs_xgb])
            self.buy_weights = raw_w / raw_w.sum()
        else:
            # W1 only: val-slice with caps (XGB ≤ 0.60, LSTM ≥ 0.12)
            raw_w = np.array([1/bs_lstm_val, 1/bs_rf_val, 1/bs_xgb_val])
            _w = raw_w / raw_w.sum()
            _XGB_CAP_W1, _LSTM_FLOOR_W1 = 0.60, 0.12
            if _w[2] > _XGB_CAP_W1:
                _excess = _w[2] - _XGB_CAP_W1; _w[2] = _XGB_CAP_W1
                _other = _w[:2].sum()
                if _other > 0: _w[:2] += _excess * (_w[:2] / _other)
            if _w[0] < _LSTM_FLOOR_W1:
                _deficit = _LSTM_FLOOR_W1 - _w[0]; _w[0] = _LSTM_FLOOR_W1
                _w[1] -= _deficit; _w[1] = max(_w[1], 0.0)
            _w = np.maximum(_w, 0.0)
            self.buy_weights = _w / _w.sum()
            _weight_source = "val-slice+caps (W1)"

        print(f"  BUY  weights ({_weight_source}) → "
              f"LSTM={self.buy_weights[0]:.3f}  RF={self.buy_weights[1]:.3f}  "
              f"XGB={self.buy_weights[2]:.3f}  "
              f"(val-diag: {bs_lstm_val:.4f} / {bs_rf_val:.4f} / {bs_xgb_val:.4f})")

        # ── Cross-window Platt calibration (R18) ─────────────────────────────
        # W2+: calibrate on previous window's raw pre-Platt test outputs.
        # W1 : val-slice CalibratedClassifierCV fallback.
        # Guard: need ≥2 positive AND ≥2 negative BUY bars.
        _prev_cal_y_arr = np.asarray(prev_cal_y) if prev_cal_y is not None else None
        _buy_cal_valid = (
            prev_cal_rf_raw is not None and prev_cal_xgb_raw is not None
            and _prev_cal_y_arr is not None
            and len(_prev_cal_y_arr) >= 10
            and _prev_cal_y_arr.sum() >= 2
            and (_prev_cal_y_arr == 0).sum() >= 2
        )
        if _buy_cal_valid:
            from sklearn.linear_model import LogisticRegression
            def _fit_platt(raw_probs, labels):
                lr = LogisticRegression(C=1e10, solver='lbfgs', max_iter=1000)
                lr.fit(raw_probs.reshape(-1, 1), labels.astype(int))
                return lr
            _rf_platt  = _fit_platt(np.asarray(prev_cal_rf_raw),  _prev_cal_y_arr)
            _xgb_platt = _fit_platt(np.asarray(prev_cal_xgb_raw), _prev_cal_y_arr)
            class _PlattWrapper:
                def __init__(self, base, platt):
                    self._base = base; self._platt = platt
                    self.classes_ = np.array([0, 1])
                def predict_proba(self, X):
                    raw = self._base.predict_proba(X)[:, 1].reshape(-1, 1)
                    cal = self._platt.predict_proba(raw)[:, 1]
                    return np.column_stack([1 - cal, cal])
            self.rf_model  = _PlattWrapper(_rf_base,  _rf_platt)
            self.xgb_model = _PlattWrapper(_xgb_base, _xgb_platt)
            self.rf_model_base  = _rf_base
            self.xgb_model_base = _xgb_base
            _cal_source = "cross-window (prev test set)"
        else:
            self.rf_model = CalibratedClassifierCV(_rf_base, cv='prefit', method='sigmoid')
            self.rf_model.fit(X_rf_v_selected, y_v)
            self.xgb_model = CalibratedClassifierCV(_xgb_base, cv='prefit', method='sigmoid')
            self.xgb_model.fit(X_rf_v_selected, y_v)
            self.rf_model_base  = _rf_base
            self.xgb_model_base = _xgb_base
            _cal_source = "val slice (W1 fallback)"
        print(f"  ✓ BUY RF and XGB calibrated (Platt — {_cal_source})")

        # ── REGIME-SPECIALIST BUY MODELS (Run 12) ────────────────────────────
        # Train one RF+XGB specialist per regime using only training bars from
        # that regime.  Each specialist gets its own LASSO feature selector and
        # Brier-calibrated weights.  LSTM stays shared (too few samples per
        # regime for stable gradient descent).
        # At inference, each test bar is routed to its specialist via per_bar_regimes.
        _MIN_SPECIALIST = 60

        _fn = list(feature_names) if feature_names is not None else []
        def _get_col(name):
            return X_rf_train[:, _fn.index(name)] if name in _fn else None

        _vp_train   = _get_col('VOL_PERCENTILE')
        _rv20_train = _get_col('REALIZED_VOL_20')
        _sma20_col  = _get_col('SI=F_SMA20')
        _sma50_col  = _get_col('SI=F_SMA50')
        _close_col  = _get_col('SI=F_Close')

        if (_vp_train is not None and _close_col is not None and
                _sma20_col is not None and _sma50_col is not None):
            _roc60 = _get_col('ROC_60') if _get_col('ROC_60') is not None else np.zeros(len(X_rf_train))
            _ann_vol = (_rv20_train / 100.0) if _rv20_train is not None else np.full(len(X_rf_train), 0.20)
            _hv_tr  = (_vp_train > 0.85) | (_ann_vol > 0.40)
            _su_tr  = (~_hv_tr) & (_roc60 > 0.08) & (_close_col > _sma20_col) & (_sma20_col > _sma50_col)
            _sd_tr  = (~_hv_tr) & (_roc60 < -0.08) & (_close_col < _sma20_col) & (_sma20_col < _sma50_col)
            _con_tr = ~_hv_tr & ~_su_tr & ~_sd_tr
            _regime_proxy_masks = {
                'HIGH_VOLATILITY':  _hv_tr,
                'STRONG_UPTREND':   _su_tr,
                'STRONG_DOWNTREND': _sd_tr,
                'CONSOLIDATION':    _con_tr,
            }
        else:
            _regime_proxy_masks = {}

        _specialist_log = []
        for _reg, _mask in _regime_proxy_masks.items():
            _n = int(_mask.sum())
            if _n < _MIN_SPECIALIST:
                _specialist_log.append(f"{_reg[:3]}={_n}(fallback)")
                continue

            _X_spec_full = X_rf_train[_mask]
            _y_spec_full = y_train[_mask]
            _spec_val_mask_in_val = _mask[split:] if split < len(_mask) else np.array([], dtype=bool)
            _X_spec_val  = X_rf_v[_spec_val_mask_in_val]
            _y_spec_val  = y_v[_spec_val_mask_in_val]

            _sel = AdvancedFeatureSelector(
                correlation_threshold=0.90, use_lasso=True,
                max_features=50, verbose=False)
            _sel.fit(_X_spec_full, _y_spec_full, feature_names)
            _X_spec_sel = _sel.transform(_X_spec_full, feature_names)

            _spw_spec = len(_y_spec_full[_y_spec_full==0]) / max(len(_y_spec_full[_y_spec_full==1]), 1)
            _rf_spec_base = RandomForestClassifier(
                n_estimators=Config.RF_ESTIMATORS, max_depth=Config.RF_MAX_DEPTH,
                min_samples_split=5, min_samples_leaf=2,
                random_state=42, class_weight='balanced', n_jobs=-1)
            _rf_spec_base.fit(_X_spec_sel, _y_spec_full)

            _xgb_spec_base = xgb.XGBClassifier(
                objective='binary:logistic', scale_pos_weight=_spw_spec,
                n_estimators=Config.XGB_ESTIMATORS, max_depth=5,
                learning_rate=0.1, subsample=0.8, colsample_bytree=0.8,
                reg_alpha=Config.XGB_REG_ALPHA, reg_lambda=Config.XGB_REG_LAMBDA,
                random_state=42, eval_metric='logloss', n_jobs=-1)
            _xgb_spec_base.fit(_X_spec_sel, _y_spec_full)

            # Brier-calibrated specialist weights
            if _spec_val_mask_in_val.sum() >= 5:
                _X_sv_sel = _sel.transform(_X_spec_val, feature_names)
                _lstm_sv = self.lstm_model.predict(
                    X_lstm_v_scaled[_spec_val_mask_in_val], verbose=0).ravel()
                _rf_sv   = _rf_spec_base.predict_proba(_X_sv_sel)[:, 1]
                _xgb_sv  = _xgb_spec_base.predict_proba(_X_sv_sel)[:, 1]
                _yv_spec = _y_spec_val
                bs_l = max(_brier(_lstm_sv, _yv_spec), 1e-6)
                bs_r = max(_brier(_rf_sv,   _yv_spec), 1e-6)
                bs_x = max(_brier(_xgb_sv,  _yv_spec), 1e-6)
                _raw_sw = np.array([1/bs_l, 1/bs_r, 1/bs_x])
                _sw = _raw_sw / _raw_sw.sum()
                _XGB_CAP_SPEC, _LSTM_FLOOR_SPEC = 0.60, 0.12
                if _sw[2] > _XGB_CAP_SPEC:
                    _excess = _sw[2] - _XGB_CAP_SPEC; _sw[2] = _XGB_CAP_SPEC
                    _other = _sw[:2].sum()
                    if _other > 0: _sw[:2] += _excess * (_sw[:2] / _other)
                if _sw[0] < _LSTM_FLOOR_SPEC:
                    _deficit = _LSTM_FLOOR_SPEC - _sw[0]; _sw[0] = _LSTM_FLOOR_SPEC
                    _sw[1] = max(_sw[1] - _deficit, 0.0)
                _spec_weights = np.maximum(_sw, 0.0); _spec_weights /= _spec_weights.sum()
            else:
                _spec_weights = self.buy_weights.copy()

            # Platt calibration for specialist
            if _spec_val_mask_in_val.sum() >= 5:
                _X_sv_sel = _sel.transform(_X_spec_val, feature_names)
                _rf_spec  = CalibratedClassifierCV(_rf_spec_base,  cv='prefit', method='sigmoid')
                _rf_spec.fit(_X_sv_sel, _y_spec_val)
                _xgb_spec = CalibratedClassifierCV(_xgb_spec_base, cv='prefit', method='sigmoid')
                _xgb_spec.fit(_X_sv_sel, _y_spec_val)
            else:
                _rf_spec  = _rf_spec_base
                _xgb_spec = _xgb_spec_base

            self.rf_specialists[_reg]         = _rf_spec
            self.xgb_specialists[_reg]        = _xgb_spec
            self.specialist_selectors[_reg]   = _sel
            self.specialist_buy_weights[_reg] = _spec_weights
            _specialist_log.append(f"{_reg[:3]}={_n}")

        if _specialist_log:
            print(f"  ✓ Regime specialists: {' | '.join(_specialist_log)}")
        else:
            print("  ⚠ No regime proxy signals — generalist only")
        # ─────────────────────────────────────────────────────────────────────
        
        # ── DEDICATED SHORT MODELS ────────────────────────────────────────────
        # Trained on SHORT_Label (price falls ≥ threshold) — NOT on inverted BUY prob.
        # A low BUY probability ≠ active short signal; only SHORT_Label captures that.
        if y_short_train is not None and y_short_train.sum() >= 10:
            print("  Training dedicated SHORT models (SHORT_Label)...")
            
            # Reuse the same temporal split index so the SHORT LSTM val window
            # is identical to the BUY LSTM val window — apples-to-apples comparison.
            y_short_t = y_short_train[:split]
            y_short_v = y_short_train[split:]
            
            # Short LSTM — val split already scaled above; reuse X_lstm_t/v_scaled
            short_pos_ratio = np.sum(y_short_t) / max(len(y_short_t), 1)
            short_class_weight = {0: 1.0,
                                  1: (1 - short_pos_ratio) / short_pos_ratio if short_pos_ratio > 0 else 1.0}
            self.lstm_short = self.build_lstm((w, f))
            short_history = self.lstm_short.fit(
                X_lstm_t_scaled, y_short_t,
                epochs=Config.LSTM_EPOCHS,
                batch_size=Config.LSTM_BATCH_SIZE,
                validation_data=(X_lstm_v_scaled, y_short_v),
                callbacks=[
                    callbacks.EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True),
                    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)
                ],
                class_weight=short_class_weight,
                verbose=0
            )
            
            # Train / val gap logging for SHORT LSTM
            _sh_train_loss = short_history.history['loss'][-1]
            _sh_val_loss   = short_history.history['val_loss'][-1]
            _sh_gap        = _sh_val_loss - _sh_train_loss
            _sh_epochs     = len(short_history.history['loss'])
            print(f"  SHORT LSTM → train_loss={_sh_train_loss:.4f}  "
                  f"val_loss={_sh_val_loss:.4f}  gap={_sh_gap:+.4f}  "
                  f"epochs={_sh_epochs}/{Config.LSTM_EPOCHS}")
            
            # Short RF trains on FULL y_short_train; val slice used only for calibration
            _rf_short_base = RandomForestClassifier(
                n_estimators=Config.RF_ESTIMATORS,
                max_depth=Config.RF_MAX_DEPTH,
                min_samples_split=5,
                min_samples_leaf=2,
                random_state=42,
                class_weight='balanced',
                n_jobs=-1
            )

            # SHORT feature selector — fitted on SHORT_Label correlation
            self.short_feature_selector = AdvancedFeatureSelector(
                correlation_threshold=0.90, use_lasso=True,
                max_features=50, verbose=False)
            self.short_feature_selector.fit(X_rf_train, y_short_train, feature_names)
            X_rf_full_short_sel = self.short_feature_selector.transform(X_rf_train, feature_names)
            X_rf_v_short_sel    = self.short_feature_selector.transform(X_rf_v,      feature_names)
            print(f"  SHORT feature selector: {X_rf_full_short_sel.shape[1]} features "
                  f"(vs {X_rf_full_selected.shape[1]} BUY features)")

            _rf_short_base.fit(X_rf_full_short_sel, y_short_train)

            # Short XGBoost -- same: full window training, val slice calibration
            short_spw = len(y_short_train[y_short_train==0]) / max(len(y_short_train[y_short_train==1]), 1)
            _xgb_short_base = xgb.XGBClassifier(
                objective='binary:logistic',
                scale_pos_weight=short_spw,
                n_estimators=Config.XGB_ESTIMATORS,
                max_depth=5,
                learning_rate=0.1,
                subsample=0.8,
                colsample_bytree=0.8,
                reg_alpha=Config.XGB_REG_ALPHA,
                reg_lambda=Config.XGB_REG_LAMBDA,
                random_state=42,
                eval_metric='logloss',
                n_jobs=-1
            )
            _xgb_short_base.fit(X_rf_full_short_sel, y_short_train)

            # Brier on base models using SHORT feature selector (pre-calibration)
            lstm_s_val     = self.lstm_short.predict(X_lstm_v_scaled, verbose=0).ravel()
            rf_s_base_val  = _rf_short_base.predict_proba(X_rf_v_short_sel)[:, 1]
            xgb_s_base_val = _xgb_short_base.predict_proba(X_rf_v_short_sel)[:, 1]

            bs_s_lstm_val = max(_brier(lstm_s_val,     y_short_v), 1e-6)
            bs_s_rf_val   = max(_brier(rf_s_base_val,  y_short_v), 1e-6)
            bs_s_xgb_val  = max(_brier(xgb_s_base_val, y_short_v), 1e-6)

            # ── SHORT OOS Brier weighting (R18) ──────────────────────────────
            if oos_brier_scores_short is not None:
                if prev2_oos_brier_scores_short is not None:
                    def _ema_s(k):
                        return 0.6 * oos_brier_scores_short[k] + 0.4 * prev2_oos_brier_scores_short[k]
                    bs_s_lstm = max(_ema_s('lstm'), 1e-6)
                    bs_s_rf   = max(_ema_s('rf'),   1e-6)
                    bs_s_xgb  = max(_ema_s('xgb'),  1e-6)
                    _sw_source = "OOS-EMA"
                else:
                    bs_s_lstm = max(oos_brier_scores_short['lstm'], 1e-6)
                    bs_s_rf   = max(oos_brier_scores_short['rf'],   1e-6)
                    bs_s_xgb  = max(oos_brier_scores_short['xgb'],  1e-6)
                    _sw_source = "OOS"
                raw_sw = np.array([1/bs_s_lstm, 1/bs_s_rf, 1/bs_s_xgb])
                self.short_weights = raw_sw / raw_sw.sum()
            else:
                # W1 fallback: val-slice with caps
                raw_sw = np.array([1/bs_s_lstm_val, 1/bs_s_rf_val, 1/bs_s_xgb_val])
                _sw = raw_sw / raw_sw.sum()
                if _sw[2] > 0.60:
                    _excess = _sw[2] - 0.60; _sw[2] = 0.60
                    _other = _sw[:2].sum()
                    if _other > 0: _sw[:2] += _excess * (_sw[:2] / _other)
                if _sw[0] < 0.12:
                    _deficit = 0.12 - _sw[0]; _sw[0] = 0.12
                    _sw[1] = max(_sw[1] - _deficit, 0.0)
                _sw = np.maximum(_sw, 0.0)
                self.short_weights = _sw / _sw.sum()
                _sw_source = "val-slice+caps (W1)"
            print(f"  SHORT weights ({_sw_source}) → "
                  f"LSTM={self.short_weights[0]:.3f}  RF={self.short_weights[1]:.3f}  "
                  f"XGB={self.short_weights[2]:.3f}  "
                  f"(val-diag: {bs_s_lstm_val:.4f} / {bs_s_rf_val:.4f} / {bs_s_xgb_val:.4f})")

            # ── SHORT cross-window Platt (R18) ────────────────────────────────
            # Min 5 positive SHORT bars + class_weight='balanced'
            _prev_cal_y_short_arr = np.asarray(prev_cal_y_short) if prev_cal_y_short is not None else None
            _short_cal_valid = (
                prev_cal_rf_raw_short is not None
                and prev_cal_xgb_raw_short is not None
                and _prev_cal_y_short_arr is not None
                and len(_prev_cal_y_short_arr) >= 10
                and _prev_cal_y_short_arr.sum() >= 5
                and (_prev_cal_y_short_arr == 0).sum() >= 2
            )
            if _short_cal_valid:
                from sklearn.linear_model import LogisticRegression
                def _fit_platt_s(raw_probs, labels):
                    lr = LogisticRegression(C=1e10, solver='lbfgs', max_iter=1000,
                                           class_weight='balanced')
                    lr.fit(raw_probs.reshape(-1, 1), labels.astype(int))
                    return lr
                _rf_platt_s  = _fit_platt_s(np.asarray(prev_cal_rf_raw_short),  _prev_cal_y_short_arr)
                _xgb_platt_s = _fit_platt_s(np.asarray(prev_cal_xgb_raw_short), _prev_cal_y_short_arr)
                class _PlattWrapperShort:
                    def __init__(self, base, platt):
                        self._base = base; self._platt = platt
                        self.classes_ = np.array([0, 1])
                    def predict_proba(self, X):
                        raw = self._base.predict_proba(X)[:, 1].reshape(-1, 1)
                        cal = self._platt.predict_proba(raw)[:, 1]
                        return np.column_stack([1 - cal, cal])
                self.rf_short  = _PlattWrapperShort(_rf_short_base,  _rf_platt_s)
                self.xgb_short = _PlattWrapperShort(_xgb_short_base, _xgb_platt_s)
                self.rf_short_base  = _rf_short_base
                self.xgb_short_base = _xgb_short_base
                _short_cal_source = "cross-window (balanced)"
            else:
                self.rf_short = CalibratedClassifierCV(_rf_short_base, cv='prefit', method='sigmoid')
                self.rf_short.fit(X_rf_v_short_sel, y_short_v)
                self.xgb_short = CalibratedClassifierCV(_xgb_short_base, cv='prefit', method='sigmoid')
                self.xgb_short.fit(X_rf_v_short_sel, y_short_v)
                self.rf_short_base  = _rf_short_base
                self.xgb_short_base = _xgb_short_base
                _short_cal_source = "val slice (fallback)"
            print(f"  ✓ Short models trained + calibrated ({int(y_short_train.sum())} SHORT samples, Platt: {_short_cal_source})")
        else:
            print("  ⚠ Insufficient SHORT samples — short models not trained this window")
        
    def predict(self, X_lstm, X_rf, regime='CONSOLIDATION', per_bar_regimes=None):
        """Generate ensemble predictions with optional specialist routing."""
        ns, w, f = X_lstm.shape

        # Fix 7 (predict-side): fill NaN before LSTM scaling
        X_lstm = X_lstm.copy()
        for _fi in range(f):
            _nan_mask = np.isnan(X_lstm[:, :, _fi])
            if _nan_mask.any():
                _fill = self.scaler_lstm.mean_[_fi] if hasattr(self.scaler_lstm, 'mean_') else 0.0
                X_lstm[:, :, _fi] = np.where(_nan_mask, _fill, X_lstm[:, :, _fi])

        X_lstm_scaled = self.scaler_lstm.transform(X_lstm.reshape(-1, f)).reshape(X_lstm.shape)
        X_rf_selected = self.feature_selector.transform(X_rf, self.feature_names)

        lstm_probs = self.lstm_model.predict(X_lstm_scaled, verbose=0).ravel()
        rf_probs_generalist  = self.rf_model.predict_proba(X_rf_selected)[:, 1]
        xgb_probs_generalist = self.xgb_model.predict_proba(X_rf_selected)[:, 1]

        # ── REGIME-SPECIALIST ROUTING ─────────────────────────────────────────
        rf_probs  = rf_probs_generalist.copy()
        xgb_probs = xgb_probs_generalist.copy()
        _bar_weights = np.tile(self.buy_weights, (ns, 1)).astype(float)

        if per_bar_regimes is not None and self.rf_specialists:
            for _i, _bar_reg in enumerate(per_bar_regimes):
                if (_bar_reg in self.rf_specialists and
                        _bar_reg in self.specialist_selectors):
                    _spec_sel  = self.specialist_selectors[_bar_reg]
                    _X_bar_sel = _spec_sel.transform(X_rf[_i:_i+1], self.feature_names)
                    rf_probs[_i]  = self.rf_specialists[_bar_reg].predict_proba(_X_bar_sel)[0, 1]
                    xgb_probs[_i] = self.xgb_specialists[_bar_reg].predict_proba(_X_bar_sel)[0, 1]
                    _bar_weights[_i] = self.specialist_buy_weights.get(_bar_reg, self.buy_weights)
        # ─────────────────────────────────────────────────────────────────────

        ensemble_probs = (
            _bar_weights[:, 0] * lstm_probs +
            _bar_weights[:, 1] * rf_probs   +
            _bar_weights[:, 2] * xgb_probs
        )

        threshold = MarketRegimeDetector.get_threshold_for_regime(regime)
        ensemble_preds = (ensemble_probs > threshold).astype(int)
        
        # Track individual model predictions for agreement
        lstm_preds = (lstm_probs > 0.5).astype(int)
        rf_preds = (rf_probs > 0.5).astype(int)
        xgb_preds = (xgb_probs > 0.5).astype(int)
        model_agreement = lstm_preds + rf_preds + xgb_preds  # 0-3
        
        # ── DEDICATED SHORT PROBABILITIES ────────────────────────────────────
        lstm_short_probs = rf_short_probs = xgb_short_probs = None
        if self.lstm_short is not None and self.rf_short is not None and self.xgb_short is not None:
            lstm_short_probs = self.lstm_short.predict(X_lstm_scaled, verbose=0).ravel()
            # Use the dedicated short feature selector if available, else fall back
            if self.short_feature_selector is not None:
                X_rf_short_sel = self.short_feature_selector.transform(X_rf, self.feature_names)
            else:
                X_rf_short_sel = X_rf_selected
            rf_short_probs   = self.rf_short.predict_proba(X_rf_short_sel)[:, 1]
            xgb_short_probs  = self.xgb_short.predict_proba(X_rf_short_sel)[:, 1]
            short_probs = (
                self.short_weights[0] * lstm_short_probs +
                self.short_weights[1] * rf_short_probs  +
                self.short_weights[2] * xgb_short_probs
            )
            # Agreement: how many dedicated short models call SHORT
            lstm_short_preds = (lstm_short_probs > 0.5).astype(int)
            rf_short_preds   = (rf_short_probs   > 0.5).astype(int)
            xgb_short_preds  = (xgb_short_probs  > 0.5).astype(int)
            short_model_agreement = lstm_short_preds + rf_short_preds + xgb_short_preds
        else:
            # Fallback: keep old inverted-buy approach so existing windows still work
            short_probs = 1 - ensemble_probs
            short_model_agreement = 3 - model_agreement
        
        return {
            'ensemble_probs':       ensemble_probs,
            'ensemble_preds':       ensemble_preds,
            'lstm_probs':           lstm_probs,
            'rf_probs':             rf_probs,
            'xgb_probs':            xgb_probs,
            'lstm_preds':           lstm_preds,
            'rf_preds':             rf_preds,
            'xgb_preds':            xgb_preds,
            'model_agreement':      model_agreement,
            # Short model outputs — used directly for short signal generation
            'short_probs':          short_probs,
            'short_model_agreement': short_model_agreement,
            'short_lstm_probs':      lstm_short_probs if self.lstm_short is not None else None,
            'short_rf_probs':        rf_short_probs   if self.rf_short  is not None else None,
            'short_xgb_probs':       xgb_short_probs  if self.xgb_short is not None else None,
            'threshold_used':       threshold
        }


# ══════════════════════════════════════════════════════════════════════
# DATA LOADING AND FEATURE ENGINEERING
# ══════════════════════════════════════════════════════════════════════

def load_market_data():
    """Load and prepare market data"""
    print("\n" + "="*70)
    print("LOADING MARKET DATA")
    print("="*70)
    
    symbols = ["SI=F", "CL=F", "GC=F", "DX-Y.NYB", "HG=F", 
               "^DJI", "^GSPC", "^IXIC", "^RUT", "^TNX", "^VIX"]
    
    # Use current date if END_DATE not specified
    end_date = Config.END_DATE if Config.END_DATE else datetime.now().strftime('%Y-%m-%d')
    
    print(f"Fetching data from {Config.START_DATE} to {end_date}...")
    raw = yf.download(symbols, start=Config.START_DATE, end=end_date,
                      interval="1d", group_by="ticker", progress=False).dropna()
    
    parts = []
    for s in symbols:
        df_s = raw[s][["Open", "High", "Low", "Close", "Volume"]].copy()
        df_s.columns = [f"{s}_{c}" for c in df_s.columns]
        parts.append(df_s)
    
    df = pd.concat(parts, axis=1).reset_index()
    df.rename(columns={"index": "Date"}, inplace=True)

    # ── CBOE CDN loader (PRIMARY source for CBOE vol indices) ────────────────
    # CBOE publishes full index history as a keyless CSV — the authoritative
    # source, and unlike Yahoo (1d/5d only for ^VXSLV) and FRED (VXSLVCLS ends
    # Feb 2022) it carries the COMPLETE series including the post-2022 revival.
    # URL: cdn.cboe.com/.../{SYM}_History.csv | Format: DATE,OPEN,HIGH,LOW,CLOSE
    def _load_cboe_index(symbol):
        import urllib.request as _urllib
        url = (f"https://cdn.cboe.com/api/global/us_indices/daily_prices"
               f"/{symbol}_History.csv")
        try:
            req = _urllib.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            with _urllib.urlopen(req, timeout=15) as resp:
                raw = resp.read().decode('utf-8')
            lines = [l for l in raw.splitlines() if l.strip()]
            dates, closes = [], []
            for line in lines[1:]:                         # skip header row
                parts = line.split(',')
                if len(parts) < 5:
                    continue
                try:
                    dates.append(pd.to_datetime(parts[0].strip(), format='%m/%d/%Y'))
                    closes.append(float(parts[4].strip()))
                except (ValueError, IndexError):
                    continue
            if len(dates) < 20:
                return None
            s = pd.Series(closes, index=pd.DatetimeIndex(dates)).sort_index()
            return s[~s.index.duplicated(keep='last')]
        except Exception as exc:
            print(f"    CBOE CDN fetch for {symbol} failed ({exc}) — trying fallbacks")
            return None

    # ── Resilient single-symbol fetch ────────────────────────────────────────
    # Source priority for CBOE volatility indices:
    #   0. CBOE CDN            — authoritative, full history incl. post-2022
    #   1. yf.download()       — Yahoo primary
    #   2. yf.Ticker().history(start,end)
    #   3. yf.Ticker().history(period='max') sliced
    #   4. FRED (fred_series=) — last resort; VXSLVCLS ends Feb-2022, reached
    #      only if CBOE + all Yahoo paths fail. Yahoo now rejects any period
    #      beyond 1d/5d for ^VXSLV ("Period 'max' is invalid, must be one of:
    #      1d, 5d"), which is why CBOE is tried first.
    # Returns a Close Series indexed by date, or None if all paths fail.
    def _resilient_close(sym, start, end, retries=2, fred_series=None, cboe_symbol=None):
        import time as _t
        # Strategy 0: CBOE CDN — the good source, try first
        if cboe_symbol is not None:
            s = _load_cboe_index(cboe_symbol)
            if s is not None:
                s = s[(s.index >= pd.to_datetime(start)) & (s.index <= pd.to_datetime(end))]
                if len(s) > 20:
                    return s
        for attempt in range(retries + 1):
            # Strategy 1: download()
            try:
                r = yf.download(sym, start=start, end=end, interval="1d",
                                progress=False, auto_adjust=False)
                if r is not None and len(r) > 20:
                    c = r["Close"]
                    if hasattr(c, "columns"):      # MultiIndex → single col
                        c = c.iloc[:, 0]
                    if c.notna().sum() > 20:
                        return c.dropna()
            except Exception:
                pass
            # Strategy 2: Ticker().history() with explicit dates
            try:
                r = yf.Ticker(sym).history(start=start, end=end, interval="1d",
                                           auto_adjust=False)
                if r is not None and len(r) > 20 and r["Close"].notna().sum() > 20:
                    return r["Close"].dropna()
            except Exception:
                pass
            # Strategy 3: full history then slice to window
            try:
                r = yf.Ticker(sym).history(period="max", interval="1d",
                                           auto_adjust=False)
                if r is not None and len(r) > 20:
                    c = r["Close"].dropna()
                    c = c[(c.index >= pd.to_datetime(start).tz_localize(c.index.tz))
                          & (c.index <= pd.to_datetime(end).tz_localize(c.index.tz))] \
                        if c.index.tz is not None else \
                        c[(c.index >= pd.to_datetime(start)) & (c.index <= pd.to_datetime(end))]
                    if len(c) > 20:
                        return c
            except Exception:
                pass
            if attempt < retries:
                _t.sleep(2 * (attempt + 1))   # brief backoff before retrying

        # Strategy 4: FRED fallback (authoritative for discontinued indices)
        if fred_series is not None:
            try:
                fr = web.DataReader(fred_series, 'fred', start, end)
                s = fr.iloc[:, 0].dropna()
                if len(s) > 20:
                    print(f"    (Yahoo unavailable for {sym} — using FRED "
                          f"{fred_series}; series ends at CBOE discontinuation)")
                    return s
            except Exception:
                pass
        return None

    def _attach_index_series(df, sym, colname, start, end,
                             fred_series=None, cboe_symbol=None):
        """Fetch sym via _resilient_close and left-merge as `colname` (ffilled)."""
        s = _resilient_close(sym, start, end, fred_series=fred_series,
                             cboe_symbol=cboe_symbol)
        if s is None:
            df[colname] = np.nan
            print(f"  ⚠ {colname} unavailable after all fetch strategies — "
                  f"column set to NaN, derived features skipped")
            return df
        s = s.copy()
        s.index = pd.to_datetime(s.index).tz_localize(None) if getattr(s.index, "tz", None) is not None \
                  else pd.to_datetime(s.index)
        tmp = pd.DataFrame({colname: s.values}, index=s.index).reset_index()
        tmp.columns = ["Date", colname]
        tmp["Date"] = pd.to_datetime(tmp["Date"])
        df = pd.merge_asof(df.sort_values("Date"), tmp.sort_values("Date"),
                           on="Date", direction="backward")
        df[colname] = df[colname].ffill()
        print(f"  ✓ {colname} loaded: {int(s.notna().sum())} observations "
              f"({s.index.min().date()} – {s.index.max().date()})")
        return df

    # ── VXSLV: CBOE Silver ETF Volatility Index ───────────────────────────────
    # Implied vol from SLV options — distinct from realised vol (REALIZED_VOL_*)
    # and VIX.  Elevated VXSLV + falling price = vol-risk event; VXSLV <
    # realised = risk-premium compression.  CBOE CDN carries full history
    # (incl. the post-2022 revival), with Yahoo then FRED as fallbacks.
    df = _attach_index_series(df, "^VXSLV", "VXSLV", Config.START_DATE, end_date,
                              fred_series="VXSLVCLS", cboe_symbol="VXSLV")

    # ── GVZ: CBOE Gold ETF Volatility Index ──────────────────────────────────
    # The VXSLV/GVZ spread captures silver-specific fear premium vs gold.
    df = _attach_index_series(df, "^GVZ", "GVZ", Config.START_DATE, end_date,
                              fred_series="GVZCLS", cboe_symbol="GVZ")

    # ── VXSLV gap proxy (2022-2025 CBOE discontinuation) ──────────────────────
    # VXSLV was discontinued Feb 2022 and resumed 2025, leaving ~750 trading
    # days with no silver-vol reading mid-sample.  GVZ (gold vol) published
    # continuously through the gap, and silver/gold implied vol are tightly
    # linked, so we estimate the missing VXSLV from GVZ.
    #
    # Method: fit VXSLV ~ a + b*GVZ by OLS on the periods where BOTH are real
    # (pre-2022 and post-2025), then predict VXSLV across the gap from GVZ.
    # A new flag column VXSLV_IS_PROXY (1 in the gap, 0 on real data) is added
    # as a feature so the models can down-weight modeled values if they wish —
    # and so the proxy never masquerades as observed data.
    # Leakage note: the regression uses only VXSLV values that genuinely exist
    # (both ends of the gap); it does not peek at the target and is fit once on
    # the full history the same way the other static market features are.
    try:
        if 'VXSLV' in df.columns and 'GVZ' in df.columns:
            df['VXSLV_IS_PROXY'] = 0
            real_mask = df['VXSLV'].notna() & df['GVZ'].notna()
            gap_mask  = df['VXSLV'].isna()  & df['GVZ'].notna()
            n_gap = int(gap_mask.sum())
            if real_mask.sum() > 100 and n_gap > 0:
                x = df.loc[real_mask, 'GVZ'].to_numpy(dtype=float)
                y = df.loc[real_mask, 'VXSLV'].to_numpy(dtype=float)
                # OLS slope/intercept via numpy polyfit (deg 1)
                b, a = np.polyfit(x, y, 1)
                corr = float(np.corrcoef(x, y)[0, 1])
                pred = a + b * df.loc[gap_mask, 'GVZ'].to_numpy(dtype=float)
                # Silver vol historically >= gold vol; clip proxy to a sane
                # floor of the contemporaneous GVZ so it never implies silver
                # calmer than gold, which is virtually never true.
                pred = np.maximum(pred, df.loc[gap_mask, 'GVZ'].to_numpy(dtype=float))
                df.loc[gap_mask, 'VXSLV'] = pred
                df.loc[gap_mask, 'VXSLV_IS_PROXY'] = 1
                print(f"  ✓ VXSLV gap proxied from GVZ: {n_gap} days modeled "
                      f"(VXSLV ≈ {a:.2f} + {b:.2f}·GVZ, r={corr:.2f})")
            else:
                print(f"  · VXSLV gap proxy skipped "
                      f"(real={int(real_mask.sum())}, gap={n_gap})")
        else:
            df['VXSLV_IS_PROXY'] = 0
    except Exception as e:
        if 'VXSLV_IS_PROXY' not in df.columns:
            df['VXSLV_IS_PROXY'] = 0
        print(f"  ⚠ VXSLV gap proxy failed ({e}) — leaving gap as-is")

    print(f"✓ Loaded {len(df)} trading days")
    print(f"  Date range: {df['Date'].min()} to {df['Date'].max()}")
    
    # Report actual date range
    actual_end = df['Date'].max()
    print(f"  Actual end date available: {actual_end}")
    
    return df


def add_cot_features(df):
    """Add COT Features with better error reporting"""
    print("\n" + "-"*70)
    print("Processing COT data...")
    
    SILVER_CODE = '084691'
    cot_features = []
    
    def download_cot_year(year):
        """Download COT data for a specific year with better error handling"""
        if year <= 2016:
            url = 'https://www.cftc.gov/files/dea/history/deacot1986_2016.zip'
            print(f"    Attempting historical file (1986-2016)...")
        else:
            url = f'https://www.cftc.gov/files/dea/history/deacot{year}.zip'
        
        try:
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req, timeout=120) as response:  # Longer timeout
                data = response.read()
            
            if data[:2] != b'PK':
                print(f"    ⚠ {year}: Not a valid ZIP file")
                return None
            
            with zipfile.ZipFile(io.BytesIO(data)) as z:
                csv_files = [f for f in z.namelist() if f.endswith('.txt') or f.endswith('.csv')]
                if not csv_files:
                    print(f"    ⚠ {year}: No CSV/TXT files in ZIP")
                    return None
                print(f"    Found file: {csv_files[0]}")
                with z.open(csv_files[0]) as f:
                    return pd.read_csv(f, low_memory=False)
        except urllib.error.HTTPError as e:
            print(f"    ⚠ {year}: HTTP Error {e.code}")
            return None
        except urllib.error.URLError as e:
            print(f"    ⚠ {year}: URL Error - {e.reason}")
            return None
        except Exception as e:
            print(f"    ⚠ {year}: Error - {type(e).__name__}: {e}")
            return None
    
    try:
        current_year = datetime.now().year
        all_data = []
        
        # Try to get historical data (2016 and earlier)
        print("  Attempting historical COT data (pre-2017)...")
        hist_df = download_cot_year(2016)
        if hist_df is not None:
            all_data.append(hist_df)
            print(f"    ✓ Historical data loaded: {len(hist_df)} rows")
        else:
            print(f"    ⚠ Historical data not available - will use 2017+ only")
        
        # Get recent years
        print("  Downloading recent COT data...")
        for year in range(2017, current_year + 1):
            year_df = download_cot_year(year)
            if year_df is not None:
                all_data.append(year_df)
                print(f"    ✓ {year}: {len(year_df)} rows")
            else:
                print(f"    ⚠ {year}: Failed")
        
        if not all_data:
            raise Exception("No COT data downloaded")
        
        combined = pd.concat(all_data, ignore_index=True)
        print(f"  Combined: {len(combined)} total rows")
        
        # Find code column
        code_col = None
        for col in ['CFTC Contract Market Code', 'CFTC_Contract_Market_Code', 'Contract Market Code']:
            if col in combined.columns:
                code_col = col
                break
        
        if code_col is None:
            print(f"  Available columns: {list(combined.columns)[:10]}...")
            raise Exception("Contract code column not found")
        
        silver_cot = combined[combined[code_col].astype(str) == SILVER_CODE].copy()
        
        if len(silver_cot) == 0:
            raise Exception(f"No silver COT data found for code {SILVER_CODE}")
        
        print(f"  Silver COT records: {len(silver_cot)}")
        
        # Find date column
        date_col = None
        for col in ['As of Date in Form YYYY-MM-DD', 'Report_Date_as_YYYY-MM-DD', 'Report Date']:
            if col in silver_cot.columns:
                date_col = col
                break
        
        if date_col:
            silver_cot['cot_date'] = pd.to_datetime(silver_cot[date_col], errors='coerce')
            silver_cot = silver_cot.dropna(subset=['cot_date'])
            silver_cot = silver_cot.sort_values('cot_date').reset_index(drop=True)
            print(f"  Date range: {silver_cot['cot_date'].min()} to {silver_cot['cot_date'].max()}")
        
        # Map columns
        col_map = {
            'comm_long': 'Commercial Positions-Long (All)',
            'comm_short': 'Commercial Positions-Short (All)',
            'noncomm_long': 'Noncommercial Positions-Long (All)',
            'noncomm_short': 'Noncommercial Positions-Short (All)',
            'open_interest': 'Open Interest (All)'
        }
        
        for new_col, old_col in col_map.items():
            if old_col in silver_cot.columns:
                silver_cot[new_col] = pd.to_numeric(silver_cot[old_col], errors='coerce')
        
        silver_cot['comm_net'] = silver_cot['comm_long'] - silver_cot['comm_short']
        silver_cot['noncomm_net'] = silver_cot['noncomm_long'] - silver_cot['noncomm_short']

        # ── Williams-style COT Index (percentile rank) ───────────────────────
        # The signal COT practitioners (Larry Williams et al.) actually trade is
        # the POSITION WITHIN THE MULTI-YEAR RANGE, expressed 0-100:
        #     index = (current - min) / (max - min)  over a long lookback
        # This is more robust than a z-score for COT because commercial
        # positioning is fat-tailed and skewed — a z-score is distorted by the
        # very extremes that carry the signal, while a rank is not.  Williams
        # uses a 3-year (156-week) window so the extreme is a MULTI-YEAR
        # extreme, which a 52-week z-score structurally cannot see.
        LB_LONG  = 156     # ~3 years — Williams' canonical COT-index window
        LB_MED   = 52      # 1 year — kept for the shorter-cycle z-scores

        def _cot_index(s, lb):
            roll = s.rolling(lb, min_periods=max(26, lb // 3))
            lo, hi = roll.min(), roll.max()
            rng = (hi - lo)
            idx = (s - lo) / rng.where(rng > 0, np.nan)
            return (idx * 100).clip(0, 100)

        silver_cot['comm_cot_index']    = _cot_index(silver_cot['comm_net'],    LB_LONG)
        silver_cot['noncomm_cot_index'] = _cot_index(silver_cot['noncomm_net'], LB_LONG)
        # Extremes (Williams: <20 = commercials heavily net-long vs their own
        # range → bullish; >80 → bearish).  Two binary flags for the trees.
        silver_cot['comm_cot_bullish_ext'] = (silver_cot['comm_cot_index'] < 20).astype(int)
        silver_cot['comm_cot_bearish_ext'] = (silver_cot['comm_cot_index'] > 80).astype(int)

        # z-scores retained (shorter cycle info the rank doesn't carry), now on
        # BOTH a 1-year and 3-year window
        lookback = LB_MED
        silver_cot['comm_net_zscore'] = (
            silver_cot['comm_net'] - silver_cot['comm_net'].rolling(lookback).mean()
        ) / (silver_cot['comm_net'].rolling(lookback).std() + 1)

        silver_cot['noncomm_net_zscore'] = (
            silver_cot['noncomm_net'] - silver_cot['noncomm_net'].rolling(lookback).mean()
        ) / (silver_cot['noncomm_net'].rolling(lookback).std() + 1)

        silver_cot['comm_net_chg_1w'] = silver_cot['comm_net'].diff(1)
        silver_cot['comm_net_chg_4w'] = silver_cot['comm_net'].diff(4)
        silver_cot['noncomm_net_chg_1w'] = silver_cot['noncomm_net'].diff(1)
        silver_cot['noncomm_net_chg_4w'] = silver_cot['noncomm_net'].diff(4)

        # COT momentum: the most actionable signal is the RATE OF CHANGE in commercial
        # positioning — especially when commercials are covering shorts (reducing short
        # exposure) even while still net-short, which historically precedes rallies.
        silver_cot['comm_covering'] = (
            (silver_cot['comm_net_chg_1w'] > 0) &
            (silver_cot['comm_net'] < 0)
        ).astype(int)
        # Sustained 4-week covering = strongest entry signal
        silver_cot['covering_momentum'] = silver_cot['comm_net_chg_1w'].rolling(4).sum()

        if 'open_interest' in silver_cot.columns:
            silver_cot['oi_chg_1w'] = silver_cot['open_interest'].diff(1)
            silver_cot['oi_chg_4w'] = silver_cot['open_interest'].diff(4)

        # Composite signal now anchored on the Williams COT INDEX (rank),
        # centered to [-1, 1] so 0 = mid-range: (50 - index)/50 makes low
        # commercial-index readings (bullish) positive.  z-scores add the
        # shorter-cycle overlay.
        _comm_rank_signal = (50.0 - silver_cot['comm_cot_index']) / 50.0
        silver_cot['cot_signal'] = (
            _comm_rank_signal.fillna(0) * 0.5 +
            silver_cot['comm_net_zscore'].clip(-2, 2) / 2 * 0.3 +
            -silver_cot['noncomm_net_zscore'].clip(-2, 2) / 2 * 0.2
        ).clip(-1, 1)

        # NOTE: raw level columns comm_net / noncomm_net are intentionally
        # kept OUT of the model feature set — they are non-stationary contract
        # counts that drift with open interest and carry little for a tree
        # model.  comm_net is still carried into df (unregistered) because the
        # downstream MIDAS macro-vol index consumes its rate-of-change.
        cot_cols = ['comm_cot_index', 'noncomm_cot_index',
                    'comm_cot_bullish_ext', 'comm_cot_bearish_ext',
                    'comm_net_zscore', 'noncomm_net_zscore',
                    'comm_net_chg_1w', 'comm_net_chg_4w',
                    'noncomm_net_chg_1w', 'noncomm_net_chg_4w', 'cot_signal',
                    'comm_covering', 'covering_momentum']
        cot_carry_only = ['comm_net']   # merged & COT_-prefixed but NOT a model feature

        if 'open_interest' in silver_cot.columns:
            cot_cols.extend(['oi_chg_1w', 'oi_chg_4w'])

        df['Date'] = pd.to_datetime(df['Date'])
        _all_merge = cot_cols + [c for c in cot_carry_only if c in silver_cot.columns]
        cot_merge = silver_cot[['cot_date'] + [c for c in _all_merge if c in silver_cot.columns]].copy()
        cot_merge = cot_merge.rename(columns={'cot_date': 'Date'})
        
        df = df.sort_values('Date')
        cot_merge = cot_merge.sort_values('Date')
        df = pd.merge_asof(df, cot_merge, on='Date', direction='backward')
        
        for col in cot_cols:
            if col in df.columns:
                df.rename(columns={col: f'COT_{col}'}, inplace=True)
                cot_features.append(f'COT_{col}')

        # Carry-only: prefix for downstream consumers (MIDAS uses COT_comm_net)
        # but do NOT append to cot_features, so it never enters model selection.
        for col in cot_carry_only:
            if col in df.columns:
                df.rename(columns={col: f'COT_{col}'}, inplace=True)
                if f'COT_{col}' in df.columns:
                    df[f'COT_{col}'] = df[f'COT_{col}'].fillna(method='ffill').fillna(0)
        
        for col in cot_features:
            if col in df.columns:
                # ffill only — COT prints weekly; a bar carries the last
                # PUBLISHED report. bfill would pull a future report backward
                # into the warmup (same leak class fixed elsewhere).
                df[col] = df[col].fillna(method='ffill').fillna(0)
        
        print(f"  ✓ Added {len(cot_features)} COT features")
        
    except Exception as e:
        print(f"  ⚠ COT processing error: {e}")
        print(f"  Creating placeholder COT features...")
        cot_features = ['COT_comm_net', 'COT_noncomm_net', 'COT_cot_signal']
        for feat in cot_features:
            df[feat] = 0
    
    return df, cot_features


# ══════════════════════════════════════════════════════════════════════
# UN COMTRADE PHYSICAL SILVER TRADE FLOWS (API v1 'Plus' architecture)
# HYBRID DIRECT + MIRROR EDITION
# ══════════════════════════════════════════════════════════════════════
# Empirical basis (three real Comtrade pulls, Jul-Aug 2026):
#   • China and Peru stopped publishing — 0 rows as reporters across every
#     tested month.  Their flows are reconstructed from MIRROR data: what a
#     fixed panel of counterparties reports trading WITH them.
#   • All-mirror was tested and REJECTED: on 6 hubs x 5 overlapping months,
#     mirror/direct level ratios ranged 0.15x-15.7x (median 1.04 only by
#     cancellation).  But mirror TRACKS direct (median corr 0.72), so it is
#     acceptable where no direct data exists, feeding transforms (yoy/mom/z)
#     that care about variation, not level.
#   • netWgt is 98.5% populated; weight-missing rows carry 1.1% of value.
#     Kilograms stay the foundation (primaryValue = price x qty would leak
#     the forecast target into the features).
#   • BUT implied unit values span $708-$3,079/kg (p10-p90) for bullion vs
#     ~$1,900 spot — misreported weights, not economics.  A unit-value
#     plausibility filter removes them (see _comtrade_clean_rows).
#   • Publication decay: 3 months back = ~86% reporter coverage, 4 months
#     = 100%.  Lag moved 3 -> 4.
#   • Powder (710610) is 18.5% of mass at $125-$3,364/kg implied — variable
#     purity, not pure metal.  It is quarantined: never mass-summed with
#     bullion, and excluded from the premium proxy.

# ── Comtrade configuration ────────────────────────────────────────────────
COMTRADE_BASE_URL      = "https://comtradeapi.un.org/data/v1/get/C/M/HS"
COMTRADE_API_KEY_ENV   = "UN_COMTRADE_API_KEY"   # set via: export UN_COMTRADE_API_KEY="..."
COMTRADE_CACHE_MAX_AGE_DAYS = 7      # re-pull weekly; monthly data changes slowly
COMTRADE_PUBLICATION_LAG_M  = 4      # 4m = 100% reporter coverage (was 3m = ~86%)
COMTRADE_PERIODS_PER_CALL   = 12     # documented max period values per request
COMTRADE_REQUEST_PAUSE_SEC  = 1.2
COMTRADE_MAX_RETRIES        = 4
COMTRADE_RETRY_BACKOFF_SEC  = 15.0
COMTRADE_TIMEOUT_SEC        = 60

COMTRADE_HS_CODES = {
    "710691": "Bullion",   # Unwrought silver — investment / macro sentiment
    "710610": "Powder",    # Silver powder  — solar, 5G, EV industrial demand
}

# ── DIRECT hubs: countries that still publish (partner = 0 World) ─────────
COMTRADE_EXPORTERS = {        # PURE SUPPLY CORRIDORS — Exports only
    "484": "Mexico",
    "124": "Canada",
    "616": "Poland",          # KGHM by-product silver — Europe's supply anchor
}
COMTRADE_IMPORTERS = {        # PURE DEMAND CORRIDORS — Imports only
    "842": "US",
    "356": "India",           # swing investment demand
    "392": "Japan",           # high-tech / electronics
    "276": "Germany",         # automotive
    "756": "Switzerland",     # global refining hub
}
COMTRADE_DUAL_HUBS = {        # DUAL-MARKET ENGINE HUBS — pull BOTH M and X
    "826": "UK",              # institutional LBMA vault turnover
}

# ── MIRROR hubs: non-reporters, reconstructed from counterparty filings ───
# Flow inversion is mandatory: the hub's IMPORTS are the panel's EXPORTS to
# it (flowCode='X', partner=hub) and vice versa.  Getting this backwards
# silently flips the sign of China_Net_Bullion_Demand.
COMTRADE_MIRROR_HUBS = {
    "156": "China",
    "604": "Peru",
}
# Fixed counterparty panel — a stable panel keeps the mirror series
# comparable across time ("whoever happened to file" does not).  Derived
# from actual 2026 counterparty concentration (HK+Macau = 88% of the China
# mirror — the entrepot route), widened with Peru's historical buyers
# (US, India, Korea, Switzerland, Germany) so earlier years are covered.
# KNOWN BLIND SPOT: Peru->China direct shipments are invisible — neither
# country reports, and no third party sees the flow.
COMTRADE_MIRROR_PANEL = {
    "344": "HongKong", "446": "Macau",  "392": "Japan",  "826": "UK",
    "842": "US",       "757": "Switzerland", "764": "Thailand",
    "410": "Korea",    "356": "India",  "276": "Germany",
    "124": "Canada",   "36":  "Australia",
}

COMTRADE_FLOW_LABEL = {"X": "Exports", "M": "Imports"}

# Unit-value plausibility bounds, as multiples of the month's cross-
# sectional MEDIAN bullion unit value (a robust, self-calibrating spot
# proxy that works before the price dataframe starts and needs no external
# coupling).  Bullion outside [0.4x, 2.5x] is a misreported weight.  Powder
# bounds are loose on the downside because low-purity powder is REAL
# (India files $94-146/kg powder) — only data-entry junk is cut.
COMTRADE_UV_BOUNDS = {
    "710691": (0.40, 2.50),
    "710610": (0.02, 3.00),
}

# Cache auto-invalidation: filename embeds a config fingerprint, so any edit
# to the hub/HS/panel universe forces a fresh pull instead of silently
# serving a cached matrix with the old column set.
import hashlib as _cmt_hashlib
_cmt_cfg_sig = "hybrid-mirror-v3-lag4|" + "|".join(sorted(
    list(COMTRADE_EXPORTERS) + list(COMTRADE_IMPORTERS) +
    [c + "D" for c in COMTRADE_DUAL_HUBS] +
    [c + "R" for c in COMTRADE_MIRROR_HUBS] +
    list(COMTRADE_MIRROR_PANEL) + list(COMTRADE_HS_CODES)))
COMTRADE_CACHE_FILE = (
    f"comtrade_silver_cache_{_cmt_hashlib.md5(_cmt_cfg_sig.encode()).hexdigest()[:8]}.csv")


def _comtrade_build_periods():
    """
    Monthly 'YYYYMM' periods from (lag + 36) months BEFORE Config.START_DATE
    through the last completed month, so every trading day from 2016-02-24
    onward carries fully-formed transformed features (24m z-score + 12m YoY
    + publication lag) — no NaN head, no bfill.
    """
    start_p = pd.Period(Config.START_DATE[:7], freq="M") - (COMTRADE_PUBLICATION_LAG_M + 36)
    end_p   = pd.Period(datetime.now().strftime("%Y-%m"), freq="M") - 1  # last full month
    return [str(p).replace("-", "") for p in pd.period_range(start_p, end_p, freq="M")]


def _comtrade_fetch_batch(session, api_key, reporter_codes, flow_code,
                          period_chunk, partner_codes="0"):
    """
    One batched GET.  Direct mode: reporters = hubs, partner_codes='0'
    (World).  Mirror mode: reporters = counterparty panel, partner_codes =
    the non-reporting hubs.  Totals only (partner2=0, all transport modes,
    all customs procedures).  Returns the raw record list ([] on empty /
    persistent failure).
    """
    import time as _time
    params = {
        "reporterCode": ",".join(reporter_codes),
        "period":       ",".join(period_chunk),
        "partnerCode":  partner_codes,
        "partner2Code": "0",
        "cmdCode":      ",".join(COMTRADE_HS_CODES.keys()),
        "flowCode":     flow_code,
        "motCode":      "0",
        "customsCode":  "C00",
        "includeDesc":  "false",
    }
    headers = {"Ocp-Apim-Subscription-Key": api_key}

    for attempt in range(1, COMTRADE_MAX_RETRIES + 1):
        try:
            resp = session.get(COMTRADE_BASE_URL, params=params,
                               headers=headers, timeout=COMTRADE_TIMEOUT_SEC)
        except Exception as exc:
            print(f"    ⚠ Comtrade network error (attempt {attempt}/{COMTRADE_MAX_RETRIES}): {exc}")
            _time.sleep(COMTRADE_RETRY_BACKOFF_SEC * attempt)
            continue

        if resp.status_code == 200:
            payload = resp.json()
            records = payload.get("data")
            return records if records is not None else []

        if resp.status_code == 429:
            print(f"    ⚠ Comtrade rate limited — backing off "
                  f"{COMTRADE_RETRY_BACKOFF_SEC * attempt:.0f}s "
                  f"(attempt {attempt}/{COMTRADE_MAX_RETRIES})")
            _time.sleep(COMTRADE_RETRY_BACKOFF_SEC * attempt)
            continue

        if resp.status_code in (401, 403):
            raise RuntimeError(
                f"Comtrade auth failure ({resp.status_code}) — check that the key in "
                f"${COMTRADE_API_KEY_ENV} is active.")

        print(f"    ⚠ Comtrade HTTP {resp.status_code} "
              f"(attempt {attempt}/{COMTRADE_MAX_RETRIES})")
        _time.sleep(COMTRADE_RETRY_BACKOFF_SEC * attempt)

    return []


def _comtrade_clean_rows(d):
    """
    Row-level quality control, applied to direct AND mirror rows alike.

    1. Weight sanity: netWgt must be positive (missing-weight rows carry
       only 1.1% of total value — cheap to drop, and keeping them would
       force a value-based imputation that leaks price).
    2. Estimated-row exclusion: UNSD back-fills unreported recent months
       with modeled values (isNetWgtEstimated) — 16% of recent mass.  With
       the 4-month lag most estimates have been replaced by actuals; the
       remainder are dropped so features reflect REPORTED trade only.
    3. Unit-value plausibility: implied USD/kg is compared to the month's
       cross-sectional median BULLION unit value.  Bullion at 0.15x or 6x
       the reference is a decimal-point error in the weight, not commerce.
       This is the one legitimate job for primaryValue: validating netWgt.

    Returns (clean_df, n_dropped_uv, n_dropped_est).
    """
    n0 = len(d)
    d = d[d["netWgt"] > 0].copy()

    n_est = 0
    if "isNetWgtEstimated" in d.columns:
        est_mask = d["isNetWgtEstimated"].astype(str).str.lower().isin(["true", "1"])
        n_est = int(est_mask.sum())
        d = d[~est_mask].copy()

    n_before_uv = len(d)
    d["_uv"] = d["primaryValue"] / d["netWgt"]
    ref = (d[d["cmdCode"] == "710691"].groupby("period")["_uv"].median()
             .rename("_uv_ref"))
    d = d.merge(ref, on="period", how="left")
    _global_ref = d.loc[d["cmdCode"] == "710691", "_uv"].median()
    d["_uv_ref"] = d["_uv_ref"].fillna(_global_ref)

    # Bool computed via plain numpy arrays (never pandas Series-on-Series
    # assignment) so there is no path to an object-dtype ~True/~False
    # bitwise-NOT wraparound corrupting the drop count.
    uv      = d["_uv"].to_numpy(dtype=float)
    uv_ref  = d["_uv_ref"].to_numpy(dtype=float)
    cmd     = d["cmdCode"].to_numpy()
    keep = np.ones(len(d), dtype=bool)
    for c, (lo, hi) in COMTRADE_UV_BOUNDS.items():
        m = (cmd == c)
        if not m.any():
            continue
        in_range = (uv[m] >= lo * uv_ref[m]) & (uv[m] <= hi * uv_ref[m])
        # A row with a missing/zero VALUE (uv is NaN) has a perfectly usable
        # netWgt and was never actually tested for plausibility — keep it
        # rather than silently dropping it as if it failed the check.
        in_range = np.where(np.isnan(uv[m]), True, in_range)
        keep[m] = in_range

    d = d[keep].drop(columns=["_uv", "_uv_ref"])
    # Counts derived from actual row totals, not a boolean reduction — this
    # is non-negative by construction regardless of any upstream dtype issue.
    n_uv = max(0, n_before_uv - len(d))
    return d, n_uv, n_est


def _comtrade_pull_monthly_matrix():
    """
    Pull + pivot the raw monthly matrix (UNLAGGED, month-start Datetime
    index).  Direct hubs come from their own filings (partner=World).
    Mirror hubs (China, Peru) are reconstructed from the counterparty
    panel's filings, WITH FLOW INVERSION, and summed across the panel.
    Columns: '<Hub>_<Flow>_<HS>__wgt' and '__val' — identical naming for
    direct and mirror, so the aggregate builder is source-agnostic.
    """
    import requests as _requests
    import time as _time

    api_key = os.environ.get(COMTRADE_API_KEY_ENV, "").strip()
    if not api_key:
        raise RuntimeError(f"${COMTRADE_API_KEY_ENV} not set")

    periods = _comtrade_build_periods()
    period_chunks = [periods[i:i + COMTRADE_PERIODS_PER_CALL]
                     for i in range(0, len(periods), COMTRADE_PERIODS_PER_CALL)]

    export_reporters = list(COMTRADE_EXPORTERS) + list(COMTRADE_DUAL_HUBS)
    import_reporters = list(COMTRADE_IMPORTERS) + list(COMTRADE_DUAL_HUBS)
    mirror_partners  = ",".join(COMTRADE_MIRROR_HUBS)
    panel_reporters  = list(COMTRADE_MIRROR_PANEL)

    # (reporters, flow, partner_codes, is_mirror)
    request_groups = [
        (export_reporters, "X", "0",             False),
        (import_reporters, "M", "0",             False),
        (panel_reporters,  "X", mirror_partners, True),   # panel X -> hub IMPORTS
        (panel_reporters,  "M", mirror_partners, True),   # panel M -> hub EXPORTS
    ]

    direct_records, mirror_records = [], []
    with _requests.Session() as session:
        for reporter_codes, flow_code, partners, is_mirror in request_groups:
            for chunk in period_chunks:
                tag = "MIRROR" if is_mirror else "direct"
                print(f"    Fetching [{tag}] flow={flow_code} "
                      f"periods={chunk[0]}..{chunk[-1]}")
                records = _comtrade_fetch_batch(session, api_key, reporter_codes,
                                                flow_code, chunk, partners)
                (mirror_records if is_mirror else direct_records).extend(records)
                _time.sleep(COMTRADE_REQUEST_PAUSE_SEC)

    if not direct_records and not mirror_records:
        return pd.DataFrame()

    def _prep(records):
        raw = pd.DataFrame(records)
        if raw.empty:
            return raw
        if "partner2Code" in raw.columns:
            raw = raw[raw["partner2Code"].astype(str) == "0"]
        if "motCode" in raw.columns:
            raw = raw[raw["motCode"].astype(str) == "0"]
        if "customsCode" in raw.columns:
            raw = raw[raw["customsCode"].astype(str) == "C00"]
        need = ["period", "reporterCode", "partnerCode", "flowCode", "cmdCode",
                "primaryValue", "netWgt"]
        if any(c not in raw.columns for c in need):
            return pd.DataFrame()
        keep_cols = need + [c for c in ["isNetWgtEstimated"] if c in raw.columns]
        d = raw[keep_cols].copy()
        for c in ["period", "reporterCode", "partnerCode", "cmdCode"]:
            d[c] = d[c].astype(str)
        d["primaryValue"] = pd.to_numeric(d["primaryValue"], errors="coerce")
        d["netWgt"]       = pd.to_numeric(d["netWgt"], errors="coerce")
        return d

    frames = []

    # ── DIRECT: hub is the reporter, partner must be World ────────────────
    dd = _prep(direct_records)
    if not dd.empty:
        dd = dd[dd["partnerCode"] == "0"]
        hub_names = {**COMTRADE_EXPORTERS, **COMTRADE_IMPORTERS, **COMTRADE_DUAL_HUBS}
        dd["hub"]  = dd["reporterCode"].map(hub_names)
        dd["flow"] = dd["flowCode"].map(COMTRADE_FLOW_LABEL)
        dd = dd.dropna(subset=["hub", "flow"])
        frames.append(dd)

    # ── MIRROR: hub is the PARTNER, flow is INVERTED ──────────────────────
    md = _prep(mirror_records)
    if not md.empty:
        md = md[md["partnerCode"].isin(COMTRADE_MIRROR_HUBS)]
        md = md[md["reporterCode"].isin(COMTRADE_MIRROR_PANEL)]
        md["hub"] = md["partnerCode"].map(COMTRADE_MIRROR_HUBS)
        # Panel exports TO the hub are the hub's IMPORTS, and vice versa.
        md["flow"] = md["flowCode"].map({"X": "Imports", "M": "Exports"})
        md = md.dropna(subset=["hub", "flow"])
        frames.append(md)

    if not frames:
        return pd.DataFrame()
    d = pd.concat(frames, ignore_index=True)

    # ── Row-level QC (weight sanity, estimated rows, unit-value filter) ───
    d = d.dropna(subset=["primaryValue", "netWgt"], how="all")
    d, n_uv, n_est = _comtrade_clean_rows(d)
    if n_uv or n_est:
        print(f"    QC: dropped {n_uv} implausible-unit-value rows, "
              f"{n_est} UNSD-estimated rows")

    d["feature"] = d["hub"] + "_" + d["flow"] + "_" + d["cmdCode"]
    d = (d.groupby(["period", "feature"], as_index=False)[["netWgt", "primaryValue"]]
           .sum(min_count=1))

    wgt = d.pivot(index="period", columns="feature", values="netWgt").add_suffix("__wgt")
    val = d.pivot(index="period", columns="feature", values="primaryValue").add_suffix("__val")
    matrix = pd.concat([wgt, val], axis=1)
    matrix.index = pd.to_datetime(matrix.index, format="%Y%m")
    matrix = matrix.sort_index()
    matrix.index.name = "Date"
    matrix.columns.name = None
    return matrix


def add_comtrade_silver_trade_features(df):
    """
    Add physical silver trade features from UN Comtrade — 9 macro-aggregates
    on a netWgt (kilogram) foundation, each with 3 stationarity transforms.

    SOURCING: direct filings for 9 reporting hubs; MIRROR reconstruction for
    China and Peru (non-reporters) from a fixed 12-country counterparty
    panel with flow inversion.  Mirror levels are biased (entrepot effects,
    partial panel) but track direct data (corr ~0.7) — acceptable for the
    transform-dominated feature set.  China/Peru aggregates are therefore
    LOWER-CONFIDENCE than the direct seven.

    THE 9 AGGREGATES (bullion/powder never mass-mixed):
      1. Supply_Mined_Proxy            Σwgt MX+CA+PL(direct)+PE(mirror) X, 710691
      2. Demand_Industrial_Proxy       Σwgt US+JP+DE imports, 710691 ONLY
                                       (powder split out — it is variable-
                                       purity material, not metal)
      3. Demand_Investment_Proxy       Σwgt IN+CH(Swiss) imports, 710691
      4. UK_Net_Vault_Flow             UK M − X, 710691 (LBMA hoard/liquidate)
      5. China_Solar_Powder_Demand     China M, 710610 [MIRROR]
      6. China_Net_Bullion_Demand      China M − X, 710691 [MIRROR]
      7. Global_SD_Imbalance           (2 + 3) − 1  (all-bullion, apples to
                                       apples)
      8. Global_Powder_Demand          Σwgt US+JP+DE(direct)+China(mirror)
                                       powder imports — the tech-cycle pull
                                       in physical units (replaces the old
                                       powder/bullion mass ratio, which mixed
                                       purities and was dominated by gaps)
      9. Physical_Premium_Proxy        Σval/Σwgt on 710691 rows ONLY —
                                       blending powder injected a fake 15%
                                       wedge that moved with product mix

    TRANSFORMS (x4 incl. level): _yoy, _mom3m, _z24m — pct_change for
    positive-definite series, 12m/3m DIFFERENCES for signed net flows.
    Publication lag: 4 months (100% reporter coverage vs 86% at 3m).
    Leakage rules unchanged: ffill only, shift BEFORE daily resample,
    merge_asof backward.  On total failure returns (df, []).
    """
    print("\n" + "-"*70)
    print("[QUANT] UN Comtrade Physical Silver Trade Flows "
          "(hybrid direct+mirror, 9 aggregates)...")

    monthly = None

    # ── 1. Cache-first load ──────────────────────────────────────────────────
    cache_fresh = False
    if os.path.exists(COMTRADE_CACHE_FILE):
        cache_age_days = (datetime.now().timestamp()
                          - os.path.getmtime(COMTRADE_CACHE_FILE)) / 86400.0
        cache_fresh = cache_age_days < COMTRADE_CACHE_MAX_AGE_DAYS
        if cache_fresh:
            try:
                monthly = pd.read_csv(COMTRADE_CACHE_FILE, index_col=0, parse_dates=True)
                print(f"  ✓ Using cached Comtrade data ({cache_age_days:.1f} days old)")
            except Exception as e:
                print(f"  ⚠ Cache unreadable ({e}) — will pull fresh")
                monthly = None

    if monthly is None:
        try:
            monthly = _comtrade_pull_monthly_matrix()
            if monthly is not None and not monthly.empty:
                monthly.to_csv(COMTRADE_CACHE_FILE)
                n_wgt = sum(1 for c in monthly.columns if c.endswith("__wgt"))
                print(f"  ✓ Pulled {n_wgt} physical (netWgt) series, "
                      f"{monthly.index.min().date()} → {monthly.index.max().date()} "
                      f"(cached to {COMTRADE_CACHE_FILE})")
        except Exception as e:
            print(f"  ⚠ Comtrade pull failed: {e}")
            monthly = None
        if (monthly is None or monthly.empty) and os.path.exists(COMTRADE_CACHE_FILE):
            try:
                monthly = pd.read_csv(COMTRADE_CACHE_FILE, index_col=0, parse_dates=True)
                print("  ⚠ Falling back to stale local cache")
            except Exception:
                monthly = None

    if monthly is None or monthly.empty:
        print("  ⚠ No Comtrade data available — skipping these features "
              f"(set ${COMTRADE_API_KEY_ENV} to enable)")
        return df, []
    if not any(c.endswith("__wgt") for c in monthly.columns):
        print("  ⚠ Cached/pulled matrix has no netWgt columns (old cache format?) "
              "— skipping these features; delete the cache file and rerun")
        return df, []

    # ── 2. Complete monthly calendar + causal gap fill ───────────────────────
    full_calendar = pd.date_range(monthly.index.min(), monthly.index.max(), freq="MS")
    monthly = monthly.reindex(full_calendar).ffill()

    # ── 3. Build the 9 aggregates on the netWgt foundation ───────────────────
    def _wgt_sum(names):
        avail = [f"{n}__wgt" for n in names if f"{n}__wgt" in monthly.columns]
        if not avail:
            return pd.Series(np.nan, index=monthly.index)
        return monthly[avail].sum(axis=1, min_count=1)

    def _wgt_one(name):
        col = f"{name}__wgt"
        return monthly[col] if col in monthly.columns else pd.Series(np.nan, index=monthly.index)

    agg = pd.DataFrame(index=monthly.index)

    # 1. Mined supply — direct MX/CA/PL + mirror Peru, unwrought bullion
    agg["Supply_Mined_Proxy"] = _wgt_sum(
        ["Mexico_Exports_710691", "Canada_Exports_710691",
         "Poland_Exports_710691", "Peru_Exports_710691"])

    # 2. Industrial demand — advanced-economy imports, BULLION ONLY
    agg["Demand_Industrial_Proxy"] = _wgt_sum(
        ["US_Imports_710691", "Japan_Imports_710691", "Germany_Imports_710691"])

    # 3. Investment demand — swing consumer + refining hub, bullion only
    agg["Demand_Investment_Proxy"] = _wgt_sum(
        ["India_Imports_710691", "Switzerland_Imports_710691"])

    # 4. LBMA vault turnover — UK net bullion flow (direct)
    agg["UK_Net_Vault_Flow"] = (_wgt_one("UK_Imports_710691")
                                - _wgt_one("UK_Exports_710691"))

    # 5. Chinese green-tech pull — powder imports [MIRROR]
    agg["China_Solar_Powder_Demand"] = _wgt_one("China_Imports_710610")

    # 6. Chinese internal macro/monetary trend — net bullion [MIRROR]
    agg["China_Net_Bullion_Demand"] = (_wgt_one("China_Imports_710691")
                                       - _wgt_one("China_Exports_710691"))

    # 7. Structural deficit metric — now all-bullion on both sides
    agg["Global_SD_Imbalance"] = (agg["Demand_Industrial_Proxy"]
                                  + agg["Demand_Investment_Proxy"]
                                  - agg["Supply_Mined_Proxy"])

    # 8. Tech-cycle pull in physical units — powder imports across demand
    #    hubs (direct) + China (mirror).  Powder-only: no purity mixing.
    agg["Global_Powder_Demand"] = _wgt_sum(
        ["US_Imports_710610", "Japan_Imports_710610",
         "Germany_Imports_710610", "China_Imports_710610"])

    # 9. Implied trade price (USD/kg) — BULLION ROWS ONLY, the sole use of
    #    primaryValue.  CIF/FOB mixing remains a known residual wedge.
    _b_val = [c for c in monthly.columns if c.endswith("_710691__val")]
    _b_wgt = [c for c in monthly.columns if c.endswith("_710691__wgt")]
    _val_tot = (monthly[_b_val].sum(axis=1, min_count=1)
                if _b_val else pd.Series(np.nan, index=monthly.index))
    _wgt_tot = (monthly[_b_wgt].sum(axis=1, min_count=1)
                if _b_wgt else pd.Series(np.nan, index=monthly.index))
    agg["Physical_Premium_Proxy"] = _val_tot / _wgt_tot.replace(0, np.nan)

    agg = agg.replace([np.inf, -np.inf], np.nan).ffill()   # causal only, no bfill

    # ── 4. Stationarity transforms — ONLY on the 9 aggregates ────────────────
    SIGNED_AGGS = {"UK_Net_Vault_Flow", "China_Net_Bullion_Demand",
                   "Global_SD_Imbalance"}
    pieces = {}
    for name in agg.columns:
        s = agg[name]
        if name in SIGNED_AGGS:
            yoy  = s.diff(12)
            mom3 = s.diff(3)
        else:
            yoy  = s.pct_change(12)
            mom3 = s.pct_change(3)
        roll = s.rolling(24, min_periods=24)
        z24  = (s - roll.mean()) / roll.std().replace(0, np.nan)
        pieces[name]            = s
        pieces[f"{name}_yoy"]   = yoy.replace([np.inf, -np.inf], np.nan)
        pieces[f"{name}_mom3m"] = mom3.replace([np.inf, -np.inf], np.nan)
        pieces[f"{name}_z24m"]  = z24.replace([np.inf, -np.inf], np.nan)
    out = pd.DataFrame(pieces, index=agg.index)

    # ── 5. Strict publication-lag shift (BEFORE daily resample) ──────────────
    out = out.shift(COMTRADE_PUBLICATION_LAG_M)
    lag_sfx = f"_lag{COMTRADE_PUBLICATION_LAG_M}m"
    out.columns = [f"CMT_{c}{lag_sfx}" for c in out.columns]

    # ── 6. Causal daily join onto the trading calendar ───────────────────────
    out = out.reset_index().rename(columns={"index": "Date"})
    out["Date"] = pd.to_datetime(out["Date"])
    df["Date"] = pd.to_datetime(df["Date"])
    df = pd.merge_asof(df.sort_values("Date"), out.sort_values("Date"),
                       on="Date", direction="backward")

    comtrade_features = [c for c in out.columns if c != "Date"]
    n_missing_aggs = int(agg.iloc[-1].isna().sum())
    for col in comtrade_features:
        df[col] = df[col].ffill().fillna(0)

    print(f"  ✓ Added {len(comtrade_features)} Comtrade features "
          f"(9 aggregates × [level, yoy, mom3m, z24m]) | lag = "
          f"{COMTRADE_PUBLICATION_LAG_M}m | netWgt foundation | "
          f"China/Peru via mirror panel")
    if n_missing_aggs > 0:
        print(f"  ⚠ {n_missing_aggs}/9 aggregates had no underlying data at the "
              f"latest month (check which hub series the API returned)")

    return df, comtrade_features


def add_real_rate_features(df):
    """Add Real Interest Rate features"""
    print("\n" + "-"*70)
    print("[QUANT] Real Interest Rate Features...")
    
    features = []
    
    try:
        series_to_fetch = {
            'DGS5': 'YIELD_5Y', 'DGS10': 'YIELD_10Y', 'DGS2': 'YIELD_2Y',
            'T5YIE': 'BREAKEVEN_5Y', 'T10YIE': 'BREAKEVEN_10Y',
            'EFFR': 'FED_FUNDS', 'DFEDTARU': 'FF_UPPER',
        }
        
        df['Date'] = pd.to_datetime(df['Date'])
        fred_end_date = Config.END_DATE if Config.END_DATE else datetime.now().strftime('%Y-%m-%d')
        
        for fred_id, col_name in series_to_fetch.items():
            try:
                data = web.DataReader(fred_id, 'fred', Config.START_DATE, fred_end_date)
                data = data.reset_index()
                data.columns = ['Date', col_name]
                data['Date'] = pd.to_datetime(data['Date'])
                df = pd.merge_asof(df.sort_values('Date'), data.sort_values('Date'), 
                                  on='Date', direction='backward')
                df[col_name] = df[col_name].fillna(method='ffill')
                features.append(col_name)
            except:
                pass
        
        if 'YIELD_5Y' in df.columns and 'BREAKEVEN_5Y' in df.columns:
            df['REAL_YIELD_5Y'] = df['YIELD_5Y'] - df['BREAKEVEN_5Y']
            features.append('REAL_YIELD_5Y')
            
            df['REAL_YIELD_5Y_ZSCORE'] = (df['REAL_YIELD_5Y'] - 
                df['REAL_YIELD_5Y'].rolling(252).mean()) / (df['REAL_YIELD_5Y'].rolling(252).std() + 0.001)
            features.append('REAL_YIELD_5Y_ZSCORE')
            
            df['REAL_YIELD_5Y_ROC_20'] = df['REAL_YIELD_5Y'].diff(20)
            df['REAL_YIELD_5Y_ROC_60'] = df['REAL_YIELD_5Y'].diff(60)
            features.extend(['REAL_YIELD_5Y_ROC_20', 'REAL_YIELD_5Y_ROC_60'])
            
            df['NEGATIVE_REAL_RATE'] = (df['REAL_YIELD_5Y'] < 0).astype(int)
            features.append('NEGATIVE_REAL_RATE')
        
        if 'YIELD_10Y' in df.columns and 'YIELD_2Y' in df.columns:
            df['YIELD_CURVE_SLOPE']      = df['YIELD_10Y'] - df['YIELD_2Y']
            df['YIELD_CURVE_INVERTED']   = (df['YIELD_CURVE_SLOPE'] < 0).astype(int)
            # Rate of change: positive = curve re-steepening (leading indicator for
            # industrial metals; steeply inverted curve historically precedes silver
            # underperformance, re-steepening precedes next rally)
            df['YIELD_CURVE_STEEPENING'] = df['YIELD_CURVE_SLOPE'].diff(20)
            # Z-score of slope vs 1-year rolling window (captures regime extremes)
            _yc_mean = df['YIELD_CURVE_SLOPE'].rolling(252).mean()
            _yc_std  = df['YIELD_CURVE_SLOPE'].rolling(252).std()
            df['YIELD_CURVE_ZSCORE']     = (df['YIELD_CURVE_SLOPE'] - _yc_mean) / (_yc_std + 0.001)
            features.extend(['YIELD_CURVE_SLOPE', 'YIELD_CURVE_INVERTED',
                             'YIELD_CURVE_STEEPENING', 'YIELD_CURVE_ZSCORE'])
        
        if 'FED_FUNDS' in df.columns:
            df['FED_FUNDS_ROC_60'] = df['FED_FUNDS'].diff(60)
            df['FED_HIKING'] = (df['FED_FUNDS_ROC_60'] > 0.25).astype(int)
            df['FED_CUTTING'] = (df['FED_FUNDS_ROC_60'] < -0.25).astype(int)
            features.extend(['FED_FUNDS_ROC_60', 'FED_HIKING', 'FED_CUTTING'])
        
        print(f"  ✓ Added {len(features)} real rate features")
        
    except Exception as e:
        print(f"  ✗ Error: {e}")
    
    return df, features


def add_dollar_features(df):
    """Add US Dollar features"""
    print("\n" + "-"*70)
    print("[QUANT] US Dollar Features...")
    
    features = []
    
    if 'DX-Y.NYB_Close' in df.columns:
        df['DXY'] = df['DX-Y.NYB_Close']
        features.append('DXY')
        
        for period in [5, 10, 20, 60]:
            df[f'DXY_ROC_{period}'] = df['DXY'].pct_change(period) * 100
            features.append(f'DXY_ROC_{period}')
        
        df['DXY_SMA_50'] = df['DXY'].rolling(50).mean()
        df['DXY_SMA_200'] = df['DXY'].rolling(200).mean()
        df['DXY_ABOVE_50'] = (df['DXY'] > df['DXY_SMA_50']).astype(int)
        df['DXY_ABOVE_200'] = (df['DXY'] > df['DXY_SMA_200']).astype(int)
        df['DXY_TREND'] = df['DXY_ABOVE_50'] + df['DXY_ABOVE_200']
        features.extend(['DXY_SMA_50', 'DXY_SMA_200', 'DXY_ABOVE_50', 'DXY_ABOVE_200', 'DXY_TREND'])
        
        df['DXY_VOLATILITY_20'] = df['DXY'].pct_change().rolling(20).std() * np.sqrt(252) * 100
        features.append('DXY_VOLATILITY_20')
        
        df['DXY_ZSCORE_60'] = (df['DXY'] - df['DXY'].rolling(60).mean()) / (df['DXY'].rolling(60).std() + 0.001)
        features.append('DXY_ZSCORE_60')
        
        print(f"  ✓ Added {len(features)} dollar features")
    
    return df, features


def add_enhanced_gold_silver_ratio(df):
    """Add Gold/Silver Ratio features"""
    print("\n" + "-"*70)
    print("[QUANT] Enhanced Gold/Silver Ratio Features...")
    
    features = []
    
    if 'GC=F_Close' in df.columns and 'SI=F_Close' in df.columns:
        df['GS_RATIO'] = df['GC=F_Close'] / df['SI=F_Close']
        features.append('GS_RATIO')
        
        df['GS_RATIO_ZSCORE'] = (df['GS_RATIO'] - df['GS_RATIO'].rolling(252).mean()) / \
                                (df['GS_RATIO'].rolling(252).std() + 0.001)
        features.append('GS_RATIO_ZSCORE')
        
        df['GS_RATIO_PERCENTILE'] = df['GS_RATIO'].rolling(252).rank(pct=True)
        features.append('GS_RATIO_PERCENTILE')
        
        df['GS_RATIO_ROC_5'] = df['GS_RATIO'].pct_change(5) * 100
        df['GS_RATIO_ROC_20'] = df['GS_RATIO'].pct_change(20) * 100
        df['GS_RATIO_ROC_60'] = df['GS_RATIO'].pct_change(60) * 100
        features.extend(['GS_RATIO_ROC_5', 'GS_RATIO_ROC_20', 'GS_RATIO_ROC_60'])
        
        df['GS_RATIO_EXTREME_HIGH'] = (df['GS_RATIO_ZSCORE'] > 1.5).astype(int)
        df['GS_RATIO_EXTREME_LOW'] = (df['GS_RATIO_ZSCORE'] < -1.5).astype(int)
        df['GS_RATIO_VERY_HIGH'] = (df['GS_RATIO'] > 80).astype(int)
        df['GS_RATIO_VERY_LOW'] = (df['GS_RATIO'] < 60).astype(int)
        features.extend(['GS_RATIO_EXTREME_HIGH', 'GS_RATIO_EXTREME_LOW', 
                        'GS_RATIO_VERY_HIGH', 'GS_RATIO_VERY_LOW'])
        
        df['GS_RATIO_SMA_50'] = df['GS_RATIO'].rolling(50).mean()
        df['GS_RATIO_ABOVE_SMA'] = (df['GS_RATIO'] > df['GS_RATIO_SMA_50']).astype(int)
        features.extend(['GS_RATIO_SMA_50', 'GS_RATIO_ABOVE_SMA'])
        
        df['GOLD_ROC_20'] = df['GC=F_Close'].pct_change(20) * 100
        df['GOLD_ROC_60'] = df['GC=F_Close'].pct_change(60) * 100
        features.extend(['GOLD_ROC_20', 'GOLD_ROC_60'])
        
        df['SILVER_VS_GOLD_20'] = (df['SI=F_Close'].pct_change(20) - df['GC=F_Close'].pct_change(20)) * 100
        df['SILVER_VS_GOLD_60'] = (df['SI=F_Close'].pct_change(60) - df['GC=F_Close'].pct_change(60)) * 100
        features.extend(['SILVER_VS_GOLD_20', 'SILVER_VS_GOLD_60'])
        
        print(f"  ✓ Added {len(features)} G/S ratio features")
    
    return df, features


def add_cross_asset_features(df):
    """Add Cross-Asset Features"""
    print("\n" + "-"*70)
    print("[QUANT] Cross-Asset Features...")
    
    features = []
    
    if 'HG=F_Close' in df.columns:
        df['COPPER_ROC_5'] = df['HG=F_Close'].pct_change(5) * 100
        df['COPPER_ROC_20'] = df['HG=F_Close'].pct_change(20) * 100
        df['COPPER_ROC_60'] = df['HG=F_Close'].pct_change(60) * 100
        features.extend(['COPPER_ROC_5', 'COPPER_ROC_20', 'COPPER_ROC_60'])
        
        df['COPPER_CORR_20'] = df['SI=F_Close'].rolling(20).corr(df['HG=F_Close'])
        df['COPPER_CORR_60'] = df['SI=F_Close'].rolling(60).corr(df['HG=F_Close'])
        features.extend(['COPPER_CORR_20', 'COPPER_CORR_60'])
        
        df['COPPER_CORR_BREAKDOWN'] = (df['COPPER_CORR_60'] < 0.3).astype(int)
        features.append('COPPER_CORR_BREAKDOWN')
        
        if 'GC=F_Close' in df.columns:
            df['COPPER_GOLD_RATIO'] = df['HG=F_Close'] / df['GC=F_Close'] * 100
            df['COPPER_GOLD_ROC_20'] = df['COPPER_GOLD_RATIO'].pct_change(20) * 100
            features.extend(['COPPER_GOLD_RATIO', 'COPPER_GOLD_ROC_20'])
    
    # Metal Rotation Score — silver leads when it outperforms both gold (monetary)
    # AND copper (industrial) simultaneously.
    # 0 = silver lagging all metals (bearish), 1 = mixed, 2 = leading (strongly bullish)
    if 'ROC_20' in df.columns and 'GOLD_ROC_20' in df.columns and 'COPPER_ROC_20' in df.columns:
        df['SI_OUTPERFORMS_GOLD']   = (df['ROC_20'] > df['GOLD_ROC_20']).astype(int)
        df['SI_OUTPERFORMS_COPPER'] = (df['ROC_20'] > df['COPPER_ROC_20']).astype(int)
        df['METAL_ROTATION_SCORE']  = df['SI_OUTPERFORMS_GOLD'] + df['SI_OUTPERFORMS_COPPER']
        features.extend(['SI_OUTPERFORMS_GOLD', 'SI_OUTPERFORMS_COPPER', 'METAL_ROTATION_SCORE'])
    
    if '^TNX_Close' in df.columns:
        df['TNX_ROC_5'] = df['^TNX_Close'].diff(5)
        df['TNX_ROC_20'] = df['^TNX_Close'].diff(20)
        features.extend(['TNX_LEVEL', 'TNX_ROC_5', 'TNX_ROC_20'])
        
        df['YIELD_SPIKE'] = (df['TNX_ROC_5'] > 0.1).astype(int)
        features.append('YIELD_SPIKE')
        
        df['SILVER_YIELD_CORR'] = df['SI=F_Close'].rolling(60).corr(df['^TNX_Close'])
        features.append('SILVER_YIELD_CORR')
    
    if 'CL=F_Close' in df.columns:
        df['OIL_ROC_5'] = df['CL=F_Close'].pct_change(5) * 100
        df['OIL_ROC_20'] = df['CL=F_Close'].pct_change(20) * 100
        features.extend(['OIL_ROC_5', 'OIL_ROC_20'])
    
    if '^GSPC_Close' in df.columns:
        df['SPX_ROC_5'] = df['^GSPC_Close'].pct_change(5) * 100
        df['SPX_ROC_20'] = df['^GSPC_Close'].pct_change(20) * 100
        features.extend(['SPX_ROC_5', 'SPX_ROC_20'])
        
        spx_sma50 = df['^GSPC_Close'].rolling(50).mean()
        spx_sma200 = df['^GSPC_Close'].rolling(200).mean()
        df['SPX_ABOVE_50'] = (df['^GSPC_Close'] > spx_sma50).astype(int)
        df['SPX_ABOVE_200'] = (df['^GSPC_Close'] > spx_sma200).astype(int)
        features.extend(['SPX_ABOVE_50', 'SPX_ABOVE_200'])
    
    if '^VIX_Close' in df.columns:
        df['VIX'] = df['^VIX_Close']
        df['VIX_SMA_20'] = df['VIX'].rolling(20).mean()
        df['VIX_PERCENTILE'] = df['VIX'].rolling(252).rank(pct=True)
        features.extend(['VIX', 'VIX_SMA_20', 'VIX_PERCENTILE'])
        
        df['VIX_LOW'] = (df['VIX'] < 15).astype(int)
        df['VIX_HIGH'] = (df['VIX'] > 25).astype(int)
        df['VIX_EXTREME'] = (df['VIX'] > 35).astype(int)
        features.extend(['VIX_LOW', 'VIX_HIGH', 'VIX_EXTREME'])
    
    if '^DJI_Close' in df.columns and 'GC=F_Close' in df.columns:
        df['DOW_GOLD_RATIO'] = df['^DJI_Close'] / df['GC=F_Close']
        df['DOW_GOLD_ZSCORE'] = (df['DOW_GOLD_RATIO'] - df['DOW_GOLD_RATIO'].rolling(252).mean()) / \
                                (df['DOW_GOLD_RATIO'].rolling(252).std() + 0.001)
        features.extend(['DOW_GOLD_RATIO', 'DOW_GOLD_ZSCORE'])
        
        df['DOW_GOLD_EXTREME_LOW'] = (df['DOW_GOLD_ZSCORE'] < -1.5).astype(int)
        features.append('DOW_GOLD_EXTREME_LOW')
    
    print(f"  ✓ Added {len(features)} cross-asset features")
    
    return df, features


def add_advanced_volatility_features(df):
    """
    Industry-grade volatility feature set for silver futures.

    Hierarchy (simple → complex):
      1. Close-to-close realized vol (baseline, multiple windows)
      2. Yang-Zhang realized vol estimator (OHLC-based, handles gaps/drift)
      3. HAR-RV decomposition + rolling OLS forecast (multi-horizon structure)
      4. EGARCH conditional vol, asymmetry γ, persistence α+β
      5. GARCH-MIDAS long/short-run vol decomposition via macro linkage
      6. Variance Risk Premium in variance terms (not vol terms)
      7. Idiosyncratic silver vol (residual from VXSLV ~ GVZ regression)
      8. Approximate Entropy (return series predictability)
      9. Vol forecast combination via Bates-Granger inverse-MSE weights
     10. VXSLV + GVZ implied vol features, Vol-of-Vol
     11. Return skewness, kurtosis, jump indicators
    """
    from arch import arch_model
    from sklearn.linear_model import LinearRegression

    print("\n" + "-"*70)
    print("[QUANT] Industry-Grade Volatility Features...")

    features = []

    # ── 1. Close-to-close realized vol — baseline ─────────────────────────────
    daily_ret = df['SI=F_Close'].pct_change()
    for window in [5, 10, 20, 60]:
        df[f'REALIZED_VOL_{window}'] = (
            daily_ret.rolling(window).std() * np.sqrt(252) * 100)
        features.append(f'REALIZED_VOL_{window}')

    df['VOL_RATIO_5_20']  = df['REALIZED_VOL_5']  / (df['REALIZED_VOL_20'] + 0.001)
    df['VOL_RATIO_10_60'] = df['REALIZED_VOL_10'] / (df['REALIZED_VOL_60'] + 0.001)
    features.extend(['VOL_RATIO_5_20', 'VOL_RATIO_10_60'])

    df['VOL_PERCENTILE'] = df['REALIZED_VOL_20'].rolling(252).rank(pct=True)
    features.append('VOL_PERCENTILE')

    df['HIGH_VOL_REGIME'] = (df['VOL_PERCENTILE'] > 0.75).astype(int)
    df['LOW_VOL_REGIME']  = (df['VOL_PERCENTILE'] < 0.25).astype(int)
    features.extend(['HIGH_VOL_REGIME', 'LOW_VOL_REGIME'])

    # ── 2. Yang-Zhang Realized Volatility Estimator ───────────────────────────
    # Parkinson uses only H-L and assumes no drift and no overnight gaps.
    # Silver futures have material overnight gaps (COMEX open, Asia session,
    # macro events).  Yang-Zhang combines three components:
    #   σ²_overnight  = var of log(Open_t / Close_{t-1})   — gap risk
    #   σ²_RS         = Rogers-Satchell drift-adjusted intraday estimator
    #                   = mean[log(H/C)·log(H/O) + log(L/C)·log(L/O)]
    #   σ²_OC         = var of log(Close_t / Open_t)       — day session risk
    #   k             = 0.34 / (1.34 + (n+1)/(n-1))        — optimal weighting
    #
    # Roughly 14× more statistically efficient than close-to-close.
    # Kept alongside Parkinson so the model can learn the differential.
    if all(c in df.columns for c in
           ['SI=F_Open', 'SI=F_High', 'SI=F_Low', 'SI=F_Close']):

        _o = df['SI=F_Open']
        _h = df['SI=F_High']
        _l = df['SI=F_Low']
        _c = df['SI=F_Close']
        _cp = _c.shift(1)                             # previous close

        log_oc  = np.log(_c / _o)                     # open-to-close return
        log_on  = np.log(_o / _cp)                    # overnight gap
        log_hc  = np.log(_h / _c)                     # intraday components
        log_ho  = np.log(_h / _o)
        log_lc  = np.log(_l / _c)
        log_lo  = np.log(_l / _o)

        n = 20
        k = 0.34 / (1.34 + (n + 1) / (n - 1))

        var_on = log_on.rolling(n).var()
        var_oc = log_oc.rolling(n).var()
        var_rs = (log_hc * log_ho + log_lc * log_lo).rolling(n).mean()

        df['YANGZHANG_VOL'] = (
            np.sqrt(np.maximum(var_on + k * var_oc + (1 - k) * var_rs, 0))
            * np.sqrt(252) * 100
        )
        # Parkinson for comparison / differential
        df['PARKINSON_VOL'] = np.sqrt(
            (1 / (4 * np.log(2))) *
            (np.log(_h / _l) ** 2).rolling(n).mean()
        ) * np.sqrt(252) * 100

        # Differential: YZ captures more risk than Parkinson when overnight
        # gaps are active (positive = gap risk is elevated above intraday)
        df['YZ_PARK_DIFF'] = df['YANGZHANG_VOL'] - df['PARKINSON_VOL']
        features.extend(['YANGZHANG_VOL', 'PARKINSON_VOL', 'YZ_PARK_DIFF'])
        print("  ✓ Yang-Zhang estimator computed")

    # ── 3. HAR-RV: Heterogeneous Autoregressive Realized Volatility ───────────
    # Different market participants operate at different horizons simultaneously:
    # day traders (1d), swing traders (1w), fund managers (1m).  HAR-RV
    # decomposes realized vol into these three components — the coefficients
    # tell you which horizon is currently driving volatility dynamics.
    #
    # HAR-RV forecast uses a rolling 252-day OLS window to predict next-day RV
    # from today's daily, weekly, and monthly RV components.  This is the
    # standard production forecasting model at commodity vol desks.
    #
    # Components (all in annualized % vol space, squared for additivity):
    #   HAR_RV_D  : daily RV component (yesterday)
    #   HAR_RV_W  : weekly RV component (5-day average)
    #   HAR_RV_M  : monthly RV component (22-day average)
    #   HAR_COEF_D/W/M : rolling OLS coefficients (which horizon dominates)
    #   HAR_RV_FORECAST : next-day RV forecast from rolling OLS
    rv1 = (daily_ret ** 2).rolling(1).sum()     # 1-day realized variance proxy
    rv1_ann = np.sqrt(rv1 * 252) * 100          # annualised vol equivalent

    df['HAR_RV_D'] = rv1_ann.shift(1)
    df['HAR_RV_W'] = rv1_ann.rolling(5).mean().shift(1)
    df['HAR_RV_M'] = rv1_ann.rolling(22).mean().shift(1)
    features.extend(['HAR_RV_D', 'HAR_RV_W', 'HAR_RV_M'])

    # Rolling OLS forecast — fit on past 252 days, predict next day
    har_forecast  = np.full(len(df), np.nan)
    har_coef_d    = np.full(len(df), np.nan)
    har_coef_w    = np.full(len(df), np.nan)
    har_coef_m    = np.full(len(df), np.nan)
    har_target    = rv1_ann.values

    har_d = df['HAR_RV_D'].values
    har_w = df['HAR_RV_W'].values
    har_m = df['HAR_RV_M'].values

    for i in range(252, len(df)):
        _X = np.column_stack([
            har_d[i-252:i],
            har_w[i-252:i],
            har_m[i-252:i]
        ])
        _y = har_target[i-251:i+1]
        mask = ~(np.isnan(_X).any(axis=1) | np.isnan(_y))
        if mask.sum() < 60:
            continue
        try:
            lr = LinearRegression().fit(_X[mask], _y[mask])
            x_pred = np.array([[har_d[i], har_w[i], har_m[i]]])
            if not np.isnan(x_pred).any():
                har_forecast[i] = float(lr.predict(x_pred)[0])
                har_coef_d[i]   = lr.coef_[0]
                har_coef_w[i]   = lr.coef_[1]
                har_coef_m[i]   = lr.coef_[2]
        except Exception:
            pass

    df['HAR_RV_FORECAST'] = har_forecast
    df['HAR_COEF_D']      = har_coef_d    # weight on daily component
    df['HAR_COEF_W']      = har_coef_w    # weight on weekly component
    df['HAR_COEF_M']      = har_coef_m    # weight on monthly component
    features.extend(['HAR_RV_FORECAST', 'HAR_COEF_D', 'HAR_COEF_W', 'HAR_COEF_M'])
    print("  ✓ HAR-RV decomposition + rolling OLS forecast computed")

    # ── 4. EGARCH Conditional Volatility, Asymmetry, Persistence ─────────────
    # GARCH(1,1) treats up-moves and down-moves identically — σ responds only
    # to |ε|.  Silver has a pronounced leverage effect: down-moves expand vol
    # harder than equivalent up-moves (short squeezes are the exception, not
    # the rule in the vol response).
    #
    # EGARCH(1,1,1) models log(σ²) to ensure positivity without constraints and
    # adds an asymmetry term:
    #   log(σ²_t) = ω + α·z_{t-1} + γ·|z_{t-1}| + β·log(σ²_{t-1})
    #   γ < 0 : negative returns → larger vol increase (leverage effect present)
    #
    # Features extracted:
    #   EGARCH_COND_VOL     : conditional vol (annualised %) — CAUSAL, see below
    #   EGARCH_ASYMMETRY    : γ parameter (time-varying, per refit block)
    #   EGARCH_PERSISTENCE  : β parameter (EGARCH persistence is β alone —
    #                         α+β is a GARCH concept and is structurally wrong
    #                         for the log-variance recursion)
    #   EGARCH_COND_PCTILE  : percentile of cond vol over rolling 252d history
    #
    # ── LOOK-AHEAD-FREE ESTIMATION (fixes full-sample leak) ──────────────────
    # The old implementation fit EGARCH ONCE on the last 2000 obs of ALL
    # history, then broadcast its conditional vol + parameters into every
    # walk-forward window — 2019 bars received parameters estimated with 2026
    # data.  Same class of leak fixed in the NW envelope.
    #
    # New scheme (mirrors the rolling-OLS pattern HAR-RV already uses):
    #   • Refit every EGARCH_REFIT_EVERY=63 bars (~quarterly) on a trailing
    #     window of up to EGARCH_FIT_LOOKBACK=1500 obs ending the bar BEFORE
    #     the block starts.
    #   • Between refits, parameters are FROZEN via arch's .fix() and run as a
    #     pure forward filter over past returns.  The GARCH filtration is
    #     causal by construction — σ²_t depends only on ε_{t-1}, σ²_{t-1} — so
    #     every bar's conditional vol uses only past data AND past-estimated
    #     parameters.
    #   • γ and β become step-wise time-varying series (constant within each
    #     refit block), matching what a live system would have known.
    #   • Warmup (first EGARCH_MIN_FIT_OBS bars) falls back to REALIZED_VOL_20
    #     — no bfill of future values.
    # Cost: ~30 refits on full history, roughly 1–3 minutes.
    EGARCH_REFIT_EVERY   = 63     # ~1 quarter of trading days
    EGARCH_FIT_LOOKBACK  = 1500   # trailing obs per refit
    EGARCH_MIN_FIT_OBS   = 250    # minimum history before first fit
    try:
        _returns_pct = daily_ret * 100
        _valid = _returns_pct.dropna()

        egarch_cond_vol = pd.Series(np.nan, index=df.index)
        egarch_gamma_s  = pd.Series(np.nan, index=df.index)
        egarch_beta_s   = pd.Series(np.nan, index=df.index)

        _frozen_params = None
        _n_refits = 0

        if len(_valid) > EGARCH_MIN_FIT_OBS:
            for _bs in range(EGARCH_MIN_FIT_OBS, len(_valid), EGARCH_REFIT_EVERY):
                _be = min(_bs + EGARCH_REFIT_EVERY, len(_valid))  # exclusive

                # ── Refit on trailing data STRICTLY BEFORE the block ─────────
                _train = _valid.iloc[max(0, _bs - EGARCH_FIT_LOOKBACK):_bs]
                try:
                    _am_fit = arch_model(_train, vol='EGARCH', p=1, q=1, o=1,
                                         dist='skewt', rescale=False)
                    _res_fit = _am_fit.fit(disp='off', show_warning=False)
                    _frozen_params = _res_fit.params
                    _n_refits += 1
                except Exception:
                    # Keep the previous block's frozen params; if none exist
                    # yet, this block stays NaN and gets the RV20 fallback.
                    pass

                if _frozen_params is None:
                    continue

                # ── Forward filter with FROZEN parameters ────────────────────
                # Run the recursion over trailing history + this block so
                # σ² state is warmed up; cond vol at bar t only uses returns
                # up to t (causal filtration).  .fix() evaluates the model at
                # the given params without re-estimating.
                try:
                    _filt_start = max(0, _bs - EGARCH_FIT_LOOKBACK)
                    _filt_data  = _valid.iloc[_filt_start:_be]
                    _am_f  = arch_model(_filt_data, vol='EGARCH', p=1, q=1, o=1,
                                        dist='skewt', rescale=False)
                    _res_f = _am_f.fix(_frozen_params)
                    _cv    = _res_f.conditional_volatility * np.sqrt(252)

                    _block_idx = _valid.index[_bs:_be]
                    egarch_cond_vol.loc[_block_idx] = (
                        _cv.iloc[_bs - _filt_start:_be - _filt_start].values)

                    _g = float(_frozen_params.get('gamma[1]', np.nan))
                    _b = float(_frozen_params.get('beta[1]',  np.nan))
                    egarch_gamma_s.loc[_block_idx] = _g
                    egarch_beta_s.loc[_block_idx]  = _b
                except Exception:
                    continue

        # Warmup / failed-block fallback: realized vol — never bfill
        df['EGARCH_COND_VOL'] = egarch_cond_vol.fillna(df['REALIZED_VOL_20'])
        df['EGARCH_COND_PCTILE'] = (
            df['EGARCH_COND_VOL'].rolling(252, min_periods=60).rank(pct=True))

        # Time-varying parameter series; neutral priors in warmup (γ=0 no
        # asymmetry, β=0.9 typical persistence) — again, no bfill
        df['EGARCH_ASYMMETRY']   = egarch_gamma_s.fillna(0.0)
        df['EGARCH_PERSISTENCE'] = egarch_beta_s.fillna(0.9)

        features.extend(['EGARCH_COND_VOL', 'EGARCH_COND_PCTILE',
                          'EGARCH_ASYMMETRY', 'EGARCH_PERSISTENCE'])
        _last_g = df['EGARCH_ASYMMETRY'].iloc[-1]
        _last_b = df['EGARCH_PERSISTENCE'].iloc[-1]
        print(f"  ✓ EGARCH rolling fit | {_n_refits} quarterly refits | "
              f"latest γ={_last_g:.3f} β={_last_b:.3f}")
    except Exception as e:
        df['EGARCH_COND_VOL']    = df['REALIZED_VOL_20']   # graceful fallback
        df['EGARCH_COND_PCTILE'] = df['VOL_PERCENTILE']
        df['EGARCH_ASYMMETRY']   = 0.0
        df['EGARCH_PERSISTENCE'] = 0.9
        features.extend(['EGARCH_COND_VOL', 'EGARCH_COND_PCTILE',
                          'EGARCH_ASYMMETRY', 'EGARCH_PERSISTENCE'])
        print(f"  ⚠ EGARCH failed ({e}) — using RV fallback")

    # ── 5. GARCH-MIDAS Long/Short-Run Vol Decomposition ──────────────────────
    # Standard GARCH operates entirely at the daily frequency.  GARCH-MIDAS
    # separates vol into:
    #   Short-run (daily GARCH residual): pure price noise
    #   Long-run (macro-driven): driven by DXY trend, real rates, COT positioning
    #
    # Implemented here as a simplified two-component decomposition:
    #   MIDAS_LONG_RUN  : slow-moving macro-driven vol (EWM of macro vol drivers)
    #   MIDAS_SHORT_RUN : EGARCH cond vol / MIDAS long run (ratio of short to long)
    #   MIDAS_RATIO > 1 : short-run noise dominating macro signal
    #   MIDAS_RATIO < 1 : macro regime driving vol more than daily noise
    #
    # Macro inputs: DXY vol (dollar risk), real yield vol (rate shock risk),
    # COT commercial net (positioning stress).  All already in the dataframe.
    midas_inputs = []
    if 'DXY_VOLATILITY_20' in df.columns:
        midas_inputs.append(df['DXY_VOLATILITY_20'].fillna(method='ffill').fillna(0))
    if 'REAL_YIELD_5Y' in df.columns:
        midas_inputs.append(
            df['REAL_YIELD_5Y'].pct_change().abs().rolling(20).mean().fillna(0) * 100)
    if 'COT_comm_net' in df.columns:
        midas_inputs.append(
            df['COT_comm_net'].pct_change().abs().rolling(20).mean().fillna(0) * 100)

    if midas_inputs:
        macro_vol_index = pd.concat(midas_inputs, axis=1).mean(axis=1)
        # Long-run component: EWM with span=63 (~1 quarter) smooths macro signal
        df['MIDAS_LONG_RUN']  = macro_vol_index.ewm(span=63, min_periods=20).mean()
        df['MIDAS_SHORT_RUN'] = df['EGARCH_COND_VOL']
        df['MIDAS_RATIO']     = (
            df['MIDAS_SHORT_RUN'] / (df['MIDAS_LONG_RUN'] + 0.001))
        # Percentile of ratio — is short-run noise historically elevated vs macro?
        df['MIDAS_RATIO_PCTILE'] = (
            df['MIDAS_RATIO'].rolling(252, min_periods=60).rank(pct=True))
        features.extend(['MIDAS_LONG_RUN', 'MIDAS_SHORT_RUN',
                          'MIDAS_RATIO', 'MIDAS_RATIO_PCTILE'])
        print("  ✓ GARCH-MIDAS long/short-run decomposition computed")

    # ── 6. Variance Risk Premium (VRP) in Variance Terms ─────────────────────
    # The VRP is the gap between implied variance and realized variance.
    # Must be computed in VARIANCE (σ²) not VOL (σ) because variance is additive
    # across time — vol is not.  A 30% implied vol does not mean 15% vol per
    # half-year; 30%² variance does scale linearly.
    #
    # VRP > 0 : market paying fear premium above realized risk (normal)
    # VRP < 0 : realized risk has overshot implied (disorderly move in progress)
    # VRP_ZSCORE: rolling 252d standardized VRP — how extreme is the current gap
    # VRP_PCTILE: same in percentile form — directly usable by tree models
    if 'VXSLV' in df.columns and df['VXSLV'].notna().sum() > 30:
        iv_var  = (df['VXSLV']         / 100) ** 2   # implied variance (decimal annual)
        rv_var  = (df['REALIZED_VOL_20'] / 100) ** 2  # realized variance (decimal annual)
        df['VRP']        = (iv_var - rv_var) * 10000  # in basis points of variance
        df['VRP_ZSCORE'] = (
            (df['VRP'] - df['VRP'].rolling(252, min_periods=60).mean()) /
            (df['VRP'].rolling(252, min_periods=60).std() + 1e-8))
        df['VRP_PCTILE'] = df['VRP'].rolling(252, min_periods=60).rank(pct=True)
        # Negative VRP flag — realized vol exceeding what options priced in
        df['VRP_NEGATIVE'] = (df['VRP'] < 0).astype(float)
        df.loc[df['VXSLV'].isna(), ['VRP','VRP_ZSCORE','VRP_PCTILE','VRP_NEGATIVE']] = np.nan
        features.extend(['VRP', 'VRP_ZSCORE', 'VRP_PCTILE', 'VRP_NEGATIVE'])
        print("  ✓ Variance Risk Premium (VRP) computed in variance space")

    # ── 7. Idiosyncratic Silver Volatility ────────────────────────────────────
    # A large portion of VXSLV on any given day is simply GVZ in disguise —
    # macro precious metals fear.  Running a rolling 60-day OLS of VXSLV on GVZ
    # decomposes silver IV into:
    #   IDIO_VOL   : silver-specific vol (residual from VXSLV ~ GVZ)
    #   BETA_TO_GVZ: how sensitive silver IV is to gold IV right now
    #
    # IDIO_VOL elevated → something silver-specific happening (COMEX positioning,
    #   industrial demand shock, short squeeze building, SLV flow anomaly)
    # IDIO_VOL near zero → silver just tracking gold; lower edge for silver model
    # BETA_TO_GVZ > 1.5 → silver amplifying gold fear (high-beta regime)
    if ('VXSLV' in df.columns and 'GVZ' in df.columns and
            df['VXSLV'].notna().sum() > 30 and df['GVZ'].notna().sum() > 30):

        idio_vol    = np.full(len(df), np.nan)
        beta_to_gvz = np.full(len(df), np.nan)
        vxslv_arr   = df['VXSLV'].values
        gvz_arr     = df['GVZ'].values

        for i in range(60, len(df)):
            _v = vxslv_arr[i-60:i]
            _g = gvz_arr[i-60:i]
            mask = ~(np.isnan(_v) | np.isnan(_g))
            if mask.sum() < 30:
                continue
            try:
                lr = LinearRegression().fit(_g[mask].reshape(-1, 1), _v[mask])
                beta_to_gvz[i] = lr.coef_[0]
                if not np.isnan(gvz_arr[i]) and not np.isnan(vxslv_arr[i]):
                    fitted = lr.predict([[gvz_arr[i]]])[0]
                    idio_vol[i] = vxslv_arr[i] - fitted
            except Exception:
                pass

        df['IDIO_VOL']        = idio_vol
        df['BETA_TO_GVZ']     = beta_to_gvz
        df['IDIO_VOL_PCTILE'] = (
            pd.Series(idio_vol, index=df.index)
            .rolling(252, min_periods=60).rank(pct=True))
        # Elevated idiosyncratic vol flag (top quartile)
        df['IDIO_VOL_HIGH']   = (df['IDIO_VOL_PCTILE'] > 0.75).astype(float)
        df.loc[df['VXSLV'].isna() | df['GVZ'].isna(),
               ['IDIO_VOL','BETA_TO_GVZ','IDIO_VOL_PCTILE','IDIO_VOL_HIGH']] = np.nan
        # CRITICAL: null the residual features across the PROXIED gap.  There,
        # VXSLV is itself a linear function of GVZ, so the residual would be a
        # fabricated ~0 ("silver perfectly tracks gold") for the whole gap —
        # a fake signal.  The level features (VXSLV_LEVEL/PCTILE/etc.) keep the
        # proxy since a GVZ-based level estimate is legitimately useful; the
        # decomposition features do not, because there is no real silver-
        # specific residual to recover from a GVZ-derived value.
        if 'VXSLV_IS_PROXY' in df.columns:
            df.loc[df['VXSLV_IS_PROXY'] == 1,
                   ['IDIO_VOL','BETA_TO_GVZ','IDIO_VOL_PCTILE','IDIO_VOL_HIGH']] = np.nan
        features.extend(['IDIO_VOL', 'BETA_TO_GVZ',
                          'IDIO_VOL_PCTILE', 'IDIO_VOL_HIGH'])
        print("  ✓ Idiosyncratic silver volatility decomposed from GVZ")

    # ── 8. Approximate Entropy (ApEn) — Return Series Predictability ─────────
    # ApEn measures the regularity/predictability of the return series.
    # Low ApEn  → returns are structured and autocorrelated; ML signals carry
    #             more weight because the environment is learnable.
    # High ApEn → returns approaching white noise; even a well-trained model
    #             has low expected edge regardless of its probability output.
    #
    # Implemented manually (no antropy dependency) using m=2, r=0.2·σ.
    # Computed on rolling 60-day windows — computationally intensive but
    # produces a regime-quality signal orthogonal to all other vol features.
    def _apen(u, m=2, r_mult=0.2):
        """Approximate Entropy on array u (Richman & Moorman 2000)."""
        n = len(u)
        if n < m + 2:
            return np.nan
        r = r_mult * np.std(u, ddof=1)
        if r == 0:
            return 0.0
        def _phi(m_):
            x = np.array([u[i:i+m_] for i in range(n - m_ + 1)])
            C = np.sum(
                np.max(np.abs(x[:, None] - x[None, :]), axis=2) <= r,
                axis=0
            ) / (n - m_ + 1)
            return np.sum(np.log(C + 1e-10)) / (n - m_ + 1)
        return abs(_phi(m) - _phi(m + 1))

    ret_arr   = daily_ret.values
    apen_vals = np.full(len(df), np.nan)
    _win      = 60
    for i in range(_win, len(df)):
        seg = ret_arr[i-_win:i]
        if np.isnan(seg).sum() > 5:
            continue
        seg = seg[~np.isnan(seg)]
        apen_vals[i] = _apen(seg)

    df['APEN_60']       = apen_vals
    df['APEN_PCTILE']   = (
        pd.Series(apen_vals, index=df.index)
        .rolling(252, min_periods=60).rank(pct=True))
    # Low entropy flag — environment is structured/learnable
    df['LOW_ENTROPY']   = (df['APEN_PCTILE'] < 0.25).astype(float)
    features.extend(['APEN_60', 'APEN_PCTILE', 'LOW_ENTROPY'])
    print("  ✓ Approximate Entropy (ApEn) computed on rolling 60-day windows")

    # ── 9. Vol Forecast Combination (Bates-Granger Optimal Weights) ───────────
    # No single vol model dominates across all regimes:
    #   HAR-RV best in trending vol environments
    #   EGARCH best around spikes and mean-reversion
    #   Yang-Zhang best when overnight gaps are material
    #
    # Bates-Granger combination uses inverse-MSE weights derived from each
    # model's historical forecast error against realized vol.  The combined
    # forecast is strictly at least as good as the best individual model in
    # expectation, and the individual weights are themselves features —
    # when EGARCH weight is high, the market is in a vol-spike regime;
    # when HAR weight is high, vol is trending in a structured way.
    if 'HAR_RV_FORECAST' in df.columns and 'EGARCH_COND_VOL' in df.columns:
        rv_actual = df['REALIZED_VOL_20'].values

        # Build forecast matrix: [HAR, EGARCH, YZ or fallback]
        f_har  = df['HAR_RV_FORECAST'].values
        f_egarch = df['EGARCH_COND_VOL'].values
        f_yz   = (df['YANGZHANG_VOL'].values
                  if 'YANGZHANG_VOL' in df.columns
                  else df['REALIZED_VOL_20'].values)

        bg_combo    = np.full(len(df), np.nan)
        bg_w_har    = np.full(len(df), np.nan)
        bg_w_egarch = np.full(len(df), np.nan)
        bg_w_yz     = np.full(len(df), np.nan)

        for i in range(252, len(df)):
            _rv   = rv_actual[i-252:i]
            _fh   = f_har[i-252:i]
            _fe   = f_egarch[i-252:i]
            _fyz  = f_yz[i-252:i]

            mask = ~(np.isnan(_rv) | np.isnan(_fh) |
                     np.isnan(_fe) | np.isnan(_fyz))
            if mask.sum() < 60:
                continue

            # MSE of each forecast against realized vol
            mse_h   = np.mean((_rv[mask] - _fh[mask])   ** 2) + 1e-10
            mse_e   = np.mean((_rv[mask] - _fe[mask])   ** 2) + 1e-10
            mse_yz  = np.mean((_rv[mask] - _fyz[mask])  ** 2) + 1e-10

            # Inverse-MSE weights (Bates-Granger)
            inv_sum = 1/mse_h + 1/mse_e + 1/mse_yz
            w_h  = (1/mse_h)  / inv_sum
            w_e  = (1/mse_e)  / inv_sum
            w_yz = (1/mse_yz) / inv_sum

            bg_w_har[i]    = w_h
            bg_w_egarch[i] = w_e
            bg_w_yz[i]     = w_yz

            fh_now  = f_har[i]
            fe_now  = f_egarch[i]
            fyz_now = f_yz[i]
            if not (np.isnan(fh_now) or np.isnan(fe_now) or np.isnan(fyz_now)):
                bg_combo[i] = w_h*fh_now + w_e*fe_now + w_yz*fyz_now

        df['VOL_COMBO_FORECAST'] = bg_combo
        df['BG_WEIGHT_HAR']      = bg_w_har      # high → trending vol regime
        df['BG_WEIGHT_EGARCH']   = bg_w_egarch   # high → spike/MR regime
        df['BG_WEIGHT_YZ']       = bg_w_yz       # high → gap-risk dominant
        df['VOL_COMBO_PCTILE']   = (
            pd.Series(bg_combo, index=df.index)
            .rolling(252, min_periods=60).rank(pct=True))
        features.extend(['VOL_COMBO_FORECAST', 'BG_WEIGHT_HAR',
                          'BG_WEIGHT_EGARCH', 'BG_WEIGHT_YZ',
                          'VOL_COMBO_PCTILE'])
        print("  ✓ Bates-Granger vol forecast combination computed")

    # ── 10. VXSLV + GVZ Implied Vol Features ─────────────────────────────────
    if 'VXSLV' in df.columns and df['VXSLV'].notna().sum() > 30:
        vxslv_mask = df['VXSLV'].notna()
        df['VXSLV_LEVEL']     = df['VXSLV']
        df['VXSLV_PCTILE']    = df['VXSLV'].rolling(252, min_periods=60).rank(pct=True)
        df['VXSLV_RV_SPREAD'] = df['VXSLV'] - df['REALIZED_VOL_20']
        df['VXSLV_ROC_5']     = df['VXSLV'].pct_change(5) * 100
        df['VXSLV_SPIKE']     = np.where(vxslv_mask,
                                          (df['VXSLV_ROC_5'] > 20).astype(float),
                                          np.nan)
        features.extend(['VXSLV_LEVEL', 'VXSLV_PCTILE', 'VXSLV_RV_SPREAD',
                          'VXSLV_ROC_5', 'VXSLV_SPIKE'])
        # Expose the proxy flag so models can distinguish real silver-vol
        # readings from GVZ-derived gap estimates (1 = modeled, 0 = observed).
        if 'VXSLV_IS_PROXY' in df.columns:
            features.append('VXSLV_IS_PROXY')

    if ('VXSLV' in df.columns and 'GVZ' in df.columns and
            df['VXSLV'].notna().sum() > 30 and df['GVZ'].notna().sum() > 30):
        both_mask = df['VXSLV'].notna() & df['GVZ'].notna()
        df['GVZ_LEVEL']               = df['GVZ']
        df['GVZ_PCTILE']              = df['GVZ'].rolling(252, min_periods=60).rank(pct=True)
        df['VXSLV_GVZ_SPREAD']        = np.where(both_mask,
                                                  df['VXSLV'] - df['GVZ'], np.nan)
        df['VXSLV_GVZ_SPREAD_PCTILE'] = (
            pd.Series(df['VXSLV_GVZ_SPREAD'], index=df.index)
            .rolling(252, min_periods=60).rank(pct=True))
        df['VXSLV_GVZ_RATIO']         = np.where(both_mask,
                                                  df['VXSLV'] / (df['GVZ'] + 0.001),
                                                  np.nan)
        features.extend(['GVZ_LEVEL', 'GVZ_PCTILE',
                          'VXSLV_GVZ_SPREAD', 'VXSLV_GVZ_SPREAD_PCTILE',
                          'VXSLV_GVZ_RATIO'])

    # Vol-of-Vol
    if 'VXSLV' in df.columns and df['VXSLV'].notna().sum() > 30:
        df['VOV_20']     = df['VXSLV'].rolling(20, min_periods=10).std()
        df['VOV_PCTILE'] = df['VOV_20'].rolling(252, min_periods=60).rank(pct=True)
        df['VOV_RATIO']  = np.where(df['VXSLV'].notna(),
                                    df['VOV_20'] / (df['VXSLV'] + 0.001), np.nan)
        features.extend(['VOV_20', 'VOV_PCTILE', 'VOV_RATIO'])

    # ── 11. Return Skewness, Kurtosis, Jump Indicators ────────────────────────
    df['SKEW_20']        = daily_ret.rolling(20, min_periods=10).skew()
    df['SKEW_60']        = daily_ret.rolling(60, min_periods=30).skew()
    df['SKEW_DIVERGENCE']= df['SKEW_20'] - df['SKEW_60']
    df['KURT_20']        = daily_ret.rolling(20, min_periods=10).kurt()
    features.extend(['SKEW_20', 'SKEW_60', 'SKEW_DIVERGENCE', 'KURT_20'])

    daily_vol   = df['REALIZED_VOL_20'] / np.sqrt(252) / 100
    jump_thresh = 2.5 * daily_vol
    abs_ret     = daily_ret.abs()
    df['JUMP_UP']        = ((daily_ret >  jump_thresh) & (abs_ret > 0.005)).astype(int)
    df['JUMP_DOWN']      = ((daily_ret < -jump_thresh) & (abs_ret > 0.005)).astype(int)
    df['JUMP_ANY']       = (df['JUMP_UP'] | df['JUMP_DOWN']).astype(int)
    df['JUMP_CLUSTER_5'] = df['JUMP_ANY'].rolling(5, min_periods=1).sum()
    df['JUMP_SIGMA']     = (daily_ret / (daily_vol + 1e-8)).clip(-10, 10)
    features.extend(['JUMP_UP', 'JUMP_DOWN', 'JUMP_ANY',
                      'JUMP_CLUSTER_5', 'JUMP_SIGMA'])

    print(f"\n  ✓ Total volatility features: {len(features)}")
    return df, features


def add_momentum_features(df):
    """Add momentum features"""
    print("\n" + "-"*70)
    print("[QUANT] Multi-Timeframe Momentum Features...")
    
    features = []
    
    for period in [5, 10, 20, 40, 60, 120, 252]:
        df[f'ROC_{period}'] = df['SI=F_Close'].pct_change(period) * 100
        features.append(f'ROC_{period}')
    
    df['MOMENTUM_SCORE'] = (df['ROC_10'] + df['ROC_20'] + df['ROC_60']) / 3
    features.append('MOMENTUM_SCORE')
    
    df['TREND_STRENGTH'] = abs(df['ROC_20']) / (df['REALIZED_VOL_20'] + 0.01)
    features.append('TREND_STRENGTH')
    
    for ma in [10, 20, 50, 100, 200]:
        sma = df['SI=F_Close'].rolling(ma).mean()
        df[f'DIST_SMA_{ma}'] = (df['SI=F_Close'] / sma - 1) * 100
        features.append(f'DIST_SMA_{ma}')
    
    sma_50 = df['SI=F_Close'].rolling(50).mean()
    sma_200 = df['SI=F_Close'].rolling(200).mean()
    df['GOLDEN_CROSS'] = ((sma_50 > sma_200) & (sma_50.shift(1) <= sma_200.shift(1))).astype(int)
    df['DEATH_CROSS'] = ((sma_50 < sma_200) & (sma_50.shift(1) >= sma_200.shift(1))).astype(int)
    df['ABOVE_200_SMA'] = (df['SI=F_Close'] > sma_200).astype(int)
    features.extend(['GOLDEN_CROSS', 'DEATH_CROSS', 'ABOVE_200_SMA'])
    
    for period in [20, 60, 252]:
        high = df['SI=F_Close'].rolling(period).max()
        low = df['SI=F_Close'].rolling(period).min()
        df[f'RANGE_POS_{period}'] = (df['SI=F_Close'] - low) / (high - low + 0.001)
        features.append(f'RANGE_POS_{period}')
    
    df['ATH_252'] = (df['SI=F_Close'] >= df['SI=F_Close'].rolling(252).max()).astype(int)
    df['NEAR_HIGH_20'] = (df['SI=F_Close'] >= df['SI=F_Close'].rolling(20).max() * 0.98).astype(int)
    features.extend(['ATH_252', 'NEAR_HIGH_20'])
    
    print(f"  ✓ Added {len(features)} momentum features")
    
    return df, features


def add_seasonality_features(df):
    """Add seasonality features"""
    print("\n" + "-"*70)
    print("[QUANT] Seasonality Features...")
    
    features = []
    
    df['MONTH'] = df['Date'].dt.month
    df['DAY_OF_WEEK'] = df['Date'].dt.dayofweek
    df['WEEK_OF_YEAR'] = df['Date'].dt.isocalendar().week.astype(int)
    df['DAY_OF_MONTH'] = df['Date'].dt.day
    df['QUARTER'] = df['Date'].dt.quarter
    features.extend(['MONTH', 'DAY_OF_WEEK', 'WEEK_OF_YEAR', 'DAY_OF_MONTH', 'QUARTER'])
    
    df['MONTH_SIN'] = np.sin(2 * np.pi * df['MONTH'] / 12)
    df['MONTH_COS'] = np.cos(2 * np.pi * df['MONTH'] / 12)
    df['DOW_SIN'] = np.sin(2 * np.pi * df['DAY_OF_WEEK'] / 5)
    df['DOW_COS'] = np.cos(2 * np.pi * df['DAY_OF_WEEK'] / 5)
    features.extend(['MONTH_SIN', 'MONTH_COS', 'DOW_SIN', 'DOW_COS'])
    
    seasonal_strength = {
        1: 0.7, 2: 0.6, 3: -0.3, 4: 0.2, 5: 0.1, 6: -0.4,
        7: 0.3, 8: 0.8, 9: 0.4, 10: 0.1, 11: 0.3, 12: 0.2,
    }
    df['SEASONAL_STRENGTH'] = df['MONTH'].map(seasonal_strength)
    features.append('SEASONAL_STRENGTH')
    
    df['MONTH_END'] = (df['DAY_OF_MONTH'] >= 25).astype(int)
    df['MONTH_START'] = (df['DAY_OF_MONTH'] <= 5).astype(int)
    df['OPEX_WEEK'] = ((df['DAY_OF_MONTH'] >= 15) & (df['DAY_OF_MONTH'] <= 21)).astype(int)
    features.extend(['MONTH_END', 'MONTH_START', 'OPEX_WEEK'])
    
    print(f"  ✓ Added {len(features)} seasonality features")
    
    return df, features


def add_liquidity_features(df):
    """Add Fed Liquidity Features"""
    print("\n" + "-"*70)
    print("[QUANT] Fed Liquidity Features...")
    
    features = []
    
    try:
        series_to_fetch = {
            'RRPONTSYD': 'RRP', 'SOFR': 'SOFR', 'EFFR': 'EFFR',
            'WALCL': 'FED_ASSETS', 'WTREGEN': 'TGA', 'M2SL': 'M2'
        }
        
        df['Date'] = pd.to_datetime(df['Date'])
        fred_end_date = Config.END_DATE if Config.END_DATE else datetime.now().strftime('%Y-%m-%d')
        
        for fred_id, col_name in series_to_fetch.items():
            try:
                data = web.DataReader(fred_id, 'fred', Config.START_DATE, fred_end_date)
                data = data.reset_index()
                data.columns = ['Date', col_name]
                data['Date'] = pd.to_datetime(data['Date'])
                df = pd.merge_asof(df.sort_values('Date'), data.sort_values('Date'),
                                  on='Date', direction='backward')
                df[col_name] = df[col_name].fillna(method='ffill')
                # ROBUSTNESS: do NOT register the raw FRED level (RRP, TGA,
                # FED_ASSETS, M2, SOFR, EFFR) as a model feature — these are
                # regime-drifting levels that leave their training range as the
                # Fed balance sheet changes.  The column is kept in df so the
                # derived stationary features below (z-scores, percentiles,
                # changes, spreads) can consume it, but the raw level itself
                # never enters model selection.
            except:
                pass
        
        if 'RRP' in df.columns:
            df['RRP_CHG_5D'] = df['RRP'].diff(5)
            df['RRP_CHG_20D'] = df['RRP'].diff(20)
            df['RRP_PCT_CHG_5D'] = df['RRP'].pct_change(5) * 100
            df['RRP_SMA_20'] = df['RRP'].rolling(20).mean()
            df['RRP_ABOVE_SMA20'] = (df['RRP'] > df['RRP_SMA_20']).astype(int)
            df['RRP_ACCELERATION'] = df['RRP_CHG_5D'].diff(5)
            # ROBUSTNESS: RRP_SMA_20 is a raw level (drifts $0→$2T over sample);
            # register RRP's 252d percentile instead so "high RRP" is regime-
            # relative.  The SMA column stays for the RRP_ABOVE_SMA20 flag.
            df['RRP_PCTILE'] = df['RRP'].rolling(252, min_periods=60).rank(pct=True)
            features.extend(['RRP_CHG_5D', 'RRP_CHG_20D', 'RRP_PCT_CHG_5D',
                           'RRP_PCTILE', 'RRP_ABOVE_SMA20', 'RRP_ACCELERATION'])
        
        if 'SOFR' in df.columns and 'EFFR' in df.columns:
            df['SOFR_FF_SPREAD'] = (df['SOFR'] - df['EFFR']) * 100
            df['SOFR_FF_SPREAD_20D_AVG'] = df['SOFR_FF_SPREAD'].rolling(20).mean()
            df['SOFR_VOL_5D'] = df['SOFR'].diff().rolling(5).std() * 100
            features.extend(['SOFR_FF_SPREAD', 'SOFR_FF_SPREAD_20D_AVG', 'SOFR_VOL_5D'])
        
        if all(col in df.columns for col in ['FED_ASSETS', 'RRP', 'TGA']):
            df['NET_LIQUIDITY'] = df['FED_ASSETS'] - df['RRP'] - df['TGA']
            df['NET_LIQ_CHG_20D'] = df['NET_LIQUIDITY'].diff(20)
            # ROBUSTNESS: the raw NET_LIQUIDITY level drifts with the Fed
            # balance sheet — a value normal in 2022 is off-scale in 2026, so a
            # tree trained on one regime can't place the other.  Express it as a
            # trailing z-score and 252d percentile so "tight vs loose" stays
            # meaningful across regimes.  Raw level kept in df (LIQ_SIGNAL etc.
            # reference it) but only the normalized forms enter the model.
            _nl_mean = df['NET_LIQUIDITY'].rolling(252, min_periods=60).mean()
            _nl_std  = df['NET_LIQUIDITY'].rolling(252, min_periods=60).std()
            df['NET_LIQ_ZSCORE']  = (df['NET_LIQUIDITY'] - _nl_mean) / (_nl_std + 1e-9)
            df['NET_LIQ_PCTILE']  = df['NET_LIQUIDITY'].rolling(252, min_periods=60).rank(pct=True)
            # NET_LIQ_CHG_20D is a difference (already ~stationary) but its
            # scale drifts too; normalize to its own trailing vol.
            _nlc_std = df['NET_LIQ_CHG_20D'].rolling(252, min_periods=60).std()
            df['NET_LIQ_CHG_20D_Z'] = df['NET_LIQ_CHG_20D'] / (_nlc_std + 1e-9)
            features.extend(['NET_LIQ_CHG_20D', 'NET_LIQ_ZSCORE',
                             'NET_LIQ_PCTILE', 'NET_LIQ_CHG_20D_Z'])
        
        if 'FED_ASSETS' in df.columns:
            df['FED_ASSETS_CHG_13W'] = df['FED_ASSETS'].diff(65)
            df['QE_REGIME'] = (df['FED_ASSETS_CHG_13W'] > 0).astype(int)
            features.extend(['FED_ASSETS_CHG_13W', 'QE_REGIME'])
        
        if 'M2' in df.columns:
            df['M2_YOY'] = df['M2'].pct_change(252) * 100
            df['M2_GROWTH_POSITIVE'] = (df['M2_YOY'] > 0).astype(int)
            features.extend(['M2_YOY', 'M2_GROWTH_POSITIVE'])
        
        if 'TGA' in df.columns:
            df['TGA_CHG_20D'] = df['TGA'].diff(20)
            df['TGA_PERCENTILE'] = df['TGA'].rolling(252).rank(pct=True)
            # normalize the 20d change to its own trailing vol (scale drifts)
            _tga_std = df['TGA_CHG_20D'].rolling(252, min_periods=60).std()
            df['TGA_CHG_20D_Z'] = df['TGA_CHG_20D'] / (_tga_std + 1e-9)
            features.extend(['TGA_CHG_20D_Z', 'TGA_PERCENTILE'])
        
        df['LIQ_SIGNAL'] = 0.0
        if 'RRP_ABOVE_SMA20' in df.columns:
            df['LIQ_SIGNAL'] += 0.25 * (1 - df['RRP_ABOVE_SMA20'] * 2)
        if 'QE_REGIME' in df.columns:
            df['LIQ_SIGNAL'] += 0.25 * (df['QE_REGIME'] * 2 - 1)
        features.append('LIQ_SIGNAL')
        
        df['LIQ_REGIME_BULLISH'] = (df['LIQ_SIGNAL'] > 0.3).astype(int)
        features.append('LIQ_REGIME_BULLISH')
        
        print(f"  ✓ Added {len(features)} liquidity features")
        
    except Exception as e:
        print(f"  ✗ Error: {e}")
    
    return df, features


def add_statistical_features(df):
    """Add statistical features"""
    print("\n" + "-"*70)
    print("[QUANT] Statistical Features...")
    
    features = []
    returns = df['SI=F_Close'].pct_change()
    
    df['SKEWNESS_20'] = returns.rolling(20).skew()
    df['KURTOSIS_20'] = returns.rolling(20).kurt()
    df['SKEWNESS_60'] = returns.rolling(60).skew()
    df['KURTOSIS_60'] = returns.rolling(60).kurt()
    features.extend(['SKEWNESS_20', 'KURTOSIS_20', 'SKEWNESS_60', 'KURTOSIS_60'])
    
    df['AUTOCORR_1'] = returns.rolling(60).apply(
        lambda x: x.autocorr(lag=1) if len(x) > 1 else 0, raw=False
    )
    features.append('AUTOCORR_1')
    
    df['UP_DAYS_20'] = (returns > 0).rolling(20).sum()
    df['UP_RATIO_20'] = df['UP_DAYS_20'] / 20
    features.extend(['UP_DAYS_20', 'UP_RATIO_20'])
    
    df['AVG_GAIN_20'] = returns.where(returns > 0, 0).rolling(20).mean() * 100
    df['AVG_LOSS_20'] = returns.where(returns < 0, 0).rolling(20).mean() * 100
    df['GAIN_LOSS_RATIO'] = abs(df['AVG_GAIN_20'] / (df['AVG_LOSS_20'] - 0.0001))
    features.extend(['AVG_GAIN_20', 'AVG_LOSS_20', 'GAIN_LOSS_RATIO'])
    
    print(f"  ✓ Added {len(features)} statistical features")
    
    return df, features


def add_technical_indicators(df):
    """Add technical indicators"""
    print("\n" + "-"*70)
    print("Computing technical indicators...")
    
    # ADX
    adx_result = ta.adx(df["SI=F_High"], df["SI=F_Low"], df["SI=F_Close"], length=14)
    if adx_result is not None and len(adx_result.columns) >= 3:
        df["SI=F_ADX"] = adx_result.iloc[:, 0]
        df["SI=F_DI_PLUS"] = adx_result.iloc[:, 1]
        df["SI=F_DI_MINUS"] = adx_result.iloc[:, 2]
        df["SI=F_DI_DIFF"] = df["SI=F_DI_PLUS"] - df["SI=F_DI_MINUS"]
        df["SI=F_TRENDING"] = (df["SI=F_ADX"] > 25).astype(int)
        df["SI=F_STRONG_TREND"] = (df["SI=F_ADX"] > 40).astype(int)
    
    # Moving Averages
    df["SI=F_SMA20"] = ta.sma(df["SI=F_Close"], length=20)
    df["SI=F_SMA50"] = ta.sma(df["SI=F_Close"], length=50)
    df["SI=F_SMA200"] = ta.sma(df["SI=F_Close"], length=200)
    df["SI=F_EMA12"] = ta.ema(df["SI=F_Close"], length=12)
    df["SI=F_EMA26"] = ta.ema(df["SI=F_Close"], length=26)
    df["SI=F_EMA50"] = ta.ema(df["SI=F_Close"], length=50)
    
    df["SI=F_Above_SMA20"] = (df["SI=F_Close"] > df["SI=F_SMA20"]).astype(int)
    df["SI=F_Above_SMA50"] = (df["SI=F_Close"] > df["SI=F_SMA50"]).astype(int)
    df["SI=F_Above_SMA200"] = (df["SI=F_Close"] > df["SI=F_SMA200"]).astype(int)
    df["SI=F_SMA20_Above_SMA50"] = (df["SI=F_SMA20"] > df["SI=F_SMA50"]).astype(int)
    df["SI=F_SMA50_Above_SMA200"] = (df["SI=F_SMA50"] > df["SI=F_SMA200"]).astype(int)
    df["SI=F_TREND_ALIGNMENT"] = (df["SI=F_Above_SMA20"] + df["SI=F_Above_SMA50"] + 
                                  df["SI=F_Above_SMA200"] + df["SI=F_SMA20_Above_SMA50"] +
                                  df["SI=F_SMA50_Above_SMA200"])
    
    # RSI
    df["SI=F_RSI_7"] = ta.rsi(df["SI=F_Close"], length=7)
    df["SI=F_RSI_14"] = ta.rsi(df["SI=F_Close"], length=14)
    df["SI=F_RSI_21"] = ta.rsi(df["SI=F_Close"], length=21)
    df["SI=F_RSI_SLOPE_5"] = df["SI=F_RSI_14"].diff(5)
    df["SI=F_PRICE_SLOPE_5"] = df["SI=F_Close"].pct_change(5) * 100
    
    # MACD
    macd_result = ta.macd(df["SI=F_Close"], fast=12, slow=26, signal=9)
    df["SI=F_MACD"] = macd_result.iloc[:, 0]
    df["SI=F_MACD_Signal"] = macd_result.iloc[:, 1]
    df["SI=F_MACD_Hist"] = macd_result.iloc[:, 2]
    df["SI=F_MACD_Hist_Chg"] = df["SI=F_MACD_Hist"].diff()
    
    # Stochastic
    stoch_result = ta.stoch(df["SI=F_High"], df["SI=F_Low"], df["SI=F_Close"], k=14, d=3)
    df["SI=F_Stoch_K"] = stoch_result.iloc[:, 0]
    df["SI=F_Stoch_D"] = stoch_result.iloc[:, 1]
    df["SI=F_Stoch_Overbought"] = (df["SI=F_Stoch_K"] > 80).astype(int)
    df["SI=F_Stoch_Oversold"] = (df["SI=F_Stoch_K"] < 20).astype(int)
    
    # CCI
    df["SI=F_CCI"] = ta.cci(df["SI=F_High"], df["SI=F_Low"], df["SI=F_Close"], length=20)
    df["SI=F_CCI_Overbought"] = (df["SI=F_CCI"] > 100).astype(int)
    df["SI=F_CCI_Oversold"] = (df["SI=F_CCI"] < -100).astype(int)
    
    # ATR
    df["SI=F_ATR"] = ta.atr(df["SI=F_High"], df["SI=F_Low"], df["SI=F_Close"], length=14)
    df["SI=F_ATR_PCT"] = df["SI=F_ATR"] / df["SI=F_Close"] * 100
    
    # Bollinger Bands
    bb_result = ta.bbands(df["SI=F_Close"], length=20, std=2)
    df["SI=F_BB_Lower"] = bb_result.iloc[:, 0]
    df["SI=F_BB_Mid"] = bb_result.iloc[:, 1]
    df["SI=F_BB_Upper"] = bb_result.iloc[:, 2]
    df["SI=F_BB_Width"] = (df["SI=F_BB_Upper"] - df["SI=F_BB_Lower"]) / df["SI=F_BB_Mid"]
    df["SI=F_BB_Position"] = (df["SI=F_Close"] - df["SI=F_BB_Lower"]) / (df["SI=F_BB_Upper"] - df["SI=F_BB_Lower"] + 0.001)
    
    # Keltner Channels
    kc_result = ta.kc(df["SI=F_High"], df["SI=F_Low"], df["SI=F_Close"], length=20)
    if kc_result is not None and len(kc_result.columns) >= 3:
        df["SI=F_KC_Lower"] = kc_result.iloc[:, 0]
        df["SI=F_KC_Mid"] = kc_result.iloc[:, 1]
        df["SI=F_KC_Upper"] = kc_result.iloc[:, 2]
        df["SI=F_SQUEEZE"] = ((df["SI=F_BB_Lower"] > df["SI=F_KC_Lower"]) & 
                              (df["SI=F_BB_Upper"] < df["SI=F_KC_Upper"])).astype(int)
    
    # OBV
    df["SI=F_OBV"] = ta.obv(df["SI=F_Close"], df["SI=F_Volume"])
    df["SI=F_OBV_SMA_20"] = df["SI=F_OBV"].rolling(20).mean()
    df["SI=F_OBV_TREND"] = (df["SI=F_OBV"] > df["SI=F_OBV_SMA_20"]).astype(int)
    
    # Volume
    df["SI=F_Volume_SMA_20"] = ta.sma(df["SI=F_Volume"], length=20)
    df["SI=F_Volume_SMA_50"] = ta.sma(df["SI=F_Volume"], length=50)
    df["SI=F_Volume_Ratio"] = df["SI=F_Volume"] / (df["SI=F_Volume_SMA_20"] + 1)
    # ROBUSTNESS: raw volume SMAs are levels with secular drift (contract
    # migration, venue shifts) — register the 50d RATIO instead so "elevated
    # volume" is relative to recent norms, not an absolute contract count.
    df["SI=F_Volume_Ratio_50"] = df["SI=F_Volume"] / (df["SI=F_Volume_SMA_50"] + 1)
    df["SI=F_Volume_Spike"] = (df["SI=F_Volume_Ratio"] > 2.0).astype(int)
    
    df["SI=F_PV_TREND"] = np.where(
        (df["SI=F_Close"] > df["SI=F_Close"].shift(1)) & (df["SI=F_Volume"] > df["SI=F_Volume_SMA_20"]),
        1,
        np.where(
            (df["SI=F_Close"] < df["SI=F_Close"].shift(1)) & (df["SI=F_Volume"] > df["SI=F_Volume_SMA_20"]),
            -1,
            0
        )
    )
    
    # Price Action
    df["SI=F_Price_Change"] = df["SI=F_Close"].pct_change()
    df["SI=F_High_Low_Range"] = (df["SI=F_High"] - df["SI=F_Low"]) / df["SI=F_Close"] * 100
    df["SI=F_Body_Size"] = abs(df["SI=F_Close"] - df["SI=F_Open"]) / df["SI=F_Close"] * 100
    df["SI=F_BULLISH_CANDLE"] = (df["SI=F_Close"] > df["SI=F_Open"]).astype(int)
    df["SI=F_DOJI"] = (df["SI=F_Body_Size"] < 0.1).astype(int)
    df["SI=F_GAP_UP"] = (df["SI=F_Open"] > df["SI=F_High"].shift(1)).astype(int)
    df["SI=F_GAP_DOWN"] = (df["SI=F_Open"] < df["SI=F_Low"].shift(1)).astype(int)
    
    print("✓ Technical indicators added")
    
    return df


def add_nadaraya_watson_envelope(df, lookback: int = 252, bandwidth: int = 50):
    """
    Add Nadaraya-Watson kernel regression envelope — rolling causal version.

    The original implementation fit KernelReg across the ENTIRE dataset, so
    NW_Position at any historical bar encoded where price went in the future
    (full look-ahead bias).  A training window from 2018 had NW_Position values
    that implicitly 'knew' the 2024 price level.

    Fix: at each bar i, the NW fit uses only the trailing `lookback` bars.
    Implemented as a closed-form Gaussian kernel weighted moving average so
    we never call KernelReg in a loop:

        w_j   = exp(-0.5 * ((i - j) / bandwidth)^2)   j in [i-lookback, i]
        fit_i = sum(w_j * price_j) / sum(w_j)

    The residual standard deviation and channel bands are also computed
    causal-only — no future data enters any bar's NW_Position value.

    Parameters
    ----------
    lookback  : bars of history per NW fit  (default 252 ~ 1 year)
    bandwidth : Gaussian kernel width in index units  (default 50)
    """
    print("\n" + "-"*70)
    print("Computing Nadaraya-Watson envelope (rolling causal)...")

    prices = df["SI=F_Close"].values.astype(float)
    n      = len(prices)

    nw_fit   = np.full(n, np.nan)
    nw_sigma = np.full(n, np.nan)

    for i in range(n):
        start   = max(0, i - lookback + 1)
        win     = prices[start : i + 1]          # causal window only
        nw      = len(win)

        # Offsets: current bar = 0, oldest bar = -(nw-1)
        offsets = np.arange(-(nw - 1), 1, dtype=float)
        w       = np.exp(-0.5 * (offsets / bandwidth) ** 2)
        w      /= w.sum()

        fit_i      = float(np.dot(w, win))
        nw_fit[i]  = fit_i

        # Per-bar fitted values across the window (for residual calc)
        # Vectorised: for each j in window, fit_j = weighted avg of win[:j+1]
        fits_in_win = np.empty(nw)
        for j in range(nw):
            sub_off = np.arange(-(j), 1, dtype=float)
            wj      = np.exp(-0.5 * (sub_off / bandwidth) ** 2)
            wj     /= wj.sum()
            fits_in_win[j] = float(np.dot(wj, win[: j + 1]))

        resid          = win - fits_in_win
        nw_sigma[i]    = resid.std() if nw > 2 else 0.0

    df["NW_Fit"]   = nw_fit
    df["NW_Upper"] = nw_fit + 2 * nw_sigma
    df["NW_Lower"] = nw_fit - 2 * nw_sigma

    # Guard: channel width can be near-zero in the first few bars (tiny
    # window → sigma ≈ 0).  Clamp channel to at least 0.1% of price so
    # NW_Position never blows up, then clip the result to [0, 1].
    channel_width = df["NW_Upper"] - df["NW_Lower"]
    min_width     = df["NW_Fit"].abs() * 0.001          # 0.1% of fitted price
    safe_width    = channel_width.clip(lower=min_width)
    df["NW_Position"] = (
        (df["SI=F_Close"] - df["NW_Lower"]) / safe_width
    ).clip(0.0, 1.0)

    # NW_Position is the top feature — but its level alone misses directionality.
    # A NW_Position of 0.70 reached in 3 days (spike) is structurally different
    # from one reached over 3 weeks (trend). Velocity captures this.
    df["NW_Position_ROC_5"]  = df["NW_Position"].diff(5)
    df["NW_Position_ROC_20"] = df["NW_Position"].diff(20)
    df["NW_VELOCITY"] = df["NW_Position_ROC_5"] / (df["NW_Position"].abs() + 0.01)

    print(f"  ✓ Nadaraya-Watson envelope added "
          f"(causal rolling, lookback={lookback}d, bw={bandwidth})")

    return df


def create_labels(df):
    """Create binary labels for prediction"""
    print("\n" + "-"*70)
    print("Creating prediction labels...")
    
    df["future_ret"] = df["SI=F_Close"].pct_change(Config.LOOKAHEAD_DAYS).shift(-Config.LOOKAHEAD_DAYS)

    # ── VOL-ADAPTIVE LABEL THRESHOLDS ─────────────────────────────────────
    # Fixed THRESHOLD_PCT = 0.02 is too lenient in high-vol regimes and too
    # strict in quiet ones.  In a sustained bull run, a flat 2% threshold
    # causes W29 to have 68% BUY rate — the model learns to always predict
    # BUY.  In quiet CONSOLIDATION, 2% is rarely crossed, producing a tiny
    # SHORT_Label population the dedicated short model can barely learn from.
    #
    # Adaptive threshold = k × σ_daily × √LOOKAHEAD_DAYS (1-sigma horizon move)
    #   k = 1.0  (one expected-move unit — mirrors options "1EM" convention)
    #   Floor: 0.015  (avoid over-filtering ultra-low-vol regimes)
    #   Cap:   0.055  (avoid under-filtering extreme vol spikes)
    #   5-day smooth: dampens single-day outliers from flipping boundaries
    _daily_vol    = df["SI=F_Close"].pct_change().rolling(20).std()
    _period_vol   = _daily_vol * np.sqrt(Config.LOOKAHEAD_DAYS)
    _adaptive_thr = (_period_vol.rolling(5, min_periods=1).mean()
                     .clip(lower=0.015, upper=0.055))
    df["ADAPTIVE_THRESHOLD"] = _adaptive_thr.fillna(Config.THRESHOLD_PCT)

    df["Label"]       = (df["future_ret"] >  df["ADAPTIVE_THRESHOLD"]).astype(int)
    # SHORT_Label: true active-short signal — price falls by at least threshold
    # This is NOT the inverse of Label (neutral periods are 0 in both)
    df["SHORT_Label"] = (df["future_ret"] < -df["ADAPTIVE_THRESHOLD"]).astype(int)
    
    initial_rows = len(df)
    critical_cols = ['SI=F_Close', 'SI=F_Open', 'SI=F_High', 'SI=F_Low', 'SI=F_Volume', 'Label', 'Date']
    
    warmup = Config.ROLLING_LOOKBACK + Config.LOOKAHEAD_DAYS
    df = df.iloc[warmup:].reset_index(drop=True)
    
    for col in critical_cols:
        if col in df.columns:
            df = df[df[col].notna()]
    
    feature_cols = [c for c in df.columns if c not in critical_cols]
    
    for col in feature_cols:
        if col in df.columns:
            # ffill only — bfill would pull future values into earlier rows
            df[col] = df[col].fillna(method='ffill').fillna(0)
            df[col] = df[col].replace([np.inf, -np.inf], 0)
    
    df = df.reset_index(drop=True)
    
    buy_count   = df['Label'].sum()
    short_count = df['SHORT_Label'].sum()
    neutral_count = len(df) - buy_count - short_count
    
    print(f"✓ Labels created")
    print(f"  Final samples: {len(df)}")
    print(f"  BUY signals:    {buy_count}   ({buy_count/len(df)*100:.1f}%)")
    print(f"  SHORT signals:  {short_count} ({short_count/len(df)*100:.1f}%)")
    print(f"  NEUTRAL:        {neutral_count} ({neutral_count/len(df)*100:.1f}%)")
    
    return df


# ══════════════════════════════════════════════════════════════════════
# ENHANCED TRADE SIMULATION WITH HIGH-SHARPE FILTERING
# ══════════════════════════════════════════════════════════════════════

def enhanced_trade_simulation(df, test_indices, predictions, regime, times_test, window_num,
                               regime_map=None):
    """Trade simulation with per-bar confirmed regime lookup.

    regime_map: dict {df_idx: confirmed_regime_str} pre-computed by
                MarketRegimeDetector.build_regime_map().  When provided, each
                bar uses its own live confirmed regime rather than the frozen
                snapshot taken at window-open.  Falls back to the static
                window-open 'regime' argument if the map is absent or the bar
                is not found in it.
    """

    # Window-open regime used as fallback (and for reporting)
    try:
        _fallback_enum = MarketRegime(regime)
    except:
        _fallback_enum = MarketRegime.CONSOLIDATION

    window_trades = []

    ensemble_preds    = predictions['ensemble_preds']
    ensemble_probs    = predictions['ensemble_probs']
    model_agreement   = predictions['model_agreement']
    # Dedicated short model outputs -- trained on SHORT_Label, not inverted BUY prob
    short_probs           = predictions.get('short_probs',           1 - ensemble_probs)
    short_model_agreement = predictions.get('short_model_agreement', 3 - model_agreement)

    for i, (pred, prob, agreement) in enumerate(zip(ensemble_preds, ensemble_probs, model_agreement)):
        test_date = times_test[i]
        df_idx_matches = df[pd.to_datetime(df['Date']) == pd.to_datetime(test_date)].index

        if len(df_idx_matches) == 0:
            continue

        df_idx = df_idx_matches[0]

        # ── PER-BAR LIVE REGIME ───────────────────────────────────────────────
        # Look up the confirmed regime for this specific bar.  If regime_map
        # is not provided (e.g. during current-signal inference) fall back to
        # the static window-open snapshot.
        if regime_map is not None:
            live_regime_str = regime_map.get(df_idx, regime)
        else:
            live_regime_str = regime
        try:
            regime_enum = MarketRegime(live_regime_str)
        except:
            regime_enum = _fallback_enum
        config = REGIME_CONFIGS[regime_enum]

        # Get MR features
        mr_feats = None
        if 'MR_SIGNAL' in df.columns:
            mr_feats = {
                'MR_SIGNAL':         df.iloc[df_idx]['MR_SIGNAL']         if df_idx < len(df) else 0,
                'ZSCORE_60':         df.iloc[df_idx]['ZSCORE_60']         if df_idx < len(df) else 0,
                'IS_MEAN_REVERTING': df.iloc[df_idx]['IS_MEAN_REVERTING'] if df_idx < len(df) else 0,
            }

        # LONG SIGNAL - with model agreement filter
        if pred == 1:
            should_trade, adj_prob, reason = EnhancedRegimeFilter.filter_signal(
                prob, regime_enum, TradeDirection.LONG, mr_feats, agreement
            )
            
            if should_trade:
                current_vol = df.iloc[df_idx]['REALIZED_VOL_20'] if 'REALIZED_VOL_20' in df.columns else 0.30
                base_contracts = max(1, int(Config.ACCOUNT_SIZE / 75000))  # More conservative
                position_size = EnhancedRegimeFilter.get_position_size(base_contracts, regime_enum, adj_prob, agreement)
                
                trade = BidirectionalTradeSimulator.simulate_trade(
                    df, df_idx, TradeDirection.LONG, position_size,
                    config.stop_loss_pct, config.take_profit_pct
                )
                
                if trade:
                    trade['window'] = window_num
                    trade['signal_prob'] = adj_prob
                    trade['regime'] = live_regime_str
                    trade['model_agreement'] = agreement
                    # ATR direction: True = vol expanding on entry (trend-following context),
                    # False = vol contracting (mean-reversion / exhaustion context).
                    # Surfaces whether HIGH_VOL gains are trending or fading.
                    if 'SI=F_ATR' in df.columns and df_idx > 0:
                        trade['atr_expanding'] = bool(
                            df.iloc[df_idx]['SI=F_ATR'] > df.iloc[df_idx - 1]['SI=F_ATR']
                        )
                    else:
                        trade['atr_expanding'] = None
                    window_trades.append(trade)
        
        # SHORT SIGNAL — use dedicated short model output directly.
        # short_probs[i] = P(price falls ≥ threshold) from a model trained on SHORT_Label.
        # This correctly separates "don't buy" from "actively short."
        if config.allow_short:
            s_prob      = short_probs[i]
            s_agreement = short_model_agreement[i]
            
            should_trade, adj_short_prob, reason = EnhancedRegimeFilter.filter_signal(
                s_prob, regime_enum, TradeDirection.SHORT, mr_feats, s_agreement
            )
            
            if should_trade:
                base_contracts = max(1, int(Config.ACCOUNT_SIZE / 65000))
                position_size  = EnhancedRegimeFilter.get_position_size(base_contracts, regime_enum, adj_short_prob, s_agreement)
                
                trade = BidirectionalTradeSimulator.simulate_trade(
                    df, df_idx, TradeDirection.SHORT, position_size,
                    config.stop_loss_pct, config.take_profit_pct
                )
                
                if trade:
                    trade['window']          = window_num
                    trade['signal_prob']     = adj_short_prob
                    trade['regime']          = live_regime_str
                    trade['model_agreement'] = s_agreement
                    if 'SI=F_ATR' in df.columns and df_idx > 0:
                        trade['atr_expanding'] = bool(
                            df.iloc[df_idx]['SI=F_ATR'] > df.iloc[df_idx - 1]['SI=F_ATR']
                        )
                    else:
                        trade['atr_expanding'] = None
                    window_trades.append(trade)
    
    return window_trades


# ══════════════════════════════════════════════════════════════════════
# MAIN EXECUTION
# ══════════════════════════════════════════════════════════════════════

def run_optimized_strategy():
    """Main function - HIGH SHARPE OPTIMIZATION"""
    
    print("\n" + "="*70)
    print("REGIME RULES (V3.4 BALANCED - High Sharpe + Smart Shorts)")
    print("="*70)
    for regime, config in REGIME_CONFIGS.items():
        print(f"\n{config.description}")
        print(f"  Allow Long: {'✓' if config.allow_long else '✗ BLOCKED'}")
        print(f"  Allow Short: {'✓' if config.allow_short else '✗ BLOCKED'}")
        print(f"  Long Threshold: {config.long_threshold:.2f}")
        print(f"  Min Agreement: {config.min_model_agreement}/3 models")
    
    # Load data
    df = load_market_data()
    
    # Feature engineering
    df, cot_features = add_cot_features(df)
    df, comtrade_features = add_comtrade_silver_trade_features(df)
    df, real_rate_features = add_real_rate_features(df)
    df, dollar_features = add_dollar_features(df)
    df, gs_features = add_enhanced_gold_silver_ratio(df)
    df, cross_asset_features = add_cross_asset_features(df)
    df, volatility_features = add_advanced_volatility_features(df)
    df, momentum_features = add_momentum_features(df)
    df, seasonality_features = add_seasonality_features(df)
    df, liquidity_features = add_liquidity_features(df)
    df, statistical_features = add_statistical_features(df)
    df, mr_features = add_mean_reversion_features(df)
    
    df = add_technical_indicators(df)
    df = add_nadaraya_watson_envelope(df)
    df = create_labels(df)
    
    # Define features
    symbols = ["SI=F", "CL=F", "GC=F", "DX-Y.NYB", "HG=F", 
               "^DJI", "^GSPC", "^IXIC", "^RUT", "^TNX", "^VIX"]
    
    tech_feats = [
        "SI=F_Open", "SI=F_High", "SI=F_Low", "SI=F_Close", "SI=F_Volume",
        "SI=F_ADX", "SI=F_DI_PLUS", "SI=F_DI_MINUS", "SI=F_DI_DIFF",
        "SI=F_TRENDING", "SI=F_STRONG_TREND",
        "SI=F_SMA20", "SI=F_SMA50", "SI=F_SMA200",
        "SI=F_EMA12", "SI=F_EMA26", "SI=F_EMA50",
        "SI=F_Above_SMA20", "SI=F_Above_SMA50", "SI=F_Above_SMA200",
        "SI=F_SMA20_Above_SMA50", "SI=F_SMA50_Above_SMA200",
        "SI=F_TREND_ALIGNMENT",
        "SI=F_RSI_7", "SI=F_RSI_14", "SI=F_RSI_21",
        "SI=F_RSI_SLOPE_5", "SI=F_PRICE_SLOPE_5",
        "SI=F_MACD", "SI=F_MACD_Signal", "SI=F_MACD_Hist", "SI=F_MACD_Hist_Chg",
        "SI=F_Stoch_K", "SI=F_Stoch_D",
        "SI=F_Stoch_Overbought", "SI=F_Stoch_Oversold",
        "SI=F_CCI", "SI=F_CCI_Overbought", "SI=F_CCI_Oversold",
        "SI=F_ATR", "SI=F_ATR_PCT",
        "SI=F_BB_Upper", "SI=F_BB_Mid", "SI=F_BB_Lower", 
        "SI=F_BB_Width", "SI=F_BB_Position",
        "SI=F_KC_Lower", "SI=F_KC_Mid", "SI=F_KC_Upper", "SI=F_SQUEEZE",
        "SI=F_OBV", "SI=F_OBV_SMA_20", "SI=F_OBV_TREND",
        "SI=F_Volume_Ratio", "SI=F_Volume_Ratio_50",
        "SI=F_Volume_Spike", "SI=F_PV_TREND",
        "SI=F_Price_Change", "SI=F_High_Low_Range", "SI=F_Body_Size",
        "SI=F_BULLISH_CANDLE", "SI=F_DOJI", "SI=F_GAP_UP", "SI=F_GAP_DOWN",
        "NW_Fit", "NW_Upper", "NW_Lower", "NW_Position",
        "NW_Position_ROC_5", "NW_Position_ROC_20", "NW_VELOCITY"
    ]
    
    all_feats = (
        tech_feats + cot_features + comtrade_features + real_rate_features +
        dollar_features + gs_features + cross_asset_features +
        volatility_features + momentum_features + seasonality_features +
        liquidity_features + statistical_features + mr_features
    )
    
    # ── ROBUSTNESS: cross-asset closes enter as STATIONARY TRANSFORMS ────────
    # Raw closes (CL=F_Close ranked #1, ^DJI_Close #5) are levels: a tree that
    # learned "crude at $70 → X" has no valid split for crude at $110 — the
    # feature exits its training range and the model faces an out-of-sample
    # cliff.  Walk-forward retraining bounds this within a window, but the
    # 2025-26 vertical move showed how far a level can travel inside one
    # window.  Raw levels also double as time proxies in trending samples
    # (crude and silver both rose for years), inviting spurious splits.
    # Each non-silver symbol therefore contributes ROC_20 / ROC_60 (momentum)
    # and a 252d percentile (position-in-range) instead.  Raw close COLUMNS
    # stay in df — ratios elsewhere (COPPER_GOLD_RATIO etc.) still use them —
    # they are simply no longer model features.  ^VIX keeps its raw level too:
    # it is already a bounded, mean-reverting index where the level itself is
    # the signal.
    _xasset_robust = []
    for _s in symbols:
        if _s == "SI=F":
            continue
        _c = f"{_s}_Close"
        if _c not in df.columns:
            continue
        df[f"{_s}_ROC_20"] = df[_c].pct_change(20) * 100
        df[f"{_s}_ROC_60"] = df[_c].pct_change(60) * 100
        df[f"{_s}_PCTILE_252"] = df[_c].rolling(252, min_periods=60).rank(pct=True)
        _xasset_robust += [f"{_s}_ROC_20", f"{_s}_ROC_60", f"{_s}_PCTILE_252"]
    if "^VIX_Close" in df.columns:
        _xasset_robust.append("^VIX_Close")
    all_feats = all_feats + _xasset_robust
    all_feats = [f for f in all_feats if f in df.columns]
    
    seen = set()
    all_feats = [x for x in all_feats if not (x in seen or seen.add(x))]
    
    print(f"\n{'='*70}")
    print(f"FEATURE SUMMARY: {len(all_feats)} total features")
    print("="*70)
    
    # Clean data
    for col in all_feats:
        df[col] = df[col].replace([np.inf, -np.inf], np.nan)
        df[col] = df[col].fillna(method='ffill').fillna(method='bfill').fillna(0)
    
    # Prepare sequences
    WINDOW = Config.WINDOW_SIZE
    A = df[all_feats].values
    y = df["Label"].values
    dates = pd.to_datetime(df["Date"]).values
    
    N = len(A) - WINDOW + 1
    X_lstm = np.stack([A[i:i+WINDOW] for i in range(N)])
    y_lstm = y[WINDOW-1:WINDOW-1+N]
    times = dates[WINDOW-1:WINDOW-1+N]
    
    # Build SHORT_Label sequence aligned to X_lstm windows
    y_short = df["SHORT_Label"].values
    y_short_lstm = y_short[WINDOW-1:WINDOW-1+N]
    
    # RevIN inside the LSTM handles per-sequence normalisation for all features
    # uniformly — including OHLC. No explicit preprocessing needed here.
    
    X_rf = X_lstm[:, -1, :]
    
    print(f"✓ LSTM sequences: {X_lstm.shape}")
    print(f"✓ RF features: {X_rf.shape}")
    
    # Walk-Forward Validation with FIXED date range
    print("\n" + "="*70)
    print("WALK-FORWARD VALIDATION (V3.4 BALANCED)")
    print("="*70)
    
    time_index = pd.to_datetime(times)
    start_date = time_index.min()
    end_date = time_index.max()
    
    print(f"Data range: {start_date.date()} to {end_date.date()}")
    
    # FIXED: Use actual data end, not config end
    # Allow testing closer to end since we're simulating trades
    first_test_start = start_date + pd.DateOffset(years=Config.TRAIN_YEARS)
    current_test_start = first_test_start
    
    walk_forward_results = []
    all_trades = []
    window_num = 0

    # ── OOS Brier state ───────────────────────────────────────────────────────
    _prev_oos_brier       = None
    _prev_oos_brier_short = None
    _prev2_oos_brier      = None
    _prev2_oos_brier_short= None

    # ── Cross-window Platt state ──────────────────────────────────────────────
    _prev_cal_rf_raw   = None
    _prev_cal_xgb_raw  = None
    _prev_cal_y        = None
    _prev_cal_rf_raw_short  = None
    _prev_cal_xgb_raw_short = None
    _prev_cal_y_short       = None
    
    while current_test_start <= end_date - pd.DateOffset(days=1):  # FIXED: was LOOKAHEAD_DAYS
        window_num += 1
        
        train_start = current_test_start - pd.DateOffset(years=Config.TRAIN_YEARS)
        train_end = current_test_start - pd.DateOffset(days=1)
        test_start = current_test_start
        test_end = current_test_start + pd.DateOffset(months=Config.TEST_MONTHS) - pd.DateOffset(days=1)
        
        # FIXED: Allow test up to actual data end
        if test_end > end_date:
            test_end = end_date
        
        if (test_end - test_start).days < 7:
            break
        
        print(f"\n{'─'*60}")
        print(f"Window {window_num}:")
        print(f"  Train: {train_start.date()} to {train_end.date()}")
        print(f"  Test:  {test_start.date()} to {test_end.date()}")
        
        train_mask = (time_index >= train_start) & (time_index <= train_end)
        test_mask = (time_index >= test_start) & (time_index <= test_end)
        
        train_indices = np.where(train_mask)[0]
        test_indices = np.where(test_mask)[0]
        
        if len(train_indices) < Config.MIN_TRAIN_SAMPLES or len(test_indices) < 10:
            print(f"  ⚠ Skipping - insufficient data")
            current_test_start += pd.DateOffset(months=Config.TEST_MONTHS)
            continue
        
        X_lstm_train = X_lstm[train_indices]
        X_rf_train = X_rf[train_indices]
        y_train = y_lstm[train_indices]
        y_short_train = y_short_lstm[train_indices]
        
        X_lstm_test = X_lstm[test_indices]
        X_rf_test = X_rf[test_indices]
        y_test = y_lstm[test_indices]
        times_test = times[test_indices]
        
        print(f"  Train: {len(X_lstm_train)} samples, Test: {len(X_lstm_test)} samples")
        print(f"  Train BUY rate: {y_train.mean():.1%}")
        
        # Window-open regime snapshot (used for reporting and as fallback)
        regime = MarketRegimeDetector.detect_regime(df.iloc[:test_indices[-1]+WINDOW])
        print(f"  Market Regime (window-open): {MarketRegimeDetector.get_regime_description(regime)}")

        # ── PER-BAR CONFIRMED REGIME MAP ─────────────────────────────────────
        # Convert test_indices (sequence indices) to df row indices, then build
        # a confirmed regime for every bar.  The test window df_indices are the
        # actual dataframe positions that correspond to each sequence endpoint.
        test_df_indices = []
        for seq_idx in test_indices:
            t = times[seq_idx]
            matches = df[pd.to_datetime(df['Date']) == pd.to_datetime(t)].index
            if len(matches) > 0:
                test_df_indices.append(matches[0])

        regime_map = MarketRegimeDetector.build_regime_map(
            df, test_df_indices, confirmation_days=5
        )

        # Log any regime transitions within this window
        unique_regimes = list(dict.fromkeys(regime_map.values()))  # ordered unique
        if len(unique_regimes) > 1:
            print(f"  Regime transitions this window: {' -> '.join(unique_regimes)}")

        try:
            regime_enum = MarketRegime(regime)
            config = REGIME_CONFIGS[regime_enum]
            if not config.allow_long:
                print(f"  ⚠️ LONGS BLOCKED in {regime}")
            if not config.allow_short:
                print(f"  ⚠️ SHORTS BLOCKED in {regime}")
        except:
            pass

        # Train ensemble (BUY models + dedicated SHORT models)
        ensemble = OptimizedEnsemble()
        ensemble.train(X_lstm_train, X_rf_train, y_train,
                       y_short_train=y_short_train, feature_names=all_feats,
                       oos_brier_scores=_prev_oos_brier,
                       oos_brier_scores_short=_prev_oos_brier_short,
                       prev2_oos_brier_scores=_prev2_oos_brier,
                       prev2_oos_brier_scores_short=_prev2_oos_brier_short,
                       prev_cal_rf_raw=_prev_cal_rf_raw,
                       prev_cal_xgb_raw=_prev_cal_xgb_raw,
                       prev_cal_y=_prev_cal_y,
                       prev_cal_rf_raw_short=_prev_cal_rf_raw_short,
                       prev_cal_xgb_raw_short=_prev_cal_xgb_raw_short,
                       prev_cal_y_short=_prev_cal_y_short)

        # Generate predictions — specialist routing via per_bar_regimes
        per_bar_regimes = []
        for _seq_idx in test_indices:
            _t = times[_seq_idx]
            _matches = df[pd.to_datetime(df['Date']) == pd.to_datetime(_t)].index
            _live_r = regime_map.get(_matches[0], regime) if len(_matches) > 0 else regime
            per_bar_regimes.append(_live_r)

        predictions = ensemble.predict(X_lstm_test, X_rf_test, regime,
                                       per_bar_regimes=per_bar_regimes)

        # ── OOS Brier + cross-window Platt raw prob extraction ────────────────
        try:
            _oos_lstm_probs = predictions['lstm_probs']
            _oos_rf_probs   = predictions['rf_probs']
            _oos_xgb_probs  = predictions['xgb_probs']
            _oos_y          = y_test.astype(float)
            def _brier_oos(p, y): return float(np.mean((np.asarray(p) - y) ** 2))
            _curr_oos_brier = {
                'lstm': max(_brier_oos(_oos_lstm_probs, _oos_y), 1e-6),
                'rf':   max(_brier_oos(_oos_rf_probs,   _oos_y), 1e-6),
                'xgb':  max(_brier_oos(_oos_xgb_probs,  _oos_y), 1e-6),
            }

            # BUY raw (pre-Platt) probs from stored base models
            try:
                _X_rf_test_sel = ensemble.feature_selector.transform(X_rf_test, all_feats)
                _curr_cal_rf_raw = ensemble.rf_model_base.predict_proba(
                    _X_rf_test_sel)[:, 1] if ensemble.rf_model_base is not None \
                    else np.asarray(_oos_rf_probs)
                _curr_cal_xgb_raw = ensemble.xgb_model_base.predict_proba(
                    _X_rf_test_sel)[:, 1] if ensemble.xgb_model_base is not None \
                    else np.asarray(_oos_xgb_probs)
                _curr_cal_y = _oos_y.copy()
            except Exception:
                _curr_cal_rf_raw  = np.asarray(_oos_rf_probs)
                _curr_cal_xgb_raw = np.asarray(_oos_xgb_probs)
                _curr_cal_y       = _oos_y.copy()

            # SHORT OOS Brier (positive bars only) + SHORT raw probs
            _y_short_test   = y_short_lstm[test_indices].astype(float)
            _short_pos_mask = (_y_short_test == 1)
            if predictions.get('short_lstm_probs') is not None and _short_pos_mask.sum() >= 1:
                _sl = np.asarray(predictions['short_lstm_probs'])[_short_pos_mask]
                _sr = np.asarray(predictions['short_rf_probs'])[_short_pos_mask]
                _sx = np.asarray(predictions['short_xgb_probs'])[_short_pos_mask]
                _sy = _y_short_test[_short_pos_mask]
                _curr_oos_brier_short = {
                    'lstm': max(_brier_oos(_sl, _sy), 1e-6),
                    'rf':   max(_brier_oos(_sr, _sy), 1e-6),
                    'xgb':  max(_brier_oos(_sx, _sy), 1e-6),
                }
                try:
                    _X_rf_test_short_sel = (
                        ensemble.short_feature_selector.transform(X_rf_test, all_feats)
                        if ensemble.short_feature_selector is not None
                        else ensemble.feature_selector.transform(X_rf_test, all_feats)
                    )
                    _rf_short_raw  = ensemble.rf_short_base.predict_proba(
                        _X_rf_test_short_sel)[:, 1] if ensemble.rf_short_base is not None \
                        else np.asarray(predictions['short_rf_probs'])
                    _xgb_short_raw = ensemble.xgb_short_base.predict_proba(
                        _X_rf_test_short_sel)[:, 1] if ensemble.xgb_short_base is not None \
                        else np.asarray(predictions['short_xgb_probs'])
                except Exception:
                    _rf_short_raw  = np.asarray(predictions['short_rf_probs'])
                    _xgb_short_raw = np.asarray(predictions['short_xgb_probs'])
                _curr_cal_rf_raw_short  = _rf_short_raw
                _curr_cal_xgb_raw_short = _xgb_short_raw
                _curr_cal_y_short       = _y_short_test.copy()
            else:
                _curr_oos_brier_short   = None
                _curr_cal_rf_raw_short  = None
                _curr_cal_xgb_raw_short = None
                _curr_cal_y_short       = None

            # Roll state: prev2 ← prev ← curr
            _prev2_oos_brier        = _prev_oos_brier
            _prev2_oos_brier_short  = _prev_oos_brier_short
            _prev_oos_brier         = _curr_oos_brier
            _prev_oos_brier_short   = _curr_oos_brier_short
            _prev_cal_rf_raw        = _curr_cal_rf_raw
            _prev_cal_xgb_raw       = _curr_cal_xgb_raw
            _prev_cal_y             = _curr_cal_y
            _prev_cal_rf_raw_short  = _curr_cal_rf_raw_short
            _prev_cal_xgb_raw_short = _curr_cal_xgb_raw_short
            _prev_cal_y_short       = _curr_cal_y_short

        except Exception as _e:
            print(f"  ⚠ OOS Brier/cal computation failed ({_e}) — next window uses val slice")
            _prev2_oos_brier        = _prev_oos_brier
            _prev2_oos_brier_short  = _prev_oos_brier_short
            _prev_oos_brier         = None
            _prev_oos_brier_short   = None
            _prev_cal_rf_raw        = None
            _prev_cal_xgb_raw       = None
            _prev_cal_y             = None
            _prev_cal_rf_raw_short  = None
            _prev_cal_xgb_raw_short = None
            _prev_cal_y_short       = None

        # Store results
        result = {
            'window': window_num,
            'train_start': train_start,
            'train_end': train_end,
            'test_start': test_start,
            'test_end': test_end,
            'regime': regime,
            'y_true': y_test,
            'times': times_test,
            **predictions
        }
        
        walk_forward_results.append(result)
        
        # Print performance
        if len(y_test) > 0:
            acc = accuracy_score(y_test, predictions['ensemble_preds'])
            prec = precision_score(y_test, predictions['ensemble_preds'], zero_division=0)
            rec = recall_score(y_test, predictions['ensemble_preds'], zero_division=0)
            signals = np.sum(predictions['ensemble_preds'])
            high_agree = np.sum(predictions['model_agreement'] >= 2)
            
            print(f"  Performance: Acc={acc:.3f}, Prec={prec:.3f}, Rec={rec:.3f}")
            print(f"  Signals: {signals}/{len(y_test)} | High agreement (2+/3): {high_agree}")
        
        # Trade simulation -- pass regime_map so each bar uses its own
        # confirmed live regime rather than the frozen window-open snapshot
        window_trades = enhanced_trade_simulation(
            df, test_indices, predictions, regime, times_test, window_num,
            regime_map=regime_map
        )
        
        all_trades.extend(window_trades)
        
        # Store trained models on LAST window for current-day inference
        # We keep the ensemble object to use after the loop
        last_ensemble = ensemble
        last_all_feats = all_feats

        # ── TF MEMORY CLEANUP ─────────────────────────────────────────────────
        # TensorFlow accumulates LSTM computational graphs across windows and
        # does not release them — by window ~20 this causes a C-level abort.
        # clear_session() flushes TF's graph registry only — trained weight
        # tensors live in last_ensemble and are NOT affected.
        # Guard: don't clear on the final window — models needed for inference.
        if window_num < 29:
            tf.keras.backend.clear_session()
            gc.collect()
        
        current_test_start += pd.DateOffset(months=Config.TEST_MONTHS)
    
    # After loop completes, store models in the last result
    # last_ensemble is guaranteed to exist if the loop ran at least once
    if walk_forward_results:
        print("\n  💾 Storing trained models for current-day inference...")
        walk_forward_results[-1]['lstm_model'] = last_ensemble.lstm_model
        walk_forward_results[-1]['rf_model'] = last_ensemble.rf_model
        walk_forward_results[-1]['xgb_model'] = last_ensemble.xgb_model
        # Dedicated short models
        walk_forward_results[-1]['lstm_short'] = last_ensemble.lstm_short
        walk_forward_results[-1]['rf_short']   = last_ensemble.rf_short
        walk_forward_results[-1]['xgb_short']  = last_ensemble.xgb_short
        walk_forward_results[-1]['scaler'] = last_ensemble.scaler_lstm
        walk_forward_results[-1]['feature_selector'] = last_ensemble.feature_selector
        # Dedicated short feature selector — required so live inference runs the
        # short ensemble through the SAME feature pipeline as predict()
        walk_forward_results[-1]['short_feature_selector'] = last_ensemble.short_feature_selector
        # Brier-calibrated ensemble weights — live inference must blend with
        # these, not equal weights, to match the backtest exactly
        walk_forward_results[-1]['buy_weights']   = np.asarray(last_ensemble.buy_weights,   dtype=float)
        walk_forward_results[-1]['short_weights'] = np.asarray(last_ensemble.short_weights, dtype=float)
        walk_forward_results[-1]['feature_cols'] = last_all_feats
        walk_forward_results[-1]['seq_len'] = Config.WINDOW_SIZE
    
    # Results Summary
    print("\n" + "="*70)
    print("WALK-FORWARD VALIDATION RESULTS (V3.4 BALANCED)")
    print("="*70)
    
    if walk_forward_results:
        all_preds = np.concatenate([r['ensemble_preds'] for r in walk_forward_results])
        all_actuals = np.concatenate([r['y_true'] for r in walk_forward_results])
        
        print(f"\nOverall Performance ({len(walk_forward_results)} windows):")
        print("-"*50)
        print(classification_report(all_actuals, all_preds, digits=3, target_names=['SELL', 'BUY']))
        
        print("\nConfusion Matrix:")
        cm = confusion_matrix(all_actuals, all_preds)
        print(f"  TN: {cm[0,0]:4d} | FP: {cm[0,1]:4d}")
        print(f"  FN: {cm[1,0]:4d} | TP: {cm[1,1]:4d}")
    
    # Trading Results
    print("\n" + "="*70)
    print("TRADING SIMULATION RESULTS (V3.4 BALANCED)")
    print("="*70)
    
    total_pnl = 0
    win_rate = 0
    sharpe = 0
    
    if all_trades:
        trades_df = pd.DataFrame(all_trades)
        
        total_trades = len(trades_df)
        profitable_trades = (trades_df['pnl'] > 0).sum()
        win_rate = profitable_trades / total_trades if total_trades > 0 else 0
        total_pnl = trades_df['pnl'].sum()
        avg_pnl = trades_df['pnl'].mean()
        avg_return = trades_df['return_pct'].mean()
        std_return = trades_df['return_pct'].std()
        sharpe = (avg_return / std_return) * np.sqrt(252/15) if std_return > 0 else 0  # Annualized
        
        print(f"\nOverall Trading Performance:")
        print("-"*50)
        print(f"  Total trades:      {total_trades}")
        print(f"  Profitable trades: {profitable_trades} ({win_rate:.1%})")
        print(f"  Total P&L:         ${total_pnl:,.0f}")
        print(f"  Average P&L:       ${avg_pnl:,.0f}")
        print(f"  Average return:    {avg_return:.2f}%")
        print(f"  Std return:        {std_return:.2f}%")
        print(f"  SHARPE RATIO:      {sharpe:.3f}")
        print(f"  Best trade:        {trades_df['return_pct'].max():.2f}%")
        print(f"  Worst trade:       {trades_df['return_pct'].min():.2f}%")
        
        print(f"\nExit Type Breakdown:")
        print("-"*50)
        for exit_type in trades_df['exit_type'].unique():
            count = (trades_df['exit_type'] == exit_type).sum()
            pnl = trades_df[trades_df['exit_type'] == exit_type]['pnl'].sum()
            print(f"  {exit_type:<15}: {count:4d} trades, ${pnl:>10,.0f} P&L")
        
        print(f"\nPerformance by Regime:")
        print("-"*50)
        regime_stats = trades_df.groupby('regime').agg({
            'pnl': ['count', 'sum', 'mean'],
            'return_pct': ['mean', 'std']
        }).round(2)
        print(regime_stats)
        
        # Calculate Sharpe by regime
        print(f"\nSharpe by Regime:")
        for regime in trades_df['regime'].unique():
            r_trades = trades_df[trades_df['regime'] == regime]
            r_sharpe = (r_trades['return_pct'].mean() / r_trades['return_pct'].std()) * np.sqrt(252/15) if r_trades['return_pct'].std() > 0 else 0
            print(f"  {regime}: {r_sharpe:.3f}")
        
        if 'direction' in trades_df.columns:
            print(f"\nPerformance by Direction:")
            dir_stats = trades_df.groupby('direction').agg({
                'pnl': ['count', 'sum', 'mean'],
                'return_pct': ['mean', 'std']
            }).round(2)
            print(dir_stats)

            # ── SHORT ROBUSTNESS CHECK ────────────────────────────────────────
            # STRONG_DOWNTREND shorts were re-enabled specifically to capture
            # declines like 2026.  That episode is IN-SAMPLE for the decision —
            # we already knew the outcome when the config was changed.  The only
            # honest test is whether shorts also pay in the OTHER drawdowns in
            # the dataset (e.g. 2020 COVID, 2021-22 unwind).  If short P&L is
            # positive only in 2026, that is curve-fitting, not an edge.
            _sh = trades_df[trades_df['direction'] == 'SHORT'].copy()
            if len(_sh) > 0 and 'entry_date' in _sh.columns:
                _sh['entry_date'] = pd.to_datetime(_sh['entry_date'], errors='coerce')
                _sh['year'] = _sh['entry_date'].dt.year
                print(f"\nSHORT robustness — P&L by year (want positive in "
                      f"MULTIPLE years, not just 2026):")
                for yr, grp in _sh.groupby('year'):
                    if pd.isna(yr):
                        continue
                    _wr = (grp['pnl'] > 0).mean() * 100
                    print(f"  {int(yr)}: {len(grp):3d} trades, {_wr:5.1f}% win rate, "
                          f"${grp['pnl'].sum():>10,.0f}")
                _pos_years = sum(1 for _, g in _sh.groupby('year') if g['pnl'].sum() > 0)
                _tot_years = _sh['year'].nunique()
                print(f"  -> profitable in {_pos_years}/{_tot_years} years")
                if 'regime' in _sh.columns:
                    _dn = _sh[_sh['regime'] == 'STRONG_DOWNTREND']
                    if len(_dn) > 0:
                        print(f"  STRONG_DOWNTREND shorts only: {len(_dn)} trades, "
                              f"{(_dn['pnl'] > 0).mean()*100:.1f}% win rate, "
                              f"${_dn['pnl'].sum():,.0f}")
                    else:
                        print(f"  STRONG_DOWNTREND shorts only: 0 trades "
                              f"(threshold 0.72 + 3/3 agreement never met)")
        
        if 'model_agreement' in trades_df.columns:
            print(f"\nPerformance by Model Agreement:")
            for agree in sorted(trades_df['model_agreement'].unique()):
                a_trades = trades_df[trades_df['model_agreement'] == agree]
                a_wr = (a_trades['pnl'] > 0).mean() * 100
                a_pnl = a_trades['pnl'].sum()
                print(f"  {agree}/3 models: {len(a_trades):3d} trades, {a_wr:.1f}% win rate, ${a_pnl:,.0f}")
    
    # Save results with error handling
    print("\n" + "="*70)
    print("SAVING RESULTS")
    print("="*70)
    
    # Use user's Documents folder for better write permissions
    import os
    try:
        docs_folder = os.path.expanduser("~/Documents")
        if not os.path.exists(docs_folder):
            docs_folder = os.getcwd()
    except:
        docs_folder = os.getcwd()
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    if all_trades:
        trades_filename = os.path.join(docs_folder, f'enhanced_trades_v34_balanced_{timestamp}.csv')
        try:
            trades_df.to_csv(trades_filename, index=False)
            print(f"✓ Saved: {trades_filename}")
        except PermissionError:
            # Try alternative filename
            alt_filename = os.path.join(docs_folder, f'trades_v34_{timestamp}.csv')
            try:
                trades_df.to_csv(alt_filename, index=False)
                print(f"✓ Saved: {alt_filename}")
            except Exception as e:
                print(f"⚠ Could not save trades CSV: {e}")
                print("  TIP: Close any Excel files that have the CSV open")
    
    summary = {
        'timestamp': [datetime.now()],
        'version': ['V3.4 Balanced'],
        'total_windows': [len(walk_forward_results)],
        'total_trades': [len(all_trades) if all_trades else 0],
        'total_pnl': [total_pnl if all_trades else 0],
        'win_rate': [win_rate if all_trades else 0],
        'sharpe': [sharpe if all_trades else 0],
    }
    
    summary_filename = os.path.join(docs_folder, f'strategy_summary_v34_{timestamp}.csv')
    try:
        pd.DataFrame(summary).to_csv(summary_filename, index=False)
        print(f"✓ Saved: {summary_filename}")
    except Exception as e:
        print(f"⚠ Could not save summary CSV: {e}")
    
    return df, walk_forward_results, all_trades


# ══════════════════════════════════════════════════════════════════════
# VISUALIZATIONS
# ══════════════════════════════════════════════════════════════════════

def plot_trades_on_price(df, trades_df, save_path='7_trades_on_price.png'):
    """
    Plot all trades on the silver price curve
    Shows entry/exit points, long vs short, wins vs losses
    """
    print("\n" + "="*70)
    print("📊 GENERATING TRADE VISUALIZATION ON PRICE CURVE")
    print("="*70)
    
    if trades_df is None or len(trades_df) == 0:
        print("No trades to visualize")
        return
    
    # Ensure date types
    df['Date'] = pd.to_datetime(df['Date'])
    trades_df['entry_date'] = pd.to_datetime(trades_df['entry_date'])
    trades_df['exit_date'] = pd.to_datetime(trades_df['exit_date'])
    
    # Color scheme
    COLORS = {
        'price': '#2c3e50',
        'long_win': '#27ae60',
        'long_loss': '#e74c3c',
        'short_win': '#3498db',
        'short_loss': '#e67e22',
        'entry': '#9b59b6',
        'exit_win': '#27ae60',
        'exit_loss': '#e74c3c',
    }
    
    # Get test period (first trade to last trade)
    min_date = trades_df['entry_date'].min()
    max_date = trades_df['exit_date'].max()
    
    # Filter price data to test period with some buffer
    buffer_days = 30
    price_mask = (df['Date'] >= min_date - pd.Timedelta(days=buffer_days)) & \
                 (df['Date'] <= max_date + pd.Timedelta(days=buffer_days))
    price_df = df[price_mask].copy()
    
    # ═══════════════════════════════════════════════════════════════════
    # FIGURE 1: Full Overview
    # ═══════════════════════════════════════════════════════════════════
    
    fig, axes = plt.subplots(4, 1, figsize=(20, 16), height_ratios=[3, 1, 1, 1])
    
    # --- Main Price Chart with Trades ---
    ax1 = axes[0]
    
    # Plot price
    ax1.plot(price_df['Date'], price_df['SI=F_Close'], color=COLORS['price'], 
             linewidth=1.5, label='Silver Price', zorder=1)
    
    # Add moving averages
    if 'SI=F_SMA50' in price_df.columns:
        ax1.plot(price_df['Date'], price_df['SI=F_SMA50'], color='orange', 
                 linewidth=1, alpha=0.7, label='SMA 50')
    if 'SI=F_SMA200' in price_df.columns:
        ax1.plot(price_df['Date'], price_df['SI=F_SMA200'], color='red', 
                 linewidth=1, alpha=0.7, label='SMA 200')
    
    # Plot each trade
    for idx, trade in trades_df.iterrows():
        entry_date = trade['entry_date']
        exit_date = trade['exit_date']
        entry_price = trade['entry_price']
        exit_price = trade['exit_price']
        is_long = trade.get('direction', 'LONG') == 'LONG'
        is_win = trade['pnl'] > 0
        
        # Determine colors
        if is_long:
            trade_color = COLORS['long_win'] if is_win else COLORS['long_loss']
            marker_entry = '^'  # Up triangle for long entry
        else:
            trade_color = COLORS['short_win'] if is_win else COLORS['short_loss']
            marker_entry = 'v'  # Down triangle for short entry
        
        marker_exit = 'o'  # Circle for exit
        
        # Draw trade line (entry to exit)
        ax1.plot([entry_date, exit_date], [entry_price, exit_price], 
                 color=trade_color, linewidth=1.5, alpha=0.6, zorder=2)
        
        # Entry marker
        ax1.scatter(entry_date, entry_price, color=trade_color, marker=marker_entry, 
                    s=80, zorder=3, edgecolors='black', linewidths=0.5)
        
        # Exit marker
        ax1.scatter(exit_date, exit_price, color=trade_color, marker=marker_exit, 
                    s=60, zorder=3, edgecolors='black', linewidths=0.5)
    
    # Create legend handles
    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], color=COLORS['price'], linewidth=2, label='Silver Price'),
        Line2D([0], [0], marker='^', color=COLORS['long_win'], markersize=10, 
               linestyle='None', label='Long Win'),
        Line2D([0], [0], marker='^', color=COLORS['long_loss'], markersize=10, 
               linestyle='None', label='Long Loss'),
        Line2D([0], [0], marker='v', color=COLORS['short_win'], markersize=10, 
               linestyle='None', label='Short Win'),
        Line2D([0], [0], marker='v', color=COLORS['short_loss'], markersize=10, 
               linestyle='None', label='Short Loss'),
    ]
    ax1.legend(handles=legend_elements, loc='upper left', fontsize=9)
    
    ax1.set_ylabel('Silver Price ($)', fontsize=12)
    ax1.set_title('🥈 SILVER PRICE WITH ALL TRADES - Walk-Forward Test Period', 
                  fontsize=14, fontweight='bold')
    ax1.grid(True, alpha=0.3)
    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax1.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    
    # Stats box
    total_trades = len(trades_df)
    wins = (trades_df['pnl'] > 0).sum()
    total_pnl = trades_df['pnl'].sum()
    long_trades = (trades_df['direction'] == 'LONG').sum() if 'direction' in trades_df.columns else total_trades
    short_trades = total_trades - long_trades
    
    stats_text = (f"Total Trades: {total_trades}\n"
                  f"Wins: {wins} ({wins/total_trades*100:.1f}%)\n"
                  f"Long: {long_trades} | Short: {short_trades}\n"
                  f"Total P&L: ${total_pnl:,.0f}")
    ax1.text(0.02, 0.98, stats_text, transform=ax1.transAxes, fontsize=10,
             verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))
    
    # --- Cumulative P&L ---
    ax2 = axes[1]
    trades_sorted = trades_df.sort_values('exit_date').copy()
    trades_sorted['cumulative_pnl'] = trades_sorted['pnl'].cumsum()
    
    ax2.fill_between(trades_sorted['exit_date'], 0, trades_sorted['cumulative_pnl'],
                     where=trades_sorted['cumulative_pnl'] >= 0, color='#27ae60', alpha=0.3)
    ax2.fill_between(trades_sorted['exit_date'], 0, trades_sorted['cumulative_pnl'],
                     where=trades_sorted['cumulative_pnl'] < 0, color='#e74c3c', alpha=0.3)
    ax2.plot(trades_sorted['exit_date'], trades_sorted['cumulative_pnl'], 
             color='#2c3e50', linewidth=2)
    ax2.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    ax2.set_ylabel('Cumulative P&L ($)')
    ax2.set_title('💰 Cumulative P&L Over Time', fontweight='bold')
    ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x:,.0f}'))
    ax2.grid(True, alpha=0.3)
    ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    
    # --- Trade P&L Bars ---
    ax3 = axes[2]
    colors = ['#27ae60' if x > 0 else '#e74c3c' for x in trades_sorted['pnl']]
    ax3.bar(range(len(trades_sorted)), trades_sorted['pnl'], color=colors, alpha=0.7)
    ax3.axhline(y=0, color='gray', linestyle='-', alpha=0.5)
    ax3.set_ylabel('Trade P&L ($)')
    ax3.set_xlabel('Trade Number')
    ax3.set_title('📊 Individual Trade P&L', fontweight='bold')
    ax3.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x:,.0f}'))
    ax3.grid(True, alpha=0.3, axis='y')
    
    # --- Regime Background on Price ---
    ax4 = axes[3]
    ax4.plot(price_df['Date'], price_df['SI=F_Close'], color=COLORS['price'], linewidth=1)
    
    # Color background by regime for each trade
    regime_colors = {
        'STRONG_UPTREND': '#27ae60',
        'STRONG_DOWNTREND': '#e74c3c',
        'CONSOLIDATION': '#3498db',
        'HIGH_VOLATILITY': '#f39c12',
    }
    
    for idx, trade in trades_df.iterrows():
        regime = trade.get('regime', 'CONSOLIDATION')
        color = regime_colors.get(regime, '#95a5a6')
        ax4.axvspan(trade['entry_date'], trade['exit_date'], alpha=0.2, color=color)
    
    # Add regime legend
    regime_handles = [plt.Rectangle((0,0), 1, 1, fc=color, alpha=0.4) 
                      for regime, color in regime_colors.items()]
    ax4.legend(regime_handles, regime_colors.keys(), loc='upper left', fontsize=8, ncol=2)
    
    ax4.set_ylabel('Price ($)')
    ax4.set_xlabel('Date')
    ax4.set_title('🎯 Trade Periods Colored by Market Regime', fontweight='bold')
    ax4.grid(True, alpha=0.3)
    ax4.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax4.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    print(f"  ✓ Saved: {save_path}")
    plt.close()
    
    # ═══════════════════════════════════════════════════════════════════
    # FIGURE 2: Zoomed Windows (4 most interesting windows)
    # ═══════════════════════════════════════════════════════════════════
    
    # Find windows with most trades
    window_counts = trades_df.groupby('window').size().sort_values(ascending=False)
    top_windows = window_counts.head(4).index.tolist()
    
    fig, axes = plt.subplots(2, 2, figsize=(18, 12))
    axes = axes.flatten()
    
    for ax_idx, window_num in enumerate(top_windows):
        ax = axes[ax_idx]
        
        window_trades = trades_df[trades_df['window'] == window_num]
        if len(window_trades) == 0:
            continue
        
        # Get date range for this window
        w_min = window_trades['entry_date'].min() - pd.Timedelta(days=10)
        w_max = window_trades['exit_date'].max() + pd.Timedelta(days=10)
        
        w_price = df[(df['Date'] >= w_min) & (df['Date'] <= w_max)]
        
        # Plot price
        ax.plot(w_price['Date'], w_price['SI=F_Close'], color=COLORS['price'], 
                linewidth=2, label='Price')
        
        # Plot trades
        for idx, trade in window_trades.iterrows():
            is_long = trade.get('direction', 'LONG') == 'LONG'
            is_win = trade['pnl'] > 0
            
            if is_long:
                color = COLORS['long_win'] if is_win else COLORS['long_loss']
                marker = '^'
            else:
                color = COLORS['short_win'] if is_win else COLORS['short_loss']
                marker = 'v'
            
            # Trade line
            ax.plot([trade['entry_date'], trade['exit_date']], 
                    [trade['entry_price'], trade['exit_price']],
                    color=color, linewidth=2, alpha=0.7)
            
            # Entry
            ax.scatter(trade['entry_date'], trade['entry_price'], 
                       color=color, marker=marker, s=120, zorder=5, 
                       edgecolors='black', linewidths=1)
            
            # Exit
            ax.scatter(trade['exit_date'], trade['exit_price'], 
                       color=color, marker='s', s=80, zorder=5,
                       edgecolors='black', linewidths=1)
        
        # Window stats
        w_pnl = window_trades['pnl'].sum()
        w_wins = (window_trades['pnl'] > 0).sum()
        w_total = len(window_trades)
        regime = window_trades['regime'].iloc[0] if 'regime' in window_trades.columns else 'Unknown'
        
        ax.set_title(f"Window {window_num}: {regime}\n"
                     f"{w_total} trades, {w_wins} wins ({w_wins/w_total*100:.0f}%), "
                     f"P&L: ${w_pnl:,.0f}", fontweight='bold')
        ax.set_ylabel('Price ($)')
        ax.grid(True, alpha=0.3)
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
    
    plt.suptitle('🔍 DETAILED VIEW: Top 4 Windows by Trade Count', 
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('8_trades_zoomed_windows.png', dpi=150, bbox_inches='tight')
    print(f"  ✓ Saved: 8_trades_zoomed_windows.png")
    plt.close()
    
    # ═══════════════════════════════════════════════════════════════════
    # FIGURE 3: Trade Analysis Charts
    # ═══════════════════════════════════════════════════════════════════
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    
    # 1. Return distribution by direction
    ax = axes[0, 0]
    if 'direction' in trades_df.columns:
        long_returns = trades_df[trades_df['direction'] == 'LONG']['return_pct']
        short_returns = trades_df[trades_df['direction'] == 'SHORT']['return_pct']
        
        ax.hist(long_returns, bins=30, alpha=0.6, color='#27ae60', label=f'Long ({len(long_returns)})')
        if len(short_returns) > 0:
            ax.hist(short_returns, bins=30, alpha=0.6, color='#3498db', label=f'Short ({len(short_returns)})')
        ax.axvline(x=0, color='black', linestyle='-', linewidth=2)
        ax.legend()
    ax.set_xlabel('Return (%)')
    ax.set_ylabel('Frequency')
    ax.set_title('📊 Return Distribution by Direction', fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    # 2. P&L by Regime
    ax = axes[0, 1]
    regime_pnl = trades_df.groupby('regime')['pnl'].sum().sort_values()
    colors = [regime_colors.get(r, '#95a5a6') for r in regime_pnl.index]
    bars = ax.barh(regime_pnl.index, regime_pnl.values, color=colors, alpha=0.8)
    ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
    ax.set_xlabel('Total P&L ($)')
    ax.set_title('💰 P&L by Market Regime', fontweight='bold')
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x:,.0f}'))
    
    # 3. Win rate by regime
    ax = axes[0, 2]
    regime_wr = trades_df.groupby('regime').apply(
        lambda x: (x['pnl'] > 0).sum() / len(x) * 100
    ).sort_values()
    colors = [regime_colors.get(r, '#95a5a6') for r in regime_wr.index]
    ax.barh(regime_wr.index, regime_wr.values, color=colors, alpha=0.8)
    ax.axvline(x=50, color='gray', linestyle='--', alpha=0.5)
    ax.set_xlabel('Win Rate (%)')
    ax.set_title('🎯 Win Rate by Regime', fontweight='bold')
    ax.set_xlim(0, 100)
    
    # 4. Monthly P&L heatmap
    ax = axes[1, 0]
    trades_df['year'] = trades_df['exit_date'].dt.year
    trades_df['month'] = trades_df['exit_date'].dt.month
    
    monthly_pnl = trades_df.pivot_table(values='pnl', index='year', columns='month', aggfunc='sum')
    month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
                   'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
    monthly_pnl.columns = [month_names[m-1] for m in monthly_pnl.columns]
    
    sns.heatmap(monthly_pnl, annot=True, fmt='.0f', cmap='RdYlGn', center=0, 
                ax=ax, cbar_kws={'label': 'P&L ($)'}, linewidths=0.5)
    ax.set_title('📅 Monthly P&L Heatmap', fontweight='bold')
    
    # 5. Trade duration vs return
    ax = axes[1, 1]
    wins = trades_df[trades_df['pnl'] > 0]
    losses = trades_df[trades_df['pnl'] <= 0]
    
    ax.scatter(wins['days_held'], wins['return_pct'], 
               color='#27ae60', alpha=0.6, s=50, label='Wins')
    ax.scatter(losses['days_held'], losses['return_pct'], 
               color='#e74c3c', alpha=0.6, s=50, label='Losses')
    ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    ax.set_xlabel('Days Held')
    ax.set_ylabel('Return (%)')
    ax.set_title('⏱️ Trade Duration vs Return', fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # 6. Exit type breakdown
    ax = axes[1, 2]
    exit_stats = trades_df.groupby('exit_type').agg({
        'pnl': ['count', 'sum', 'mean']
    })
    exit_stats.columns = ['Count', 'Total P&L', 'Avg P&L']
    
    exit_colors = {
        'TAKE_PROFIT': '#27ae60',
        'STOP_LOSS': '#e74c3c',
        'TIME_EXIT': '#3498db'
    }
    
    x = np.arange(len(exit_stats))
    width = 0.35
    
    bars1 = ax.bar(x - width/2, exit_stats['Count'], width, 
                   label='Count', color='#3498db', alpha=0.8)
    ax2_twin = ax.twinx()
    bars2 = ax2_twin.bar(x + width/2, exit_stats['Total P&L'], width, 
                          label='P&L', color='#27ae60', alpha=0.8)
    
    ax.set_xticks(x)
    ax.set_xticklabels(exit_stats.index, rotation=15)
    ax.set_ylabel('Count')
    ax2_twin.set_ylabel('Total P&L ($)')
    ax2_twin.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x:,.0f}'))
    ax.set_title('🎯 Exit Type Analysis', fontweight='bold')
    ax.legend(loc='upper left')
    ax2_twin.legend(loc='upper right')
    
    plt.tight_layout()
    plt.savefig('9_trade_analysis.png', dpi=150, bbox_inches='tight')
    print(f"  ✓ Saved: 9_trade_analysis.png")
    plt.close()
    
    # ═══════════════════════════════════════════════════════════════════
    # FIGURE 4: Year-by-Year Performance
    # ═══════════════════════════════════════════════════════════════════
    
    years = sorted(trades_df['year'].unique())
    n_years = len(years)
    
    if n_years > 0:
        fig, axes = plt.subplots(min(n_years, 6), 1, figsize=(18, 4*min(n_years, 6)))
        if n_years == 1:
            axes = [axes]
        
        for ax_idx, year in enumerate(years[:6]):  # Max 6 years
            ax = axes[ax_idx]
            
            year_trades = trades_df[trades_df['year'] == year]
            year_price = df[(df['Date'].dt.year == year)]
            
            # Plot price
            ax.plot(year_price['Date'], year_price['SI=F_Close'], 
                    color=COLORS['price'], linewidth=1.5)
            
            # Plot trades
            for idx, trade in year_trades.iterrows():
                is_long = trade.get('direction', 'LONG') == 'LONG'
                is_win = trade['pnl'] > 0
                
                if is_long:
                    color = COLORS['long_win'] if is_win else COLORS['long_loss']
                    marker = '^'
                else:
                    color = COLORS['short_win'] if is_win else COLORS['short_loss']
                    marker = 'v'
                
                ax.scatter(trade['entry_date'], trade['entry_price'], 
                           color=color, marker=marker, s=100, zorder=5,
                           edgecolors='black', linewidths=0.5)
            
            # Year stats
            y_pnl = year_trades['pnl'].sum()
            y_wins = (year_trades['pnl'] > 0).sum()
            y_total = len(year_trades)
            y_wr = y_wins/y_total*100 if y_total > 0 else 0
            
            ax.set_title(f"{year}: {y_total} trades, {y_wins} wins ({y_wr:.0f}%), "
                         f"P&L: ${y_pnl:,.0f}", fontweight='bold')
            ax.set_ylabel('Price ($)')
            ax.grid(True, alpha=0.3)
            ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
            ax.xaxis.set_major_locator(mdates.MonthLocator())
        
        plt.suptitle('📅 YEAR-BY-YEAR TRADE VISUALIZATION', 
                     fontsize=14, fontweight='bold', y=1.01)
        plt.tight_layout()
        plt.savefig('10_yearly_trades.png', dpi=150, bbox_inches='tight')
        print(f"  ✓ Saved: 10_yearly_trades.png")
        plt.close()
    
    print("\n" + "="*70)
    print("📊 ALL VISUALIZATIONS COMPLETE")
    print("="*70)
    print("Files saved:")
    print("  7_trades_on_price.png    - Full overview with all trades")
    print("  8_trades_zoomed_windows.png - Detailed view of top windows")
    print("  9_trade_analysis.png     - Trade statistics and analysis")
    print("  10_yearly_trades.png     - Year-by-year breakdown")


def generate_current_signal(df, all_feats, walk_forward_results):
    """
    Generate trading signal for the CURRENT day

    V3.5 LIVE-INFERENCE PARITY VERSION:
    - Retrieves trained models from last walk-forward window
    - Extracts CURRENT features (not stored predictions)
    - Runs FRESH inference on today's data
    - BUY side blends LSTM/RF/XGB with the stored Brier-calibrated
      buy_weights — NOT equal (lstm+rf+xgb)/3 weights
    - SHORT side runs the DEDICATED short ensemble (lstm_short / rf_short /
      xgb_short) through short_feature_selector, blended with short_weights —
      exactly like OptimizedEnsemble.predict().  The old
      short_prob = 1 − buy_prob / agreement = 3 − buy_agreement inversion is
      GONE: "don't buy" ≠ "actively short"
    - Both directions are gated through EnhancedRegimeFilter.filter_signal()
      identically to enhanced_trade_simulation(); if both qualify, the
      direction with the larger threshold margin wins
    """
    print("\n" + "="*70)
    print("📡 CURRENT DAY SIGNAL GENERATION")
    print("="*70)

    if not walk_forward_results:
        print("No model results available")
        return None

    # Get the most recent window's trained models
    last_result = walk_forward_results[-1]

    # Current market data
    current_date = df['Date'].iloc[-1]
    current_price = df['SI=F_Close'].iloc[-1]

    print(f"\nDate: {current_date}")
    print(f"Silver Price: ${current_price:.2f}")

    # Detect current regime
    regime = MarketRegimeDetector.detect_regime(df)
    regime_desc = MarketRegimeDetector.get_regime_description(regime)
    print(f"Market Regime: {regime_desc}")

    try:
        regime_enum = MarketRegime(regime)
    except:
        regime_enum = MarketRegime.CONSOLIDATION
    config = REGIME_CONFIGS[regime_enum]

    # ══════════════════════════════════════════════════════════════════════
    # FRESH INFERENCE ON CURRENT DAY FEATURES — BUY AND SHORT ENSEMBLES
    # ══════════════════════════════════════════════════════════════════════

    has_models = ('lstm_model' in last_result and 'rf_model' in last_result and 'xgb_model' in last_result)

    buy_prob = short_prob = None
    buy_agreement = short_agreement = 0
    lstm_prob = rf_prob = xgb_prob = 0.5
    lstm_short_prob = rf_short_prob = xgb_short_prob = None
    used_dedicated_shorts = False

    if has_models:
        print(f"\n{'─'*50}")
        print("RUNNING FRESH INFERENCE ON CURRENT FEATURES...")
        print(f"{'─'*50}")

        try:
            # ── BUY-side models ──────────────────────────────────────────────
            lstm_model = last_result['lstm_model']
            rf_model = last_result['rf_model']
            xgb_model = last_result['xgb_model']
            scaler = last_result.get('scaler')
            feature_cols = last_result.get('feature_cols', all_feats)

            # ── SHORT-side models (dedicated, trained on SHORT_Label) ────────
            lstm_short = last_result.get('lstm_short')
            rf_short   = last_result.get('rf_short')
            xgb_short  = last_result.get('xgb_short')
            short_feature_selector = last_result.get('short_feature_selector')

            # ── Brier-calibrated ensemble weights (learned in train()) ───────
            buy_weights = last_result.get('buy_weights')
            if buy_weights is None:
                buy_weights = np.array([Config.LSTM_WEIGHT, Config.RF_WEIGHT, Config.XGB_WEIGHT])
            buy_weights = np.asarray(buy_weights, dtype=float)

            short_weights = last_result.get('short_weights')
            if short_weights is None:
                short_weights = np.array([Config.LSTM_WEIGHT, Config.RF_WEIGHT, Config.XGB_WEIGHT])
            short_weights = np.asarray(short_weights, dtype=float)

            # Extract current features (last row)
            current_features = df[feature_cols].iloc[-1:].copy()
            current_features = current_features.fillna(method='ffill').fillna(0)

            if scaler is not None:
                current_scaled = scaler.transform(current_features)
            else:
                current_scaled = current_features.values

            # ── BUY tabular models through the BUY feature selector ──────────
            feature_selector = last_result.get('feature_selector')
            if feature_selector is not None and feature_selector.features_to_keep is not None:
                rf_xgb_features = feature_selector.transform(current_features.values, feature_names=feature_cols)
                if rf_xgb_features.shape[1] == 0:
                    print("  ⚠️ Feature selection returned 0 features, using all features")
                    rf_xgb_features = current_scaled
            else:
                rf_xgb_features = current_scaled

            rf_prob = float(rf_model.predict_proba(rf_xgb_features)[0, 1])
            xgb_prob = float(xgb_model.predict_proba(rf_xgb_features)[0, 1])

            # ── LSTM sequence input (shared by buy + short LSTMs) ────────────
            seq_len = last_result.get('seq_len', 20)
            lstm_input = None
            if len(df) >= seq_len:
                lstm_features = df[feature_cols].iloc[-seq_len:].copy()
                lstm_features = lstm_features.fillna(method='ffill').fillna(0)
                if scaler is not None:
                    lstm_scaled = scaler.transform(lstm_features)
                else:
                    lstm_scaled = lstm_features.values
                lstm_input = lstm_scaled.reshape(1, seq_len, -1)
                lstm_prob = float(lstm_model.predict(lstm_input, verbose=0)[0, 0])
            else:
                lstm_prob = 0.5  # Neutral if not enough data

            # ── BUY ensemble: Brier-calibrated weights, NOT equal weights ────
            buy_prob = float(
                buy_weights[0] * lstm_prob +
                buy_weights[1] * rf_prob +
                buy_weights[2] * xgb_prob
            )
            buy_agreement = sum([
                1 if lstm_prob > 0.5 else 0,
                1 if rf_prob > 0.5 else 0,
                1 if xgb_prob > 0.5 else 0
            ])

            # ── SHORT ensemble: dedicated models + short selector + weights ──
            # Mirrors OptimizedEnsemble.predict() exactly.
            if (lstm_short is not None and rf_short is not None
                    and xgb_short is not None and lstm_input is not None):
                lstm_short_prob = float(lstm_short.predict(lstm_input, verbose=0)[0, 0])

                if (short_feature_selector is not None
                        and getattr(short_feature_selector, 'features_to_keep', None) is not None):
                    rf_xgb_short_features = short_feature_selector.transform(
                        current_features.values, feature_names=feature_cols)
                    if rf_xgb_short_features.shape[1] == 0:
                        rf_xgb_short_features = rf_xgb_features
                else:
                    rf_xgb_short_features = rf_xgb_features

                rf_short_prob  = float(rf_short.predict_proba(rf_xgb_short_features)[0, 1])
                xgb_short_prob = float(xgb_short.predict_proba(rf_xgb_short_features)[0, 1])

                short_prob = float(
                    short_weights[0] * lstm_short_prob +
                    short_weights[1] * rf_short_prob +
                    short_weights[2] * xgb_short_prob
                )
                short_agreement = sum([
                    1 if lstm_short_prob > 0.5 else 0,
                    1 if rf_short_prob > 0.5 else 0,
                    1 if xgb_short_prob > 0.5 else 0
                ])
                used_dedicated_shorts = True
            else:
                # Legacy fallback only if dedicated short models were never
                # trained/stored — same fallback predict() uses.
                print("  ⚠️ Dedicated short models unavailable — falling back to inverted BUY proxy")
                short_prob = 1 - buy_prob
                short_agreement = 3 - buy_agreement

            print(f"  ✓ Fresh inference complete "
                  f"({'dedicated short ensemble' if used_dedicated_shorts else 'inverted-BUY short fallback'})")

        except Exception as e:
            print(f"  ⚠️ Fresh inference failed: {e}")
            print(f"  Falling back to stored predictions...")
            buy_prob = float(last_result['ensemble_probs'][-1])
            buy_agreement = int(last_result['model_agreement'][-1])
            lstm_prob = float(last_result['lstm_probs'][-1])
            rf_prob = float(last_result['rf_probs'][-1])
            xgb_prob = float(last_result['xgb_probs'][-1])
            # Stored per-window predictions include the dedicated short outputs
            if last_result.get('short_probs') is not None:
                short_prob = float(np.asarray(last_result['short_probs'])[-1])
                short_agreement = int(np.asarray(last_result['short_model_agreement'])[-1])
                used_dedicated_shorts = last_result.get('short_lstm_probs') is not None
            else:
                short_prob = 1 - buy_prob
                short_agreement = 3 - buy_agreement

    elif 'ensemble_probs' in last_result and len(last_result['ensemble_probs']) > 0:
        # No stored models - use stored predictions
        print(f"\n{'─'*50}")
        print("USING STORED PREDICTIONS (models not saved)")
        print(f"{'─'*50}")
        buy_prob = float(last_result['ensemble_probs'][-1])
        buy_agreement = int(last_result['model_agreement'][-1])
        lstm_prob = float(last_result['lstm_probs'][-1])
        rf_prob = float(last_result['rf_probs'][-1])
        xgb_prob = float(last_result['xgb_probs'][-1])
        if last_result.get('short_probs') is not None:
            short_prob = float(np.asarray(last_result['short_probs'])[-1])
            short_agreement = int(np.asarray(last_result['short_model_agreement'])[-1])
            used_dedicated_shorts = last_result.get('short_lstm_probs') is not None
        else:
            short_prob = 1 - buy_prob
            short_agreement = 3 - buy_agreement

    if buy_prob is not None:

        print(f"\n{'─'*50}")
        print("MODEL PREDICTIONS:")
        print(f"{'─'*50}")
        print(f"  BUY ENSEMBLE (Brier-weighted):")
        print(f"    LSTM Probability:     {lstm_prob:.1%}")
        print(f"    Random Forest Prob:   {rf_prob:.1%}")
        print(f"    XGBoost Probability:  {xgb_prob:.1%}")
        print(f"    ─────────────────────────────")
        print(f"    BUY Probability:      {buy_prob:.1%}")
        print(f"    BUY Agreement:        {buy_agreement}/3")
        print(f"  SHORT ENSEMBLE ({'dedicated models' if used_dedicated_shorts else 'inverted-BUY fallback'}):")
        if used_dedicated_shorts and lstm_short_prob is not None:
            print(f"    LSTM-Short Prob:      {lstm_short_prob:.1%}")
            print(f"    RF-Short Prob:        {rf_short_prob:.1%}")
            print(f"    XGB-Short Prob:       {xgb_short_prob:.1%}")
            print(f"    ─────────────────────────────")
        print(f"    SHORT Probability:    {short_prob:.1%}")
        print(f"    SHORT Agreement:      {short_agreement}/3")

        # ── SIGNAL ANALYSIS — same gate as the backtest ──────────────────────
        print(f"\n{'─'*50}")
        print("SIGNAL ANALYSIS:")
        print(f"{'─'*50}")
        print(f"  Long Threshold:       {config.long_threshold:.0%}")
        print(f"  Short Threshold:      {config.short_threshold:.0%}")
        print(f"  Required Agreement:   {config.min_model_agreement}/3")

        # MR features for the filter — identical to enhanced_trade_simulation
        mr_feats = None
        if 'MR_SIGNAL' in df.columns:
            mr_feats = {
                'MR_SIGNAL':         df['MR_SIGNAL'].iloc[-1],
                'ZSCORE_60':         df['ZSCORE_60'].iloc[-1] if 'ZSCORE_60' in df.columns else 0,
                'IS_MEAN_REVERTING': df['IS_MEAN_REVERTING'].iloc[-1] if 'IS_MEAN_REVERTING' in df.columns else 0,
            }

        # Run BOTH directions through EnhancedRegimeFilter, exactly like the
        # backtest.  filter_signal enforces allow_long/allow_short, the
        # regime's threshold, min agreement, MIN_SIGNAL_CONFIDENCE, and the
        # symmetric MR boost.
        long_pass, long_adj_prob, long_reason = EnhancedRegimeFilter.filter_signal(
            buy_prob, regime_enum, TradeDirection.LONG, mr_feats, buy_agreement
        )
        short_pass, short_adj_prob, short_reason = EnhancedRegimeFilter.filter_signal(
            short_prob, regime_enum, TradeDirection.SHORT, mr_feats, short_agreement
        )

        print(f"  LONG  gate: {'PASS' if long_pass else 'FAIL'} — {long_reason} "
              f"(prob {long_adj_prob:.1%})")
        print(f"  SHORT gate: {'PASS' if short_pass else 'FAIL'} — {short_reason} "
              f"(prob {short_adj_prob:.1%})")

        # Direction resolution: larger threshold margin wins if both qualify
        long_margin  = long_adj_prob  - config.long_threshold  if long_pass  else -np.inf
        short_margin = short_adj_prob - config.short_threshold if short_pass else -np.inf

        signal = "NO TRADE"
        direction = None
        reason = ""
        signal_prob = buy_prob
        signal_agreement = buy_agreement
        threshold = config.long_threshold

        if long_pass and short_pass:
            if long_margin >= short_margin:
                signal, direction = "🟢 LONG", TradeDirection.LONG
                signal_prob, signal_agreement, threshold = long_adj_prob, buy_agreement, config.long_threshold
                reason = (f"Both directions qualified — LONG margin "
                          f"{long_margin:+.1%} > SHORT margin {short_margin:+.1%}")
            else:
                signal, direction = "🔴 SHORT", TradeDirection.SHORT
                signal_prob, signal_agreement, threshold = short_adj_prob, short_agreement, config.short_threshold
                reason = (f"Both directions qualified — SHORT margin "
                          f"{short_margin:+.1%} > LONG margin {long_margin:+.1%}")
        elif long_pass:
            signal, direction = "🟢 LONG", TradeDirection.LONG
            signal_prob, signal_agreement, threshold = long_adj_prob, buy_agreement, config.long_threshold
            reason = f"Long conditions met ({long_adj_prob:.1%} ≥ {config.long_threshold:.0%}, {buy_agreement}/3 agree)"
        elif short_pass:
            signal, direction = "🔴 SHORT", TradeDirection.SHORT
            signal_prob, signal_agreement, threshold = short_adj_prob, short_agreement, config.short_threshold
            reason = f"Dedicated short ensemble fired ({short_adj_prob:.1%} ≥ {config.short_threshold:.0%}, {short_agreement}/3 agree)"
        else:
            if not config.allow_long and not config.allow_short:
                signal = "🚫 BLOCKED"
                reason = f"All trading blocked in {regime}"
            else:
                signal = "⏸️ WAIT"
                reason = f"LONG: {long_reason} | SHORT: {short_reason}"
            # Report the direction that came closer to qualifying
            if short_adj_prob - config.short_threshold > buy_prob - config.long_threshold:
                signal_prob, signal_agreement, threshold = short_prob, short_agreement, config.short_threshold
            else:
                signal_prob, signal_agreement, threshold = buy_prob, buy_agreement, config.long_threshold

        print(f"\n  ╔══════════════════════════════════════╗")
        print(f"  ║  SIGNAL: {signal:<28} ║")
        print(f"  ╚══════════════════════════════════════╝")
        print(f"  Reason: {reason}")

        # Position sizing recommendation
        if "LONG" in signal or "SHORT" in signal:
            base_contracts = max(1, int(100000 / 50000))  # More aggressive base
            position_size = EnhancedRegimeFilter.get_position_size(
                base_contracts, regime_enum, signal_prob, signal_agreement)

            print(f"\n  RECOMMENDED POSITION:")
            print(f"  ─────────────────────")
            print(f"  Contracts: {position_size}")

            # Calculate stops/targets based on direction
            if "SHORT" in signal:
                stop_price = current_price * (1 + config.stop_loss_pct)  # ABOVE for short
                target_price = current_price * (1 - config.take_profit_pct)  # BELOW for short
                print(f"  Stop Loss: {config.stop_loss_pct:.1%} (${stop_price:.2f}) ⬆️ above entry")
                print(f"  Take Profit: {config.take_profit_pct:.1%} (${target_price:.2f}) ⬇️ below entry")
            else:  # LONG
                stop_price = current_price * (1 - config.stop_loss_pct)  # BELOW for long
                target_price = current_price * (1 + config.take_profit_pct)  # ABOVE for long
                print(f"  Stop Loss: {config.stop_loss_pct:.1%} (${stop_price:.2f}) ⬇️ below entry")
                print(f"  Take Profit: {config.take_profit_pct:.1%} (${target_price:.2f}) ⬆️ above entry")

            print(f"  Max Hold: 30 days")

        # Key indicators
        print(f"\n{'─'*50}")
        print("KEY INDICATORS:")
        print(f"{'─'*50}")

        if 'REAL_YIELD_5Y' in df.columns:
            real_yield = df['REAL_YIELD_5Y'].iloc[-1]
            print(f"  Real 5Y Yield: {real_yield:+.2f}% {'(BEARISH)' if real_yield > 1 else '(BULLISH)'}")

        if 'GS_RATIO' in df.columns:
            gs_ratio = df['GS_RATIO'].iloc[-1]
            gs_zscore = df['GS_RATIO_ZSCORE'].iloc[-1] if 'GS_RATIO_ZSCORE' in df.columns else 0
            signal_gs = "SILVER CHEAP → BUY" if gs_zscore > 1.5 else ("SILVER EXPENSIVE → SELL" if gs_zscore < -1.5 else "NEUTRAL")
            print(f"  G/S Ratio: {gs_ratio:.1f} (Z: {gs_zscore:+.2f}) → {signal_gs}")

        if 'MOMENTUM_SCORE' in df.columns:
            momentum = df['MOMENTUM_SCORE'].iloc[-1]
            print(f"  Momentum Score: {momentum:+.2f}% {'(BULLISH)' if momentum > 5 else ('(BEARISH)' if momentum < -5 else '(NEUTRAL)')}")

        if 'VOL_PERCENTILE' in df.columns:
            vol_pct = df['VOL_PERCENTILE'].iloc[-1]
            print(f"  Volatility: {vol_pct*100:.0f}th percentile")

        if 'HURST_60' in df.columns:
            hurst = df['HURST_60'].iloc[-1]
            mr_regime = "MEAN REVERTING" if hurst < 0.45 else ("TRENDING" if hurst > 0.55 else "RANDOM WALK")
            print(f"  Hurst (60d): {hurst:.3f} → {mr_regime}")

        if 'MR_SIGNAL' in df.columns:
            mr_signal = df['MR_SIGNAL'].iloc[-1]
            print(f"  MR Signal: {mr_signal:+.2f}")

        return {
            'date': current_date,
            'price': current_price,
            'regime': regime,
            'signal': signal,
            'direction': direction.value if direction is not None else None,
            # Direction-appropriate values — NO downstream inversion needed:
            # for SHORT these ARE the dedicated short-ensemble outputs.
            'probability': signal_prob,
            'model_agreement': signal_agreement,
            'threshold': threshold,
            'reason': reason,
            # Full detail for diagnostics
            'buy_prob': buy_prob,
            'buy_agreement': buy_agreement,
            'short_prob': short_prob,
            'short_agreement': short_agreement,
            'used_dedicated_shorts': used_dedicated_shorts,
        }

    return None


# Update main block
if __name__ == "__main__":
    df, walk_forward_results, all_trades = run_optimized_strategy()
    
    print("\n" + "="*70)
    print("✅ STRATEGY EXECUTION COMPLETE")
    print("="*70)
    
    # Generate visualizations
    if all_trades and len(all_trades) > 0:
        trades_df = pd.DataFrame(all_trades)
        plot_trades_on_price(df, trades_df)
    
    # Generate current signal
    if walk_forward_results:
        # We need to get all_feats - reconstruct it
        symbols = ["SI=F", "CL=F", "GC=F", "DX-Y.NYB", "HG=F", 
                   "^DJI", "^GSPC", "^IXIC", "^RUT", "^TNX", "^VIX"]
        all_feats = [c for c in df.columns if c not in ['Date', 'Label', 'future_ret']]
        current_signal = generate_current_signal(df, all_feats, walk_forward_results)
        
        # ══════════════════════════════════════════════════════════════════════
        # COMEX FUTURES ANALYSIS
        # ══════════════════════════════════════════════════════════════════════
        if current_signal:
            print("\n" + "="*70)
            print("📈 COMEX SILVER FUTURES ANALYSIS")
            print("="*70)
            
            # Get current market data
            current_price = current_signal['price']
            regime = current_signal['regime']
            signal = current_signal['signal']
            probability = current_signal['probability']
            model_agreement = current_signal['model_agreement']
            
            # Get realized volatility
            realized_vol = df['REALIZED_VOL_20'].iloc[-1] / 100 if 'REALIZED_VOL_20' in df.columns else 0.30
            
            # Get regime config for stop/target
            try:
                regime_enum = MarketRegime(regime)
                config = REGIME_CONFIGS[regime_enum]
            except:
                config = REGIME_CONFIGS[MarketRegime.CONSOLIDATION]
            
            # Determine if SHORT signal
            is_short = 'SHORT' in str(signal)
            
            # Prepare signal info for trade ticket.
            # NO inversion: generate_current_signal already returns the
            # direction-appropriate probability/agreement — for SHORT these
            # come from the DEDICATED short ensemble, not 1 − buy_prob.
            signal_info = {
                'signal': signal,
                'price': current_price,
                'regime': regime,
                'probability': probability,
                'model_agreement': model_agreement,
                'stop_loss_pct': config.stop_loss_pct,
                'take_profit_pct': config.take_profit_pct,
                'reason': current_signal.get('reason', '')
            }
            
            # Calculate position for each contract type
            sizer = FuturesPositionSizer(account_size=100000, max_risk_pct=0.02)
            
            print("\n" + "─"*70)
            print("POSITION SIZING BY CONTRACT TYPE:")
            print("─"*70)
            
            for contract_type in ["MICRO", "MINI", "FULL"]:
                position = sizer.calculate_position(
                    current_price=current_price,
                    stop_loss_pct=config.stop_loss_pct,
                    signal_confidence=signal_info['probability'],
                    realized_vol=realized_vol,
                    contract_type=contract_type
                )
                
                print(f"\n  {contract_type} ({position['symbol']}) - {position['contract_size_oz']:,} oz:")
                print(f"    Contracts: {position['recommended_contracts']}")
                print(f"    Margin: ${position['total_margin']:,.0f} ({position['margin_pct_account']:.1f}% of account)")
                print(f"    Risk: ${position['total_risk']:,.0f} ({position['risk_pct_account']:.1f}% of account)")
                print(f"    VaR (30d): ${position['var_30d_95']:,.0f} ({position['var_pct_account']:.1f}% of account)")
            
            # Generate trade ticket for MICRO (most accessible)
            micro_position = sizer.calculate_position(
                current_price=current_price,
                stop_loss_pct=config.stop_loss_pct,
                signal_confidence=signal_info['probability'],
                realized_vol=realized_vol,
                contract_type="MICRO"
            )
            
            print("\n" + generate_futures_trade_ticket(signal_info, micro_position, "V3.4 Balanced"))
            
            # Options hedge analysis
            if signal in ['🔴 SHORT', '🟢 LONG']:
                print(generate_options_hedge_analysis(
                    current_price, 
                    micro_position['recommended_contracts'],
                    realized_vol, 
                    is_short
                ))
    
    print("\n" + "="*70)
    print("🎯 SILVER SWING TRADING V3.5 COMPLETE")
    print("="*70)

ENHANCED SILVER SWING TRADING SYSTEM V3.4
BALANCED: HIGH SHARPE + SMART SHORTS

REGIME RULES (V3.4 BALANCED - High Sharpe + Smart Shorts)

📈 Strong Uptrend - Quality longs only
  Allow Long: ✓
  Allow Short: ✗ BLOCKED
  Long Threshold: 0.65
  Min Agreement: 2/3 models

📉 Strong Downtrend - ALL TRADING BLOCKED
  Allow Long: ✗ BLOCKED
  Allow Short: ✗ BLOCKED
  Long Threshold: 0.99
  Min Agreement: 3/3 models

➡️ Consolidation - Bidirectional
  Allow Long: ✓
  Allow Short: ✓
  Long Threshold: 0.62
  Min Agreement: 2/3 models

⚡ High Volatility - Bidirectional, high conviction only
  Allow Long: ✓
  Allow Short: ✓
  Long Threshold: 0.60
  Min Agreement: 2/3 models

LOADING MARKET DATA
Fetching data from 2016-02-24 to 2026-08-30...
  ✓ VXSLV loaded: 1825 observations (2016-02-24 – 2026-08-28)
  ✓ GVZ loaded: 2644 observations (2016-02-24 – 2026-08-28)
  · VXSLV gap proxy skipped (real=2641, gap=0)
✓ Loaded 2641 trading days
  Date range: 2016-02-24 00:00:00 to 2026-08-28 00:00:00
  Actual 

In [2]:
# ══════════════════════════════════════════════════════════════════════════
# DIAGNOSTIC CELL  —  paste into a NEW cell and run AFTER:
#     df, walk_forward_results, all_trades = run_optimized_strategy()
# Nothing is re-run or re-fit; it only reads objects already in memory.
# ══════════════════════════════════════════════════════════════════════════
import numpy as np
import pandas as pd


# ──────────────────────────────────────────────────────────────────────────
# PART 1 — FEATURE IMPORTANCE (which features actually drive the model)
# ──────────────────────────────────────────────────────────────────────────
def show_feature_importance(walk_forward_results, top_n=50):
    if not walk_forward_results:
        print("No walk_forward_results — run the strategy first."); return None
    last = walk_forward_results[-1]

    selector = last.get('feature_selector')
    selected = None
    if selector is not None:
        selected = (getattr(selector, 'features_to_keep', None)
                    or getattr(selector, 'final_features', None))
    if not selected:
        selected = last.get('feature_cols')
    if not selected:
        print("Could not find the selected feature list. Keys:", list(last.keys())); return None
    selected = list(selected)

    def _imp(res):
        for key in ('rf_model_base', 'rf_model', 'xgb_model_base', 'xgb_model'):
            m = res.get(key)
            if m is None:
                continue
            imp = getattr(m, 'feature_importances_', None)
            if imp is None:
                for attr in ('_base', 'base_estimator', 'estimator'):
                    inner = getattr(m, attr, None)
                    if inner is not None and hasattr(inner, 'feature_importances_'):
                        imp = inner.feature_importances_; break
            if imp is not None and len(imp) == len(selected):
                return np.asarray(imp), key
        return None, None

    imp, src = _imp(last)
    if imp is None:
        acc = np.zeros(len(selected)); hits = 0
        for res in walk_forward_results:
            i2, _ = _imp(res)
            if i2 is not None and len(i2) == len(selected):
                acc += i2; hits += 1
        if hits:
            imp, src = acc / hits, f"averaged over {hits} windows"
        else:
            print("No tree model with feature_importances_ found."); return None

    fi = (pd.DataFrame({'feature': selected, 'importance': imp})
            .sort_values('importance', ascending=False).reset_index(drop=True))
    fi['rank'] = fi.index + 1
    fi['pct'] = 100 * fi['importance'] / fi['importance'].sum()

    print(f"\n{'='*60}\nFEATURE IMPORTANCE  (source: {src}, {len(selected)} feats)\n{'='*60}")
    print(f"{'rank':>4}  {'feature':<36}{'imp%':>7}")
    for _, r in fi.head(top_n).iterrows():
        print(f"{int(r['rank']):>4}  {r['feature']:<36}{r['pct']:>6.2f}%")

    # Did the robustness transforms and COT rebuild land where intended?
    print(f"\n{'—'*60}\nWATCHLIST (features we specifically changed)\n{'—'*60}")
    watch = {
        'crude ROC (was raw CL=F_Close #1)': fi['feature'].str.startswith('CL=F_ROC'),
        'DJI ROC (was raw ^DJI_Close #5)'  : fi['feature'].str.startswith('^DJI_ROC'),
        'NET_LIQ normalized'               : fi['feature'].str.startswith('NET_LIQ_'),
        'TGA/RRP percentile'               : fi['feature'].isin(['TGA_PERCENTILE','RRP_PCTILE'])
                                             | fi['feature'].str.contains('TGA_CHG_20D_Z'),
        'volume ratio (was SMA)'           : fi['feature'].str.startswith('SI=F_Volume_Ratio'),
        'COT Williams index (rebuild)'     : fi['feature'].str.contains('cot_index', case=False),
        'CMT trade-flow'                   : fi['feature'].str.startswith('CMT_'),
    }
    for label, mask in watch.items():
        sub = fi[mask]
        if sub.empty:
            print(f"  {label:<36} —  not selected")
        else:
            ranks = ", ".join(f"#{int(r)}({p:.2f}%)" for r, p in zip(sub['rank'], sub['pct']))
            print(f"  {label:<36} {ranks}")

    # Dead-weight tail
    dead = fi[fi['pct'] < 0.10]
    print(f"\n  Dead weight (<0.10% importance): {len(dead)} of {len(fi)} features")
    if len(dead):
        print("   ", ", ".join(dead['feature'].tolist()))
    return fi


# ──────────────────────────────────────────────────────────────────────────
# PART 2 — P&L SPLIT BY YEAR AND REGIME (how much is the 2026 parabola?)
# ──────────────────────────────────────────────────────────────────────────
def show_pnl_breakdown(all_trades):
    if not all_trades:
        print("No trades to analyze."); return None
    t = pd.DataFrame(all_trades)
    if 'entry_date' in t.columns:
        t['entry_date'] = pd.to_datetime(t['entry_date'], errors='coerce')
        t['year'] = t['entry_date'].dt.year

    total = t['pnl'].sum()
    print(f"\n{'='*60}\nP&L BY YEAR  (total ${total:,.0f}, {len(t)} trades)\n{'='*60}")
    print(f"{'year':>6}{'trades':>8}{'win%':>7}{'P&L':>14}{'% of total':>12}")
    if 'year' in t.columns:
        cum_ex_2026 = 0.0
        for yr, g in t.groupby('year'):
            if pd.isna(yr):
                continue
            wr = (g['pnl'] > 0).mean() * 100
            share = 100 * g['pnl'].sum() / total if total else 0
            print(f"{int(yr):>6}{len(g):>8}{wr:>6.1f}%{g['pnl'].sum():>14,.0f}{share:>11.1f}%")
            if int(yr) < 2026:
                cum_ex_2026 += g['pnl'].sum()
        print(f"\n  Pre-2026 subtotal (your repeatable edge): ${cum_ex_2026:,.0f}")
        print(f"  2026 alone (the parabola)              : ${total - cum_ex_2026:,.0f}")
        print(f"  → 2026 is {100*(total-cum_ex_2026)/total:.0f}% of all profit"
              if total else "")

    if 'regime' in t.columns:
        print(f"\n{'='*60}\nP&L BY REGIME\n{'='*60}")
        print(f"{'regime':<22}{'trades':>8}{'win%':>7}{'P&L':>14}{'mean/trade':>12}")
        for rg, g in t.groupby('regime'):
            wr = (g['pnl'] > 0).mean() * 100
            print(f"{str(rg):<22}{len(g):>8}{wr:>6.1f}%{g['pnl'].sum():>14,.0f}"
                  f"{g['pnl'].mean():>12,.0f}")

    if 'direction' in t.columns:
        print(f"\n{'='*60}\nP&L BY DIRECTION\n{'='*60}")
        for dr, g in t.groupby('direction'):
            wr = (g['pnl'] > 0).mean() * 100
            print(f"  {str(dr):<6} {len(g):>4} trades, {wr:5.1f}% win, "
                  f"${g['pnl'].sum():>12,.0f}  (mean ${g['pnl'].mean():,.0f})")
    return t


# ── RUN BOTH ──────────────────────────────────────────────────────────────
fi = show_feature_importance(walk_forward_results, top_n=50)
tt = show_pnl_breakdown(all_trades)


FEATURE IMPORTANCE  (source: rf_model, 50 feats)
rank  feature                                imp%
   1  COT_cot_signal                        9.41%
   2  COPPER_ROC_60                         7.35%
   3  CL=F_PCTILE_252                       6.65%
   4  VAR_RATIO_5                           5.94%
   5  HG=F_PCTILE_252                       5.60%
   6  NET_LIQ_ZSCORE                        5.44%
   7  SOFR_FF_SPREAD_20D_AVG                5.19%
   8  CMT_Supply_Mined_Proxy_yoy_lag4m      4.85%
   9  COT_comm_net_zscore                   3.84%
  10  COPPER_CORR_20                        3.63%
  11  ^DJI_ROC_60                           3.27%
  12  FED_FUNDS_ROC_60                      3.25%
  13  SEASONAL_STRENGTH                     2.55%
  14  CMT_China_Solar_Powder_Demand_lag4m   2.38%
  15  HAR_COEF_W                            2.29%
  16  VOV_PCTILE                            2.27%
  17  VXSLV_PCTILE                          2.19%
  18  YIELD_CURVE_SLOPE                     2.16%


CONSOLIDATION: 271 trades, 45.4% win, $411,984

  LONG : 144 trades,  55.6% win, $   396,919, mean $  2,756
  SHORT: 127 trades,  33.9% win, $    15,065, mean $    119

  agree 2/3: 155 trades,  55.5% win, $   437,672
  agree 3/3: 116 trades,  31.9% win, $   -25,688



Computing permutation importance over 1 window(s), 10 shuffles each...

  window 1: single-class test set even after alignment (21 rows, class [0]) — skipped

No windows produced usable permutation scores. Most likely the stored model expects a different feature space than df provides (scaling/selection mismatch).  The impurity-based diagnostic (feature_importance_cell) is the fallback.


In [3]:
print('df' in dir(), 'walk_forward_results' in dir(), 'all_trades' in dir())

False False False


In [1]:
# -*- coding: utf-8 -*-
"""
SILVER SWING V3.5 — INTEGRATED SIGNAL DASHBOARD
==================================================================================
This is the standalone dashboard (silver_dashboard_v3_5_16) rewired to run at the
END of the strategy run, off the SAME in-memory objects the strategy already
built.  Nothing is re-downloaded and nothing is re-computed: no second yfinance
pull, no second 200 MB CFTC download, no second Comtrade pull, no second EGARCH
fit.  What you see on the chart is literally what the model saw.

WHAT CHANGED vs THE STANDALONE DASHBOARD
----------------------------------------
1. DELETED the entire duplicate data layer — load_market_data(), add_cot_features(),
   add_real_rate_features(), add_volatility_features(), add_momentum_features(),
   add_cross_asset_features(), add_statistical_features(), add_comtrade_*(),
   add_nadaraya_watson_envelope(), load_trades() and main().  The strategy is now
   the single source of truth for every series.  This also removes a whole class
   of silent drift: the standalone copy had its own VOL_PERCENTILE, its own NW
   envelope and its own COT parsing that could diverge from the strategy's.

2. COLUMN HARMONISATION.  The dashboard was written against v3.6 names, the
   strategy emits v3.5 names.  ensure_dashboard_columns() aliases them:
        SMA20 / SMA50 / SMA200   <-  SI=F_SMA20 / SI=F_SMA50 / SI=F_SMA200
        BB_Upper / BB_Mid / BB_Lower <- SI=F_BB_Upper / SI=F_BB_Mid / SI=F_BB_Lower
        RSI14                    <-  SI=F_RSI_14
        MACD_Hist                <-  SI=F_MACD_Hist
   Everything else already matches exactly (COT_*, CMT_*, NET_LIQ_ZSCORE,
   SOFR_FF_SPREAD_20D_AVG, RRP_PCTILE, VAR_RATIO_5, HURST_60, HALF_LIFE,
   IS_MEAN_REVERTING, VXSLV_PCTILE, VXSLV_IS_PROXY, EGARCH_COND_PCTILE,
   HAR_COEF_W, KURT_20, SKEW_DIVERGENCE, GS_RATIO*, SEASONAL_STRENGTH,
   REAL_YIELD_5Y, YIELD_CURVE_SLOPE, COPPER_ROC_60, COPPER_CORR_60,
   CL=F_PCTILE_252, HG=F_PCTILE_252, ^DJI_ROC_60, NW_Fit/Upper/Lower).
   Aliases are written to a COPY — the model's feature frame is never mutated.

3. TRADES come from the in-memory all_trades list, not a glob of ~/Documents.
   No risk of charting last week's CSV against this run's data.

4. REGIME RULES BOX is generated from the live REGIME_CONFIGS object.  The
   hardcoded text said "CONSOLIDATION: LONG-only", which is stale — V3.5 Run 13
   enabled CONSOLIDATION shorts.  It now always matches the config that ran.

5. LIVE SIGNAL BANNER.  If you pass the dict from generate_current_signal(),
   today's signal / regime / BUY prob / SHORT prob is printed in the chart title.

6. BUG FIX — _key() recursion.  In the standalone file the 11 panel "KEY:"
   legends had been pasted INSIDE _key()'s own body, so _key called itself 11
   times on entry.  Nothing called it, so it never blew the stack, but it meant
   every panel legend was dead code and no KEY box was ever drawn.  Each legend
   is now attached to the panel it describes.

7. THEME LEAK FIX.  The dark rcParams were applied at module import, which
   would have restyled the strategy's own plot_trades_on_price() figures.  They
   are now applied inside a plt.rc_context() around the dashboard only.

HOW TO USE
----------
Option A — keep as a separate file next to the strategy:

    from silver_dashboard_integrated import generate_dashboard
    generate_dashboard(df, all_trades, current_signal=current_signal)

Option B — paste this whole file at the bottom of the strategy script (there are
no name collisions: the config class here is DashConfig, not Config), then call
generate_dashboard() from __main__.  REGIME_CONFIGS is auto-detected in that case.

Either way, add this to the strategy's `if __name__ == "__main__":` block, after
generate_current_signal():

    generate_dashboard(df, all_trades, current_signal=current_signal)
"""

import os
import sys
from typing import Dict, List, Optional, Sequence

import matplotlib
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D


# ══════════════════════════════════════════════════════════════════════
# DASHBOARD CONFIG  (named DashConfig so it can coexist with the
# strategy's Config class when this file is pasted inline)
# ══════════════════════════════════════════════════════════════════════

class DashConfig:
    """Output settings.  All data settings now live in the strategy's Config."""

    # Windows to render, and the filename for each.
    PERIODS = ("1m", "3m", "1y", "3y")
    OUT_PATTERN = "silver_dashboard_v35_{period}.png"

    # Where charts are written.  None -> same folder the strategy saves its
    # trades CSV to (~/Documents, falling back to cwd), so the whole run's
    # artefacts land together.
    OUTDIR: Optional[str] = None

    # Annotation-only thresholds (the live values come from REGIME_CONFIGS)
    NW_LOWER_ZONE = 0.20
    NW_UPPER_ZONE = 0.80
    REAL_YIELD_BEARISH = 1.5


# ══════════════════════════════════════════════════════════════════════
# COLUMN HARMONISATION  (v3.5 strategy names  ->  dashboard panel names)
# ══════════════════════════════════════════════════════════════════════

# Strategy column  ->  dashboard column.  Pure renames; no recomputation, so
# the panel plots the exact series the model was fed.
COLUMN_ALIASES: Dict[str, str] = {
    "SI=F_SMA20":    "SMA20",
    "SI=F_SMA50":    "SMA50",
    "SI=F_SMA200":   "SMA200",
    "SI=F_BB_Upper": "BB_Upper",
    "SI=F_BB_Mid":   "BB_Mid",
    "SI=F_BB_Lower": "BB_Lower",
    "SI=F_RSI_14":   "RSI14",
    "SI=F_MACD_Hist": "MACD_Hist",
}

# Every column any panel touches.  Anything missing is created as all-NaN so a
# panel degrades to its "not available" note instead of raising KeyError.
REQUIRED_COLUMNS: Sequence[str] = (
    # panel 0 — price
    "Date", "SI=F_Close", "NW_Fit", "NW_Upper", "NW_Lower",
    "BB_Upper", "BB_Lower", "SMA20", "SMA50", "SMA200",
    # 1 COT
    "COT_cot_signal", "COT_comm_net_zscore", "COT_comm_cot_index",
    # 2 copper
    "COPPER_ROC_60", "COPPER_CORR_60", "HG=F_PCTILE_252",
    # 3 crude & equity risk
    "CL=F_PCTILE_252", "^DJI_ROC_60", "^GSPC_ROC_60",
    # 4 dollar liquidity
    "NET_LIQ_ZSCORE", "SOFR_FF_SPREAD_20D_AVG", "RRP_PCTILE",
    # 5 mean-reversion state
    "VAR_RATIO_5", "HURST_60", "HALF_LIFE", "IS_MEAN_REVERTING",
    # 7 implied vol
    "VXSLV_PCTILE", "VXSLV_IS_PROXY", "EGARCH_COND_PCTILE",
    # 8 gold/silver + seasonality
    "GS_RATIO", "GS_RATIO_ZSCORE", "SEASONAL_STRENGTH",
    # 10 real yields
    "REAL_YIELD_5Y", "YIELD_CURVE_SLOPE", "RSI14",
    # 11 vol structure
    "HAR_COEF_W", "KURT_20", "SKEW_DIVERGENCE",
)


def ensure_dashboard_columns(df: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    """
    Return a COPY of the strategy dataframe with dashboard-facing column names.

    Works on a copy on purpose: the strategy's df is the model's feature frame
    and adding duplicate alias columns to it would change what a later
    feature-selection pass sees.
    """
    out = df.copy()

    renamed, derived, missing = [], [], []

    for src, dst in COLUMN_ALIASES.items():
        if dst in out.columns:
            continue
        if src in out.columns:
            out[dst] = out[src]
            renamed.append(f"{src}->{dst}")

    # SMA/BB/RSI fallbacks: if the strategy ever runs without add_technical_
    # indicators(), rebuild from Close so panel 0 still renders.  Same formulas
    # the strategy uses, so the line is identical when both paths exist.
    if "SI=F_Close" in out.columns:
        close = out["SI=F_Close"]
        for ma in (20, 50, 200):
            col = f"SMA{ma}"
            if col not in out.columns or out[col].notna().sum() == 0:
                out[col] = close.rolling(ma).mean()
                derived.append(col)
        if "BB_Upper" not in out.columns or out["BB_Upper"].notna().sum() == 0:
            _m = close.rolling(20).mean()
            _s = close.rolling(20).std()
            out["BB_Upper"], out["BB_Mid"], out["BB_Lower"] = _m + 2 * _s, _m, _m - 2 * _s
            derived.append("BB_*")
        if "RSI14" not in out.columns or out["RSI14"].notna().sum() == 0:
            _d = close.diff()
            _g = _d.clip(lower=0).ewm(alpha=1 / 14, adjust=False).mean()
            _l = (-_d.clip(upper=0)).ewm(alpha=1 / 14, adjust=False).mean()
            out["RSI14"] = 100 - 100 / (1 + _g / (_l + 1e-12))
            derived.append("RSI14")

    for col in REQUIRED_COLUMNS:
        if col not in out.columns:
            out[col] = np.nan
            missing.append(col)

    if verbose:
        print("\n" + "-" * 70)
        print("[DASHBOARD] Harmonising columns from the strategy dataframe...")
        if renamed:
            print(f"  aliased : {', '.join(renamed)}")
        if derived:
            print(f"  derived : {', '.join(derived)}  (strategy column absent/empty)")
        if missing:
            print(f"  ⚠ absent : {', '.join(missing)}")
            print("    → those panels will show a 'not available' note, not crash")
        else:
            print("  ✓ every panel column present — no gaps")

    return out


# ══════════════════════════════════════════════════════════════════════
# TRADES: in-memory list  ->  dashboard frame
# ══════════════════════════════════════════════════════════════════════

def trades_to_frame(all_trades) -> Optional[pd.DataFrame]:
    """
    Accepts the strategy's all_trades list-of-dicts, an already-built
    DataFrame, or a CSV path.  Returns the frame the panels expect, or None.
    """
    if all_trades is None:
        return None

    if isinstance(all_trades, str):
        if not os.path.exists(all_trades):
            print(f"  ⚠ trades file not found: {all_trades}")
            return None
        t = pd.read_csv(all_trades)
    elif isinstance(all_trades, pd.DataFrame):
        t = all_trades.copy()
    else:
        if len(all_trades) == 0:
            print("  ⚠ all_trades is empty — price panel will show no markers")
            return None
        t = pd.DataFrame(all_trades)

    for col in ("entry_date", "exit_date"):
        if col in t.columns:
            t[col] = pd.to_datetime(t[col])

    if "direction" not in t.columns:
        t["direction"] = "LONG"
    t["direction"] = t["direction"].astype(str).str.upper()

    if "pnl" not in t.columns:
        t["pnl"] = 0.0

    print(f"  ✓ {len(t)} trades taken from this run "
          f"({(t['direction'] == 'LONG').sum()} long / "
          f"{(t['direction'] == 'SHORT').sum()} short)")
    return t


# ══════════════════════════════════════════════════════════════════════
# LIVE REGIME RULES + SIGNAL BANNER  (read from the strategy's own objects)
# ══════════════════════════════════════════════════════════════════════

def _resolve_regime_configs(regime_configs=None):
    """Use the caller's REGIME_CONFIGS if not passed explicitly."""
    if regime_configs is not None:
        return regime_configs
    main_mod = sys.modules.get("__main__")
    return getattr(main_mod, "REGIME_CONFIGS", None)


def build_regime_txt(regime_configs=None) -> Optional[str]:
    """
    Render the regime-rules box straight from REGIME_CONFIGS, so the chart can
    never disagree with the config that actually produced the trades.
    """
    cfgs = _resolve_regime_configs(regime_configs)
    if not cfgs:
        return None

    lines = ["  Regime rules (live config):\n"]
    for regime, cfg in cfgs.items():
        name = getattr(regime, "value", str(regime))
        if not cfg.allow_long and not cfg.allow_short:
            rule = "BLOCKED"
        elif cfg.allow_long and cfg.allow_short:
            rule = f"L{cfg.long_threshold:.2f}/S{cfg.short_threshold:.2f}"
        elif cfg.allow_long:
            rule = f"LONG-only {cfg.long_threshold:.2f}"
        else:
            rule = f"SHORT-only {cfg.short_threshold:.2f}"
        lines.append(f"  · {name[:17]:<17}\n")
        lines.append(f"      {rule}  {cfg.min_model_agreement}/3  "
                     f"x{cfg.position_size_mult:.2f}\n")
    lines.append("  (threshold  agreement  size-mult)\n")
    return "".join(lines)


def build_signal_banner(current_signal: Optional[dict]) -> str:
    """One-line summary of today's live signal for the chart title."""
    if not current_signal:
        return ""

    sig = str(current_signal.get("signal", "—"))
    prob = current_signal.get("probability")
    agree = current_signal.get("model_agreement")
    regime = current_signal.get("regime", "—")
    buy_p = current_signal.get("buy_prob")
    short_p = current_signal.get("short_prob")

    parts = [f"  TODAY: {sig}", f"Regime: {regime}"]
    if prob is not None:
        parts.append(f"Conf: {prob:.1%}")
    if agree is not None:
        parts.append(f"Agree: {agree}/3")
    if buy_p is not None and short_p is not None:
        parts.append(f"BUY {buy_p:.0%} / SHORT {short_p:.0%}")
        if not current_signal.get("used_dedicated_shorts", True):
            parts.append("[short=inverted-BUY fallback]")
    return "  |  ".join(parts)


# ══════════════════════════════════════════════════════════════════════
# ENTRY POINT — call this at the end of the strategy run
# ══════════════════════════════════════════════════════════════════════

def generate_dashboard(df: pd.DataFrame,
                       all_trades=None,
                       current_signal: Optional[dict] = None,
                       regime_configs=None,
                       periods: Sequence[str] = DashConfig.PERIODS,
                       outdir: Optional[str] = None,
                       out_pattern: str = DashConfig.OUT_PATTERN) -> List[str]:
    """
    Render the 12-panel dashboard for each window, using ONLY objects the
    strategy already produced.

    Parameters
    ----------
    df             : the dataframe returned by run_optimized_strategy()
    all_trades     : the all_trades list from run_optimized_strategy()
                     (a DataFrame or a CSV path also works)
    current_signal : dict from generate_current_signal() — drives the title banner
    regime_configs : REGIME_CONFIGS; auto-detected from __main__ if omitted
    periods        : any of '1m', '3m', '1y', '3y'
    outdir         : defaults to the strategy's save folder (~/Documents, else cwd)

    Returns the list of files written.
    """
    print("\n" + "=" * 70)
    print("📊 SIGNAL DASHBOARD  (in-memory — no re-download, no re-compute)")
    print("=" * 70)

    if df is None or len(df) == 0:
        print("  ⚠ empty dataframe — nothing to plot")
        return []
    if "Date" not in df.columns:
        print("  ⚠ no 'Date' column — cannot plot")
        return []

    if outdir is None:
        outdir = DashConfig.OUTDIR
    if outdir is None:
        docs = os.path.expanduser("~/Documents")
        outdir = docs if os.path.isdir(docs) else os.getcwd()
    os.makedirs(outdir, exist_ok=True)

    # 1. same columns the model used, under the names the panels expect
    dash_df = ensure_dashboard_columns(df)

    # 2. composite score from those same columns
    dash_df = compute_composite_score(dash_df)
    dash_df = dash_df.ffill()

    # 3. this run's trades
    trades = trades_to_frame(all_trades)

    # 4. live config + live signal
    regime_txt = build_regime_txt(regime_configs)
    if regime_txt is None:
        print("  ⚠ REGIME_CONFIGS not found — regime box will say so")
    banner = build_signal_banner(current_signal)

    written = []
    for period in periods:
        outfile = os.path.join(outdir, out_pattern.format(period=period))
        try:
            plot_dashboard(dash_df, trades, period=period, outfile=outfile,
                           signal_banner=banner, regime_txt=regime_txt)
            written.append(outfile)
        except Exception as exc:
            # One bad window must not take down the rest of the run.
            print(f"  ⚠ {period} dashboard failed: {type(exc).__name__}: {exc}")

    print("\n" + "=" * 70)
    print("  DASHBOARD COMPLETE")
    for f in written:
        print(f"  ✓  {f}")
    if not written:
        print("  ⚠ no charts written")
    print("=" * 70)
    return written
# ══════════════════════════════════════════════════════════════════════
# COMPOSITE SCORE  (RF-weighted blend, W23-W29)
# ══════════════════════════════════════════════════════════════════════

def compute_composite_score(df: pd.DataFrame) -> pd.DataFrame:
    """
    RF-importance-weighted composite signal score.
    Weights reflect ACTUAL v3.5.15 top-50 RF importances (not the old
    W23-W29 regime where NW_Position/GS_RATIO dominated — both have since
    rotated out of the top 50).  Score in [-1, +1]: positive = bullish.
    Each component contributes only if its column exists, so the blend
    degrades gracefully when FRED/COT/Comtrade data is missing.
    """
    print("\n" + "-" * 70)
    print("[QUANT] Computing Composite Signal Score (v3.5.15 RF weights)...")

    components = []

    # ① COT_cot_signal — #1 driver (9.4%).  Already signed: low commercial
    #    positioning (bullish) → positive.  Highest weight.
    if "COT_cot_signal" in df.columns and df["COT_cot_signal"].abs().max() > 0:
        components.append((df["COT_cot_signal"].clip(-1, 1), 0.18))
    elif "COT_comm_net_zscore" in df.columns:               # fallback
        components.append((df["COT_comm_net_zscore"].clip(-3, 3) / 3, 0.18))

    # ② COPPER_ROC_60 — #2 (7.4%).  Industrial-demand momentum, bullish silver.
    if "COPPER_ROC_60" in df.columns and df["COPPER_ROC_60"].abs().max() > 0:
        components.append((np.tanh(df["COPPER_ROC_60"] / 15), 0.13))

    # ③ CL=F_PCTILE_252 — #3 (6.7%).  Crude high in its range = reflation/
    #    industrial tailwind for silver.  Centered on 0.5.
    if "CL=F_PCTILE_252" in df.columns and df["CL=F_PCTILE_252"].notna().any():
        components.append(((df["CL=F_PCTILE_252"].fillna(0.5) - 0.5) * 2, 0.11))

    # ④ VAR_RATIO_5 — #4 (5.9%).  >1 = trending (momentum bullish in a bull
    #    market); <1 = mean-reverting.  Mild directional tilt.
    if "VAR_RATIO_5" in df.columns and df["VAR_RATIO_5"].abs().max() > 0:
        components.append(((df["VAR_RATIO_5"].fillna(1) - 1).clip(-1, 1) * 0.5, 0.09))

    # ⑤ HG=F_PCTILE_252 — #5 (5.6%).  Copper high-in-range = risk-on/industrial.
    if "HG=F_PCTILE_252" in df.columns and df["HG=F_PCTILE_252"].notna().any():
        components.append(((df["HG=F_PCTILE_252"].fillna(0.5) - 0.5) * 2, 0.09))

    # ⑥ NET_LIQ_ZSCORE — #6 (5.4%).  Ample dollar liquidity = bullish silver.
    if "NET_LIQ_ZSCORE" in df.columns and df["NET_LIQ_ZSCORE"].abs().max() > 0:
        components.append((df["NET_LIQ_ZSCORE"].clip(-3, 3) / 3, 0.08))

    # ⑦ SOFR_FF_SPREAD — #7 (5.2%).  Widening funding spread = stress; treat
    #    elevated spread as mildly bearish (risk-off).
    if "SOFR_FF_SPREAD_20D_AVG" in df.columns and df["SOFR_FF_SPREAD_20D_AVG"].abs().max() > 0:
        sp = df["SOFR_FF_SPREAD_20D_AVG"]
        sp_mx = sp.abs().rolling(252, min_periods=30).max().replace(0, 1)
        components.append((-(sp / sp_mx).clip(-1, 1), 0.07))

    # ⑧ CMT_Supply_Mined_yoy — #8 (4.9%).  Contracting mined supply (negative
    #    YoY) = tightening physical market = bullish → sign-flip.
    _cmt = next((c for c in df.columns if c.startswith("CMT_Supply_Mined_Proxy_yoy")), None)
    if _cmt and df[_cmt].abs().max() > 0:
        components.append((-np.tanh(df[_cmt] / 25), 0.06))

    # ⑨ COT_comm_net_zscore — #9 (3.8%).  Secondary COT confirmation.
    if "COT_comm_net_zscore" in df.columns and "COT_cot_signal" in df.columns:
        components.append((df["COT_comm_net_zscore"].clip(-3, 3) / 3, 0.05))

    # ⑩ ^DJI_ROC_60 / equity momentum — #11 (3.3%).  Risk-on backdrop.
    for _eq in ("^DJI_ROC_60", "^GSPC_ROC_60"):
        if _eq in df.columns and df[_eq].abs().max() > 0:
            components.append((np.tanh(df[_eq] / 10), 0.04)); break

    # Real yields — context (neg real = bullish silver).
    if "REAL_YIELD_5Y" in df.columns and df["REAL_YIELD_5Y"].abs().max() > 0:
        ry = df["REAL_YIELD_5Y"]
        components.append((np.tanh(-ry / 1.5), 0.04))

    if components:
        total_w = sum(w for _, w in components)
        df["COMPOSITE_SCORE"] = (sum(s * w for s, w in components) / total_w).clip(-1, 1)
    else:
        df["COMPOSITE_SCORE"] = 0.0

    print("  ✓ Composite score computed")
    return df
# ══════════════════════════════════════════════════════════════════════
# VISUALIZATION
# ══════════════════════════════════════════════════════════════════════

# Dark GitHub-style colors
BG     = "#0d1117"
PANEL  = "#161b22"
BORDER = "#30363d"
TEXT   = "#e6edf3"
TDIM   = "#8b949e"
GREEN  = "#3fb950"
RED    = "#f85149"
BLUE   = "#58a6ff"
YELLOW = "#d29922"
PURPLE = "#bc8cff"
ORANGE = "#ffa657"
TEAL   = "#39d353"
SILVER = "#c9d1d9"

DASH_RC = {
    "figure.facecolor":  BG,
    "axes.facecolor":    PANEL,
    "axes.edgecolor":    BORDER,
    "axes.labelcolor":   TEXT,
    "xtick.color":       TDIM,
    "ytick.color":       TDIM,
    "text.color":        TEXT,
    "grid.color":        BORDER,
    "grid.linewidth":    0.4,
    "grid.alpha":        0.5,
    "font.family":       "monospace",
    "font.size":         8.5,
    "legend.facecolor":  PANEL,
    "legend.edgecolor":  BORDER,
    "legend.framealpha": 0.9,
    "legend.fontsize":   7,
}


def _fmt(ax, ylabel="", title="", show_zero=False, ylim=None):
    ax.set_ylabel(ylabel, color=TDIM, fontsize=7.5, labelpad=3)
    if title:
        ax.set_title(title, color=TEXT, fontsize=8, fontweight="bold", loc="left", pad=2)
    ax.grid(True, alpha=0.3, lw=0.35)
    if show_zero:
        ax.axhline(0, color=BORDER, lw=0.7, ls="--")
    if ylim:
        ax.set_ylim(*ylim)
    ax.tick_params(labelsize=7.5)
    for sp in ax.spines.values():
        sp.set_edgecolor(BORDER)


def _xfmt(ax, period="3m"):
    if period == "1m":
        ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=mdates.MO, interval=1))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
    elif period == "3m":
        ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=mdates.MO, interval=2))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
    elif period == "3y":
        ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%b '%y"))
    else:
        ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%b '%y"))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha="right", fontsize=7)


def _twin(ax):
    ax2 = ax.twinx()
    ax2.tick_params(labelsize=7.5)
    ax2.spines["right"].set_edgecolor(BORDER)
    return ax2


def _key(ax, text):
    """Small how-to-read legend, upper-left inside the panel."""
    ax.text(0.008, 0.94, text, transform=ax.transAxes, ha="left", va="top",
            color=TDIM, fontsize=6.0, fontfamily="monospace", zorder=20,
            bbox=dict(boxstyle="round,pad=0.35", fc=BG, ec=BORDER, lw=0.6, alpha=0.85))


def _badge(ax, text, color=YELLOW):
    ax.text(1.003, 0.97, text, transform=ax.transAxes, fontsize=6,
            color=color, va="top", ha="left",
            bbox=dict(boxstyle="round,pad=0.25", fc=BG, ec=color, lw=0.5, alpha=0.9))


def _annotate_last(ax, dates, series, fmt="{:.2f}", color=SILVER):
    try:
        last_x = dates.iloc[-1]
        last_y = float(pd.Series(series).iloc[-1])
        if pd.notna(last_y):
            ax.annotate(fmt.format(last_y), xy=(last_x, last_y), xytext=(7, 0),
                        textcoords="offset points", color=color,
                        fontsize=7.5, va="center", zorder=10)
    except Exception:
        pass


def _plot_trades(ax, trades, start_date, end_date):
    if trades is None or "entry_date" not in trades.columns:
        return
    mask = (trades["entry_date"] >= start_date) & (trades["entry_date"] <= end_date)
    for _, t in trades[mask].iterrows():
        is_long = str(t.get("direction", "LONG")).upper() == "LONG"
        is_win  = float(t.get("pnl", 0)) >= 0
        col     = GREEN if is_win else RED
        mkr     = "^" if is_long else "v"
        ax.scatter(t["entry_date"], t["entry_price"],
                   color=col, marker=mkr, s=100, zorder=7,
                   edgecolors="white", linewidths=0.6, alpha=0.95)
        if "exit_date" in t and "exit_price" in t:
            ax.scatter(t["exit_date"], t["exit_price"],
                       color=col, marker="D", s=45, zorder=7,
                       edgecolors="white", linewidths=0.4, alpha=0.80)
            ax.plot([t["entry_date"], t["exit_date"]],
                    [t["entry_price"], t["exit_price"]],
                    color=col, lw=0.9, alpha=0.40, zorder=3)


def _trade_stats(trades, start_date, end_date):
    if trades is None or "entry_date" not in trades.columns:
        return None
    mask = (trades["entry_date"] >= start_date) & (trades["entry_date"] <= end_date)
    sub  = trades[mask]
    if sub.empty:
        return None
    n      = len(sub)
    wins   = (sub["pnl"] > 0).sum()
    pnl    = sub["pnl"].sum()
    longs  = (sub["direction"].str.upper() == "LONG").sum()
    return n, wins, wins / n * 100, pnl, longs, n - longs


def _plot_dashboard_impl(df: pd.DataFrame, trades: Optional[pd.DataFrame],
                         period: str, outfile: str,
                         signal_banner: str = "", regime_txt: Optional[str] = None):
    """
    Build a 12-panel signal dashboard for the given time window.
    Period: '1m', '3m', '1y', or '3y'
    """
    print("\n" + "=" * 70)
    print(f"📊 GENERATING DASHBOARD — {period.upper()} — {outfile}")
    print("=" * 70)

    end_date = df["Date"].max()
    if period == "1m":
        start_date = end_date - pd.DateOffset(months=1);  title_sfx = "Last 1 Month"
    elif period == "3m":
        start_date = end_date - pd.DateOffset(months=3);  title_sfx = "Last 3 Months"
    elif period == "3y":
        start_date = end_date - pd.DateOffset(years=3);   title_sfx = "Last 3 Years"
    else:
        start_date = end_date - pd.DateOffset(months=12); title_sfx = "Last 12 Months"

    sub   = df[(df["Date"] >= start_date) & (df["Date"] <= end_date)].copy()
    dates = sub["Date"]
    if len(sub) < 10:
        print(f"  ⚠ Not enough data for period {period}"); return

    close        = sub["SI=F_Close"]
    latest_price = close.iloc[-1]

    ts = _trade_stats(trades, start_date, end_date)
    if ts:
        n, wins, wr, pnl, longs, shorts = ts
        trade_summary = (f"  Trades: {n}  |  W/L: {wins}/{n-wins}  "
                         f"|  Win%: {wr:.0f}%  |  P&L: ${pnl:+,.0f}  "
                         f"|  L: {longs} / S: {shorts}")
    else:
        trade_summary = "  (no trades in period — load enhanced_trades_v36_*.csv)"

    heights  = [4.5, 1.3, 1.2, 1.3, 1.3, 1.3, 1.3, 1.3, 1.4, 1.3, 1.4, 1.4]
    fig      = plt.figure(figsize=(22, 34), facecolor=BG)
    fig.suptitle(
        f"  🥈  SILVER SWING V3.5  ·  {title_sfx}  ·  ${latest_price:.2f}  ·  "
        f"{start_date.strftime('%b %d %Y')} → {end_date.strftime('%b %d %Y')}\n"
        f"{trade_summary}"
        + (f"\n{signal_banner}" if signal_banner else ""),
        fontsize=11.5, fontweight="bold", color=SILVER, y=0.998, fontfamily="monospace",
    )

    gs_layout = gridspec.GridSpec(
        len(heights), 1, figure=fig, hspace=0.09, height_ratios=heights,
        left=0.055, right=0.895, top=0.985, bottom=0.028,
    )
    axes = [fig.add_subplot(gs_layout[i]) for i in range(len(heights))]

    # ─────────────────────────────────────────────────────────────────
    # 0  PRICE + NW CHANNEL + BOLLINGER + SMAs + TRADES
    # ─────────────────────────────────────────────────────────────────
    ax = axes[0]
    if "NW_Fit" in sub.columns and sub["NW_Fit"].notna().any():
        ax.fill_between(dates, sub["NW_Lower"], sub["NW_Upper"],
                        color=BLUE, alpha=0.09, zorder=0)
        ax.plot(dates, sub["NW_Fit"],   color=BLUE, lw=1.4, ls="--", alpha=0.75,
                zorder=2, label="NW Fit")
        ax.plot(dates, sub["NW_Upper"], color=BLUE, lw=0.6, alpha=0.45, zorder=2)
        ax.plot(dates, sub["NW_Lower"], color=BLUE, lw=0.6, alpha=0.45, zorder=2)
    if "BB_Upper" in sub.columns:
        ax.fill_between(dates, sub["BB_Lower"], sub["BB_Upper"],
                        color=PURPLE, alpha=0.06, zorder=0)
        ax.plot(dates, sub["BB_Upper"], color=PURPLE, lw=0.5, ls=":", alpha=0.5, zorder=2)
        ax.plot(dates, sub["BB_Lower"], color=PURPLE, lw=0.5, ls=":", alpha=0.5, zorder=2)
    if period in ("1y", "3y") and "SMA200" in sub.columns:
        ax.plot(dates, sub["SMA200"], color=ORANGE, lw=1.0, alpha=0.6, label="SMA200")
    if "SMA50" in sub.columns:
        ax.plot(dates, sub["SMA50"], color=YELLOW, lw=1.0, alpha=0.75, label="SMA50")
    if "SMA20" in sub.columns:
        ax.plot(dates, sub["SMA20"], color=TEAL,   lw=0.9, alpha=0.80, label="SMA20")
    ax.plot(dates, close, color=SILVER, lw=2.0, zorder=5, label="Silver (SI=F)")
    _plot_trades(ax, trades, start_date, end_date)
    leg_h = [
        Line2D([0],[0], color=SILVER, lw=2.0, label="Silver"),
        Line2D([0],[0], color=BLUE,   lw=1.4, ls="--", label="NW Fit"),
        Line2D([0],[0], color=PURPLE, lw=0.5, ls=":",  label="BBands"),
        Line2D([0],[0], color=TEAL,   lw=0.9, label="SMA20"),
        Line2D([0],[0], color=YELLOW, lw=1.0, label="SMA50"),
        Line2D([0],[0], color="w", lw=0),
        Line2D([0],[0], color=GREEN, marker="^", lw=0, ms=8, label="Long Win"),
        Line2D([0],[0], color=RED,   marker="^", lw=0, ms=8, label="Long Loss"),
        Line2D([0],[0], color=GREEN, marker="v", lw=0, ms=8, label="Short Win"),
        Line2D([0],[0], color=RED,   marker="v", lw=0, ms=8, label="Short Loss"),
    ]
    ax.legend(handles=leg_h, loc="upper left", ncol=5, framealpha=0.88, fontsize=7)
    _fmt(ax, ylabel="Price ($)",
         title="  SILVER PRICE  ·  NW Channel (causal rolling)  +  Bollinger  +  Trade Signals")
    _xfmt(ax, period); ax.set_xlim(start_date, end_date)

    # ═════════════════════════════════════════════════════════════════
    # PANELS RE-MAPPED TO V3.5.15 FEATURE IMPORTANCE (top drivers first).
    # Each panel guards for missing columns → shows a note instead of
    # crashing.  Ordered by RF importance so the eye lands on what moves
    # price NOW, not what mattered in older regimes.
    # ═════════════════════════════════════════════════════════════════
    def _note(ax, msg):
        ax.text(0.5, 0.5, msg, transform=ax.transAxes, ha="center", va="center",
                color=TDIM, fontsize=8, fontfamily="monospace",
                bbox=dict(boxstyle="round,pad=0.6", fc=PANEL, ec=BORDER, lw=0.7))

    # ─────────────────────────────────────────────────────────────────
    # 1  COT POSITIONING  [#1 driver: COT_cot_signal 9.4%]
    # ─────────────────────────────────────────────────────────────────
    ax = axes[1]
    has_sig = "COT_cot_signal" in sub.columns and sub["COT_cot_signal"].abs().max() > 0
    has_cz  = "COT_comm_net_zscore" in sub.columns and sub["COT_comm_net_zscore"].abs().max() > 0
    has_idx = "COT_comm_cot_index" in sub.columns and sub["COT_comm_cot_index"].abs().max() > 0
    if has_sig:
        s = sub["COT_cot_signal"].clip(-1, 1)
        ax.plot(dates, s, color=YELLOW, lw=1.5, zorder=5, label="COT signal")
        ax.fill_between(dates, 0, s, where=s > 0, color=GREEN, alpha=0.18)
        ax.fill_between(dates, 0, s, where=s < 0, color=RED,   alpha=0.16)
        ax.set_ylim(-1.15, 1.15)
        _annotate_last(ax, dates, s, "sig={:+.2f}", YELLOW)
    if has_cz:
        ax2 = _twin(ax)
        ax2.plot(dates, sub["COT_comm_net_zscore"].clip(-3, 3),
                 color=BLUE, lw=0.9, ls="--", alpha=0.8)
        _annotate_last(ax2, dates, sub["COT_comm_net_zscore"].clip(-3, 3), "commZ={:+.1f}", BLUE)
        ax2.set_ylabel("comm Z", color=BLUE, fontsize=7)
        ax2.tick_params(axis="y", labelcolor=BLUE)
    if has_idx:
        ax3 = _twin(ax); ax3.spines["right"].set_position(("axes", 1.06))
        ax3.plot(dates, sub["COT_comm_cot_index"], color=PURPLE, lw=0.8, ls=":", alpha=0.7)
        ax3.set_ylim(0, 100); ax3.axhline(20, color=GREEN, lw=0.4, ls=":"); ax3.axhline(80, color=RED, lw=0.4, ls=":")
    if not (has_sig or has_cz):
        _note(ax, "COT data unavailable — run with CFTC download enabled")
    _fmt(ax, ylabel="COT signal [-1,1]",
         title="  ① COT POSITIONING  —  cot_signal (smart-money, Williams %-rank anchored)",
         show_zero=True)
    _badge(ax, "v3.5.15 RF #1: COT_cot_signal 9.4% | comm_net_zscore #9 3.8% | Williams 156w index")
    _key(ax, "KEY: orange=COT signal (up=bullish)\nblue=commercial Z  purple=Williams rank\ngreen=bullish  red=bearish")
    _xfmt(ax, period); ax.set_xlim(start_date, end_date)

    # ─────────────────────────────────────────────────────────────────
    # 2  COPPER COMPLEX  [COPPER_ROC_60 #2 7.4% + HG=F_PCTILE_252 #5 5.6%]
    # ─────────────────────────────────────────────────────────────────
    ax = axes[2]
    has_cr = "COPPER_ROC_60" in sub.columns and sub["COPPER_ROC_60"].abs().max() > 0
    has_cp = "HG=F_PCTILE_252" in sub.columns and sub["HG=F_PCTILE_252"].notna().any()
    if has_cr:
        cr = sub["COPPER_ROC_60"]
        ax.bar(dates, cr, width=1.0, color=[TEAL if v >= 0 else RED for v in cr], alpha=0.55)
        ax.axhline(0, color=BORDER, lw=0.6)
        _annotate_last(ax, dates, cr, "CuROC60={:+.1f}%", TEAL)
    if has_cp:
        ax2 = _twin(ax)
        ax2.plot(dates, sub["HG=F_PCTILE_252"], color=ORANGE, lw=1.1, alpha=0.85)
        ax2.set_ylim(0, 1); ax2.set_ylabel("Cu %ile", color=ORANGE, fontsize=7)
        ax2.tick_params(axis="y", labelcolor=ORANGE)
        _annotate_last(ax2, dates, sub["HG=F_PCTILE_252"], "pct={:.0%}", ORANGE)
    if "COPPER_CORR_60" in sub.columns:
        ax.plot(dates, sub["COPPER_CORR_60"] * (sub["COPPER_ROC_60"].abs().max() or 1),
                color=BLUE, lw=0.7, ls=":", alpha=0.5)
    if not (has_cr or has_cp):
        _note(ax, "Copper (HG=F) data unavailable")
    _fmt(ax, ylabel="Cu ROC60 %",
         title="  ② COPPER COMPLEX  —  ROC_60 (momentum)  +  252d percentile (position-in-range)",
         show_zero=True)
    _badge(ax, "v3.5.15 RF: COPPER_ROC_60 #2 (7.4%) | HG=F_PCTILE_252 #5 (5.6%) | COPPER_CORR_20 #10")
    _key(ax, "KEY: bars=Cu 60d momentum(grn+/red-)\norange=Cu 252d pctile\nCu up=industrial=Ag bullish")
    _xfmt(ax, period); ax.set_xlim(start_date, end_date)

    # ─────────────────────────────────────────────────────────────────
    # 3  CRUDE & EQUITY RISK  [CL=F_PCTILE_252 #3 6.7% + ^DJI_ROC_60 #11]
    # ─────────────────────────────────────────────────────────────────
    ax = axes[3]
    has_clp = "CL=F_PCTILE_252" in sub.columns and sub["CL=F_PCTILE_252"].notna().any()
    has_dji = "^DJI_ROC_60" in sub.columns and sub["^DJI_ROC_60"].abs().max() > 0
    if not has_dji and "^GSPC_ROC_60" in sub.columns:  # fall back to SPX if DJI absent
        has_dji = sub["^GSPC_ROC_60"].abs().max() > 0
        dji_col = "^GSPC_ROC_60"; dji_lbl = "SPX ROC60"
    else:
        dji_col = "^DJI_ROC_60"; dji_lbl = "DJI ROC60"
    if has_clp:
        clp = sub["CL=F_PCTILE_252"]
        ax.plot(dates, clp, color=ORANGE, lw=1.4, zorder=5)
        ax.fill_between(dates, 0.5, clp, where=clp > 0.5, color=ORANGE, alpha=0.15)
        ax.fill_between(dates, 0.5, clp, where=clp < 0.5, color=BLUE,   alpha=0.15)
        ax.set_ylim(0, 1); ax.axhline(0.5, color=BORDER, lw=0.5, ls=":")
        _annotate_last(ax, dates, clp, "crude={:.0%}", ORANGE)
    if has_dji:
        ax2 = _twin(ax)
        dj = sub[dji_col]
        ax2.bar(dates, dj, width=1.0, color=[GREEN if v >= 0 else RED for v in dj], alpha=0.35)
        ax2.set_ylabel(dji_lbl, color=GREEN, fontsize=7)
        ax2.tick_params(axis="y", labelcolor=GREEN)
    if not (has_clp or has_dji):
        _note(ax, "Crude (CL=F) / equity data unavailable")
    _fmt(ax, ylabel="Crude %ile",
         title="  ③ CRUDE & RISK  —  CL=F 252d percentile  +  equity ROC_60 (industrial/energy read)")
    _badge(ax, "v3.5.15 RF: CL=F_PCTILE_252 #3 (6.7%) | ^DJI_ROC_60 #11 (3.3%) | ^RUT_ROC_20 #22")
    _key(ax, "KEY: orange=crude 252d pctile(shaded>0.5)\nbars=equity 60d ROC\nhigh crude+stocks=risk-on")
    _xfmt(ax, period); ax.set_xlim(start_date, end_date)

    # ─────────────────────────────────────────────────────────────────
    # 4  DOLLAR LIQUIDITY  [NET_LIQ_ZSCORE #6 5.4% + SOFR_FF_SPREAD #7 5.2%]
    # ─────────────────────────────────────────────────────────────────
    ax = axes[4]
    has_nl = "NET_LIQ_ZSCORE" in sub.columns and sub["NET_LIQ_ZSCORE"].abs().max() > 0
    has_sofr = "SOFR_FF_SPREAD_20D_AVG" in sub.columns and sub["SOFR_FF_SPREAD_20D_AVG"].abs().max() > 0
    if has_nl:
        nl = sub["NET_LIQ_ZSCORE"].clip(-3, 3)
        ax.plot(dates, nl, color=GREEN, lw=1.4, zorder=5)
        ax.fill_between(dates, 0, nl, where=nl > 0, color=GREEN, alpha=0.16)
        ax.fill_between(dates, 0, nl, where=nl < 0, color=RED,   alpha=0.16)
        _annotate_last(ax, dates, nl, "liqZ={:+.1f}", GREEN)
    if has_sofr:
        ax2 = _twin(ax)
        ax2.plot(dates, sub["SOFR_FF_SPREAD_20D_AVG"], color=PURPLE, lw=1.0, ls="--", alpha=0.85)
        ax2.set_ylabel("SOFR-FF sprd", color=PURPLE, fontsize=7)
        ax2.tick_params(axis="y", labelcolor=PURPLE)
        _annotate_last(ax2, dates, sub["SOFR_FF_SPREAD_20D_AVG"], "sprd={:+.3f}", PURPLE)
    if "RRP_PCTILE" in sub.columns and sub["RRP_PCTILE"].notna().any():
        ax.plot(dates, sub["RRP_PCTILE"] * 3 - 1.5, color=BLUE, lw=0.7, ls=":", alpha=0.5)
    if not (has_nl or has_sofr):
        _note(ax, "Liquidity (FRED) data unavailable — pip install pandas-datareader")
    _fmt(ax, ylabel="Net-Liq Z",
         title="  ④ DOLLAR LIQUIDITY  —  NET_LIQ z-score  +  SOFR-Fed Funds spread (funding stress)",
         show_zero=True)
    _badge(ax, "v3.5.15 RF: NET_LIQ_ZSCORE #6 (5.4%) | SOFR_FF_SPREAD #7 (5.2%) | FED_FUNDS_ROC60 #12")
    _key(ax, "KEY: green=NetLiq Z(up=ample=bullish)\npurple=SOFR-FF spread(up=stress)\nblue=RRP pctile")
    _xfmt(ax, period); ax.set_xlim(start_date, end_date)

    # ─────────────────────────────────────────────────────────────────
    # 5  MEAN-REVERSION STATE  [VAR_RATIO_5 #4 5.9% + HALF_LIFE #19 + HURST]
    # ─────────────────────────────────────────────────────────────────
    ax = axes[5]
    has_vr = "VAR_RATIO_5" in sub.columns and sub["VAR_RATIO_5"].abs().max() > 0
    if has_vr:
        vr = sub["VAR_RATIO_5"]
        ax.plot(dates, vr, color=TEAL, lw=1.4, zorder=5)
        ax.axhline(1.0, color=BORDER, lw=0.6, ls="--")
        ax.fill_between(dates, 1.0, vr, where=vr < 1.0, color=BLUE,  alpha=0.16)  # <1 = mean-reverting
        ax.fill_between(dates, 1.0, vr, where=vr > 1.0, color=ORANGE, alpha=0.14) # >1 = trending
        _annotate_last(ax, dates, vr, "VR5={:.2f}", TEAL)
    if "HURST_60" in sub.columns:
        ax2 = _twin(ax)
        ax2.plot(dates, sub["HURST_60"], color=PURPLE, lw=0.9, ls="--", alpha=0.8)
        ax2.axhline(0.5, color=BORDER, lw=0.3, ls=":")
        ax2.set_ylabel("Hurst", color=PURPLE, fontsize=7); ax2.set_ylim(0, 1)
        _annotate_last(ax2, dates, sub["HURST_60"], "H={:.2f}", PURPLE)
    if "IS_MEAN_REVERTING" in sub.columns:
        mr = sub["IS_MEAN_REVERTING"]
        for i in range(len(sub) - 1):
            if mr.iloc[i] == 1:
                ax.axvspan(dates.iloc[i], dates.iloc[i+1], color=BLUE, alpha=0.10, zorder=0)
    if not has_vr:
        _note(ax, "Mean-reversion features unavailable")
    _fmt(ax, ylabel="Var Ratio (5)",
         title="  ⑤ MEAN-REVERSION STATE  —  VAR_RATIO_5 (<1 revert / >1 trend)  +  Hurst  ·  bg=MR-gate")
    _badge(ax, "v3.5.15 RF: VAR_RATIO_5 #4 (5.9%) | HURST_60 (3.4%) | HALF_LIFE #19 | IS_MEAN_REVERTING")
    _key(ax, "KEY: green=VarRatio5(>1 trend/<1 revert)\npurple=Hurst  blue bg=MR-gate ON\nno shade=TRENDING now")
    _xfmt(ax, period); ax.set_xlim(start_date, end_date)

    # ─────────────────────────────────────────────────────────────────
    # 6  PHYSICAL TRADE FLOWS  [CMT_Supply_Mined_yoy #8 4.9% + China powder #14]
    # ─────────────────────────────────────────────────────────────────
    ax = axes[6]
    cmt_supply = next((c for c in sub.columns if c.startswith("CMT_Supply_Mined_Proxy_yoy")), None)
    cmt_powder = next((c for c in sub.columns if c.startswith("CMT_China_Solar_Powder_Demand") and not c.endswith(("_yoy","_mom3m","_z24m"))), None)
    has_supply = cmt_supply and sub[cmt_supply].abs().max() > 0
    has_powder = cmt_powder and sub[cmt_powder].abs().max() > 0
    if has_supply:
        sp = sub[cmt_supply].clip(-100, 100)
        ax.bar(dates, sp, width=1.0, color=[RED if v < 0 else GREEN for v in sp], alpha=0.5)
        ax.axhline(0, color=BORDER, lw=0.6)
        _annotate_last(ax, dates, sp, "supplyYoY={:+.0f}%", GREEN)
    if has_powder:
        ax2 = _twin(ax)
        ax2.plot(dates, sub[cmt_powder], color=PURPLE, lw=1.0, alpha=0.8)
        ax2.set_ylabel("China powder (kg)", color=PURPLE, fontsize=7)
        ax2.tick_params(axis="y", labelcolor=PURPLE)
    if not (has_supply or has_powder):
        _note(ax, "UN Comtrade trade-flow features unavailable\n(set UN_COMTRADE_API_KEY)")
    _fmt(ax, ylabel="Mined supply YoY %",
         title="  ⑥ PHYSICAL TRADE FLOWS  —  mined-supply YoY (MX+CA+PL+PE)  +  China solar powder demand",
         show_zero=True)
    _badge(ax, "v3.5.15 RF: CMT_Supply_Mined_yoy #8 (4.9%) | CMT_China_Powder #14 (2.4%) · 4m lag")
    _key(ax, "KEY: bars=supply YoY(red=tightening=bullish)\npurple=China powder kg\n4mo lag, monthly")
    _xfmt(ax, period); ax.set_xlim(start_date, end_date)

    # ─────────────────────────────────────────────────────────────────
    # 7  SILVER IMPLIED VOL  [VXSLV_PCTILE #17 + VOV_PCTILE #16 + EGARCH]
    # ─────────────────────────────────────────────────────────────────
    ax = axes[7]
    has_vx = "VXSLV_PCTILE" in sub.columns and sub["VXSLV_PCTILE"].notna().any()
    if has_vx:
        vx = sub["VXSLV_PCTILE"]
        ax.plot(dates, vx, color=RED, lw=1.3, zorder=5)
        ax.fill_between(dates, 0, vx, color=RED, alpha=0.12)
        ax.set_ylim(0, 1); ax.axhline(0.8, color=RED, lw=0.4, ls=":")
        _annotate_last(ax, dates, vx, "VXSLV={:.0%}", RED)
        if "VXSLV_IS_PROXY" in sub.columns:  # shade GVZ-proxied gap
            pr = sub["VXSLV_IS_PROXY"]
            for i in range(len(sub) - 1):
                if pr.iloc[i] == 1:
                    ax.axvspan(dates.iloc[i], dates.iloc[i+1], color=YELLOW, alpha=0.08, zorder=0)
    if "EGARCH_COND_PCTILE" in sub.columns and sub["EGARCH_COND_PCTILE"].notna().any():
        ax2 = _twin(ax)
        ax2.plot(dates, sub["EGARCH_COND_PCTILE"], color=BLUE, lw=0.9, ls="--", alpha=0.75)
        ax2.set_ylim(0, 1); ax2.set_ylabel("EGARCH %ile", color=BLUE, fontsize=7)
        ax2.tick_params(axis="y", labelcolor=BLUE)
    if not has_vx:
        _note(ax, "VXSLV unavailable (CBOE CDN / discontinued gap)")
    _fmt(ax, ylabel="VXSLV %ile",
         title="  ⑦ SILVER IMPLIED VOL  —  VXSLV percentile (yellow bg = GVZ-proxied 2022-25 gap)  +  EGARCH")
    _badge(ax, "v3.5.15 RF: VXSLV_PCTILE #17 (2.2%) | VOV_PCTILE #16 (2.3%) | EGARCH_COND_PCTILE")
    _key(ax, "KEY: red=VXSLV pctile(high=fear)\nYELLOW bg=GVZ-proxied 22-25 gap\nblue=EGARCH vol pctile")
    _xfmt(ax, period); ax.set_xlim(start_date, end_date)

    # ─────────────────────────────────────────────────────────────────
    # 8  GS RATIO + SEASONALITY  [SEASONAL_STRENGTH #13 + GS_RATIO context]
    # ─────────────────────────────────────────────────────────────────
    ax = axes[8]
    if "GS_RATIO" in sub.columns and sub["GS_RATIO"].notna().any():
        gs = sub["GS_RATIO"]
        ax.plot(dates, gs, color=SILVER, lw=1.3, zorder=5)
        _annotate_last(ax, dates, gs, "GS={:.1f}", SILVER)
        if "GS_RATIO_ZSCORE" in sub.columns:
            ax2 = _twin(ax)
            gz = sub["GS_RATIO_ZSCORE"].clip(-3, 3)
            ax2.fill_between(dates, 0, gz, where=gz > 1.5, color=GREEN, alpha=0.25)  # silver cheap
            ax2.fill_between(dates, 0, gz, where=gz < -1.5, color=RED, alpha=0.22)   # silver expensive
            ax2.plot(dates, gz, color=BLUE, lw=0.8, ls="--", alpha=0.7)
            ax2.set_ylabel("GS z", color=BLUE, fontsize=7); ax2.axhline(0, color=BORDER, lw=0.3)
    else:
        _note(ax, "Gold (GC=F) data unavailable")
    if "SEASONAL_STRENGTH" in sub.columns:
        ax3 = _twin(ax); ax3.spines["right"].set_position(("axes", 1.06))
        ax3.plot(dates, sub["SEASONAL_STRENGTH"], color=ORANGE, lw=0.7, ls=":", alpha=0.6)
        ax3.set_ylabel("seasonal", color=ORANGE, fontsize=6.5)
    _fmt(ax, ylabel="Gold/Silver",
         title="  ⑧ GOLD/SILVER RATIO  —  level + z-score (>1.5 silver cheap)  +  SEASONAL_STRENGTH")
    _badge(ax, "v3.5.15 RF: SEASONAL_STRENGTH #13 (2.6%) | GS_RATIO context (rotated out of top-50)")
    _key(ax, "KEY: white=GS level  blue=GS z-score\ngreen(z>1.5)=Ag cheap\nred(z<-1.5)=Ag expensive")
    _xfmt(ax, period); ax.set_xlim(start_date, end_date)

    # ─────────────────────────────────────────────────────────────────
    # 9  COMPOSITE SIGNAL SCORE  (RF-weighted blend, v3.5.15 weights)
    # ─────────────────────────────────────────────────────────────────
    ax = axes[9]
    if "COMPOSITE_SCORE" in sub.columns and sub["COMPOSITE_SCORE"].abs().max() > 0:
        cs = sub["COMPOSITE_SCORE"]
        ax.plot(dates, cs, color=YELLOW, lw=1.6, zorder=5)
        ax.fill_between(dates, 0, cs, where=cs > 0, color=GREEN, alpha=0.18)
        ax.fill_between(dates, 0, cs, where=cs < 0, color=RED,   alpha=0.16)
        ax.axhline(0.3, color=GREEN, lw=0.4, ls=":"); ax.axhline(-0.3, color=RED, lw=0.4, ls=":")
        _annotate_last(ax, dates, cs, "score={:+.2f}", YELLOW)
    else:
        _note(ax, "COMPOSITE_SCORE not computed for this window")
    _fmt(ax, ylabel="Composite [-1,1]",
         title="  ⑨ COMPOSITE SIGNAL  —  importance-weighted blend of the drivers above",
         show_zero=True)
    _badge(ax, "v3.5.15: COT + copper + crude + liquidity + MR-state, RF-importance weighted")
    _key(ax, "KEY: weighted blend of all drivers\n>0=bullish  <0=bearish\ndotted=+/-0.3 strong thresholds")
    _xfmt(ax, period); ax.set_xlim(start_date, end_date)

    # ─────────────────────────────────────────────────────────────────
    # 10  REAL YIELDS + RSI  [REAL_YIELD_5Y context + RSI14 + YIELD_CURVE]
    # ─────────────────────────────────────────────────────────────────
    ax = axes[10]
    if "REAL_YIELD_5Y" in sub.columns and sub["REAL_YIELD_5Y"].abs().max() > 0:
        ry = sub["REAL_YIELD_5Y"]
        ax.plot(dates, ry, color=TEAL, lw=1.3, zorder=5)
        ax.fill_between(dates, 0, ry, where=ry < 0, color=GREEN, alpha=0.16)  # neg real = bullish silver
        ax.axhline(0, color=BORDER, lw=0.6, ls="--")
        _annotate_last(ax, dates, ry, "realY={:+.2f}%", TEAL)
    else:
        _note(ax, "Real yields (FRED) unavailable")
    if "YIELD_CURVE_SLOPE" in sub.columns:
        ax2 = _twin(ax)
        ax2.plot(dates, sub["YIELD_CURVE_SLOPE"], color=PURPLE, lw=0.8, ls="--", alpha=0.7)
        ax2.set_ylabel("2s10s", color=PURPLE, fontsize=7); ax2.axhline(0, color=BORDER, lw=0.3)
    if "RSI14" in sub.columns:
        ax3 = _twin(ax); ax3.spines["right"].set_position(("axes", 1.06))
        ax3.plot(dates, sub["RSI14"], color=ORANGE, lw=0.7, alpha=0.55)
        ax3.set_ylim(0, 100); ax3.axhline(70, color=RED, lw=0.3, ls=":"); ax3.axhline(30, color=GREEN, lw=0.3, ls=":")
        ax3.set_ylabel("RSI", color=ORANGE, fontsize=6.5)
    _fmt(ax, ylabel="Real 5Y %",
         title="  ⑩ REAL YIELDS  —  5Y real (neg = bullish silver)  +  YIELD_CURVE_SLOPE #18  +  RSI",
         show_zero=True)
    _badge(ax, "v3.5.15 RF: YIELD_CURVE_SLOPE #18 (2.2%) | REAL_YIELD context | RSI14 technical")
    _key(ax, "KEY: green=5Y real(NEG=bullish Ag)\npurple=2s10s curve  orange=RSI\nhigh real yield=headwind")
    _xfmt(ax, period); ax.set_xlim(start_date, end_date)

    # ─────────────────────────────────────────────────────────────────
    # 11  HAR-RV & DISTRIBUTION  [HAR_COEF_W #15 + KURT_20 #20 + SKEW_DIV #24]
    # ─────────────────────────────────────────────────────────────────
    ax = axes[11]
    has_har = "HAR_COEF_W" in sub.columns and sub["HAR_COEF_W"].abs().max() > 0
    if has_har:
        hw = sub["HAR_COEF_W"]
        ax.plot(dates, hw, color=BLUE, lw=1.2, zorder=5, label="HAR weekly β")
        _annotate_last(ax, dates, hw, "HARw={:.2f}", BLUE)
    if "KURT_20" in sub.columns and sub["KURT_20"].abs().max() > 0:
        ax2 = _twin(ax)
        ax2.plot(dates, sub["KURT_20"], color=ORANGE, lw=0.9, ls="--", alpha=0.75)
        ax2.set_ylabel("Kurt20", color=ORANGE, fontsize=7)
        ax2.tick_params(axis="y", labelcolor=ORANGE)
        _annotate_last(ax2, dates, sub["KURT_20"], "kurt={:.1f}", ORANGE)
    if "SKEW_DIVERGENCE" in sub.columns and sub["SKEW_DIVERGENCE"].abs().max() > 0:
        ax.plot(dates, sub["SKEW_DIVERGENCE"], color=PURPLE, lw=0.7, ls=":", alpha=0.6)
    if not has_har:
        _note(ax, "HAR-RV / distribution features unavailable")
    _fmt(ax, ylabel="HAR weekly β",
         title="  ⑪ VOL STRUCTURE  —  HAR-RV weekly coef  +  KURT_20 (tail risk)  +  SKEW_DIVERGENCE",
         show_zero=True)
    _badge(ax, "v3.5.15 RF: HAR_COEF_W #15 (2.3%) | KURT_20 #20 (1.7%) | SKEW_DIVERGENCE #24 | JUMP_SIGMA #32")
    _key(ax, "KEY: blue=HAR weekly beta\norange=kurtosis(tail risk)\npurple=skew divergence")
    _xfmt(ax, period); ax.set_xlim(start_date, end_date)

    # Hide x-ticks on all but bottom panel
    for ax_i in axes[:-1]:
        plt.setp(ax_i.xaxis.get_majorticklabels(), visible=False)

    # ── Right-side RF importance legend (v3.5.15 ACTUAL top-50) ──────
    rf_txt = (
        "  RF Feature Importance  (v3.5.15)\n"
        "  ──────────────────────────────────\n"
        "  ① COT_cot_signal        9.4% ★\n"
        "  ② COPPER_ROC_60         7.4%\n"
        "  ③ CL=F_PCTILE_252       6.7% ✦\n"
        "  ④ VAR_RATIO_5           5.9%\n"
        "  ⑤ HG=F_PCTILE_252       5.6% ✦\n"
        "  ⑥ NET_LIQ_ZSCORE        5.4% ✦\n"
        "  ⑦ SOFR_FF_SPREAD_20D    5.2%\n"
        "  ⑧ CMT_Supply_Mined_yoy  4.9%\n"
        "  ⑨ COT_comm_net_zscore   3.8%\n"
        "  ⑩ COPPER_CORR_20        3.6%\n"
        "     ^DJI_ROC_60          3.3% ✦\n"
        "     FED_FUNDS_ROC_60     3.3%\n"
        "     SEASONAL_STRENGTH    2.6%\n"
        "     CMT_China_Powder     2.4%\n"
        "     HAR_COEF_W           2.3%\n"
        "     VOV_PCTILE           2.3%\n"
        "     VXSLV_PCTILE         2.2% ✦\n"
        "     YIELD_CURVE_SLOPE    2.2%\n"
        "     HALF_LIFE            2.0%\n"
        "     KURT_20              1.7%\n"
        "     COT_oi_chg_4w        1.7%\n"
        "  ──────────────────────────────────\n"
        "  ✦ = v3.5.15 ROBUST transform\n"
        "    (raw level → percentile / ROC /\n"
        "     z-score; de-registered levels\n"
        "     that drifted out-of-range)\n"
        "  ★ COT rebuilt: Williams 156w\n"
        "    percentile index → now #1\n"
        "  ──────────────────────────────────\n"
        "  Panels re-ordered by importance:\n"
        "  ① COT  ② Copper  ③ Crude/risk\n"
        "  ④ Liquidity  ⑤ Mean-reversion\n"
        "  ⑥ Trade-flows  ⑦ Silver-vol\n"
        "  ⑧ GS-ratio  ⑨ Composite\n"
        "  ⑩ Real-yields  ⑪ Vol-structure\n"
        "  ──────────────────────────────────\n"
        + (regime_txt if regime_txt else
           "  Regime rules: (REGIME_CONFIGS\n"
           "  not supplied — pass regime_configs\n"
           "  to generate_dashboard)\n")
    )
    fig.text(0.899, 0.85, rf_txt, fontsize=6.5, color=TDIM,
             va="top", ha="left", fontfamily="monospace",
             bbox=dict(boxstyle="round,pad=0.5", fc=PANEL, ec=BORDER, lw=0.8))

    plt.savefig(outfile, dpi=150, bbox_inches="tight", facecolor=BG)
    print(f"  ✓ Saved: {outfile}")
    plt.close()

def plot_dashboard(df: pd.DataFrame, trades: Optional[pd.DataFrame],
                   period: str, outfile: str,
                   signal_banner: str = "", regime_txt: Optional[str] = None):
    """
    Public entry point.  The dark theme is applied inside an rc_context so it
    does NOT leak into the strategy's own matplotlib output
    (plot_trades_on_price etc. keep their seaborn-v0_8 light styling).
    """
    with plt.rc_context(DASH_RC):
        _plot_dashboard_impl(df, trades, period, outfile,
                             signal_banner=signal_banner, regime_txt=regime_txt)